In [ ]:
# %% Cell 1 — Imports
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 150,
    'font.size': 12,
    'font.weight': 'medium',
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.labelweight': 'bold',
    'axes.linewidth': 2.0,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.constrained_layout.use': True,
    'legend.fontsize': 10,
    'legend.title_fontsize': 11,
})

C = {
    # Core
    'pink':    '#C27C8A',
    'mauve':   '#C9A9C4',
    'plum':    '#8D6B94',
    'purple':  '#6B5B7B',
    'navy':    '#2B2D42',
    'slate':   '#7B8BA6',
    'coral':   '#E07A5F',
}

In [ ]:
# %% Cell 1c — Software versions (for the manuscript methods section)
import sys, platform
import scipy, statsmodels, matplotlib

print(f"Python       {sys.version.split()[0]}   ({platform.platform()})")
for _m in (pd, np, scipy, statsmodels, matplotlib):
    print(f"{_m.__name__:12s} {_m.__version__}")


In [ ]:
# %% Cell 1b — Local execution
# The original notebook ran on Google Colab and mounted Drive at this point.
# This version runs locally; all paths are set in the next cell.

In [ ]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────
# PROJECT_DIR is the folder holding the final dataset and the outputs/ tree.
# It defaults to wherever the notebook was opened from, captured once so that
# re-running this cell after the os.chdir() below does not move it. Set it by
# hand if the export lives somewhere else.
PROJECT_DIR = globals().get('PROJECT_DIR') or os.path.abspath(os.getcwd())
DATA_FILE   = os.path.join(PROJECT_DIR, 'Final Dataset 9_21_26.xlsx')
assert os.path.exists(DATA_FILE), f"data file not found under {PROJECT_DIR}"

# ── Output routing ─────────────────────────────────────────────────────────
# Figures go to FIGURE_DIR. Tables are saved with a './' path, so the working
# directory is pointed at OUTPUT_DIR. Tables hold patient-level rows, as does
# the zero-SCC roster figure, so OUTPUT_DIR stays out of the code repository;
# FIGURE_DIR holds only aggregate or unlabelled figures.
# This run: 10-years-zero exclusion, the team's final dataset (Cell 2b).
OUTPUT_DIR  = 'final_10yearszero'
OUTPUT_BASE = os.path.join(PROJECT_DIR, 'outputs')
FIGURE_DIR  = os.path.join(PROJECT_DIR, 'figures')

_OUT = os.path.join(OUTPUT_BASE, OUTPUT_DIR)
os.makedirs(_OUT, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)
os.chdir(_OUT)
print(f"✔ Tables  → {os.getcwd()}")
print(f"✔ Figures → {FIGURE_DIR}")

In [ ]:
# %% Cell 2 — Load raw data
df_raw = pd.read_excel(DATA_FILE)
print(f"Raw dataset: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

In [ ]:
# %% Cell 2b — Study-team exclusions and the patient flow (updated 2026-09-21)
# ===========================================================================
# The source file is the team's final dataset. It already carries every
# correction requested so far, so nothing is edited in code here:
#   · the five pre-treatment counts from 2026-08-15 (Study IDs 169, 78, 167,
#     59 and UT Austin / Patient 8),
#   · Study ID 78, months 5 and 8 set to 0 SCCs (2026-09-08),
#   · three side-effect checkboxes (Study IDs 104, 127, 214; 2026-09-08),
#   · the 11 UT Austin rows renumbered to Study IDs 241-251, so Study ID is
#     now unique. (Before this, Study ID 9 on the exclusion list also matched
#     UT Austin / Patient 6; that patient fails the 3-month rule anyway.)
# The corrections are verified below, not re-applied.
#
# Two sets of exclusions from the team (2026-08-15):
#   · 12 patients to drop by Study ID, no data rule given, and
#   · 17 patients whose Year-1 pre-treatment SCC count is "Unknown/not
#     reported" (their red rows). That one is applied as a rule in Cell 6 and
#     checked here against their list.
# ===========================================================================

_YCOL = {y: [c for c in df_raw.columns
             if f'Year #{y} before' in c and 'invasive SCCs' in c][0]
         for y in range(1, 11)}
_MCOL = {m: [c for c in df_raw.columns
             if f'month #{m}' in c and 'invasive SCCs' in c][0]
         for m in (5, 8)}
_SECOL = {lab: [c for c in df_raw.columns
                if 'acitretin side effects' in c and f'choice={lab})' in c][0]
          for lab in ('None', 'Other')}

assert df_raw['Study ID'].is_unique, "Study ID must be unique in the final dataset"

# ── Verify the requested corrections are in the file ────────────────────────
EXPECTED_VALUES = [
    # (Study ID, column, expected value)
    (169, _YCOL[1], 1),
    ( 78, _YCOL[1], 3),
    (250, _YCOL[1], 1),        # UT Austin / Patient 8, formerly Study ID 10
    (167, _YCOL[1], 2),
    ( 59, _YCOL[2], 4),
    ( 78, _MCOL[5], 0),
    ( 78, _MCOL[8], 0),
    (104, _SECOL['None'],  'Unchecked'),
    (214, _SECOL['None'],  'Unchecked'),
    (127, _SECOL['Other'], 'Unchecked'),
]
_by_id = df_raw.set_index('Study ID')
def _matches(have, want):
    if isinstance(want, str):
        return str(have).strip() == want
    return pd.to_numeric(have, errors='coerce') == want

_bad = [(sid, col[:60], _by_id.at[sid, col], want)
        for sid, col, want in EXPECTED_VALUES
        if not _matches(_by_id.at[sid, col], want)]
assert not _bad, f"corrections missing from the data file: {_bad}"
print(f"✔ All {len(EXPECTED_VALUES)} requested corrections are present in the data file")

# ── Patient exclusions ──────────────────────────────────────────────────────
EXCLUDE_PLAIN  = [15, 66, 58, 37, 157, 110, 93, 50, 52, 84, 119, 55]
EXCLUDE_YR1_NA = [9, 22, 30, 32, 33, 35, 36, 38, 39, 41,
                  79, 92, 96, 105, 111, 125, 166]

_yr1_raw = df_raw[_YCOL[1]].astype(str).str.strip()
_yr1_unknown = sorted(int(v) for v in
                      df_raw.loc[_yr1_raw == 'Unknown/not reported', 'Study ID'])
assert _yr1_unknown == sorted(EXCLUDE_YR1_NA), \
    "the 'Unknown/not reported' rule no longer reproduces the team's red list"
print(f"✔ 'Unknown/not reported' Year-1 rule reproduces the team's red list "
      f"({len(_yr1_unknown)} patients)")

# ===========================================================================
#  PATIENT FLOW — every row of the dataset, from collected to analysed
# ===========================================================================
# Each criterion is evaluated on all rows, then applied in FLOW_ORDER; a
# patient is counted under the first criterion they fail. The criteria overlap,
# so the per-step numbers depend on the order (the "alone" column does not).
# The same rules are applied for real in Cells 3 and 6; Cell 6 asserts that it
# lands on the same n.
# ---------------------------------------------------------------------------

def _parse_dose_flow(rawv, otherv):
    if pd.isna(rawv):
        return np.nan
    v = str(rawv).strip()
    if v == '0':
        return 0.0
    if 'mg' in v:
        return float(v.replace(' mg', ''))
    if v == 'Other':
        return float(otherv) if pd.notna(otherv) else np.nan
    return np.nan

_on3 = pd.Series(True, index=df_raw.index)
for _m in range(1, 4):
    _dc = [c for c in df_raw.columns
           if c.startswith(f'Acitretin average monthly dosing: Month {_m}')][0]
    if _m == 1:
        _oc = [c for c in df_raw.columns if 'If Other, round average dose' in c
               and not any(c.endswith(f'.{i}') for i in range(1, 24))][0]
    else:
        _oc = [c for c in df_raw.columns if 'If Other, round average dose' in c
               and c.endswith(f'.{_m - 1}')][0]
    _oth = pd.to_numeric(df_raw[_oc], errors='coerce')
    _on3 &= pd.Series([_parse_dose_flow(a, b) for a, b in zip(df_raw[_dc], _oth)],
                      index=df_raw.index) > 0

_pre_raw = pd.DataFrame({f'yr{y}': pd.to_numeric(df_raw[_YCOL[y]], errors='coerce')
                         for y in range(1, 11)}, index=df_raw.index)

FLOW_CRITERIA = {
    'Survey not complete':
        df_raw['Complete?'] != 'Complete',
    'DKBeta site':
        df_raw['Site ID'].str.strip().str.lower() == 'dkbeta',
    'No documented SCC in the 10 years before treatment':
        ~(_pre_raw.sum(axis=1, min_count=1) > 0),
    'Not on acitretin for the first 3 consecutive months':
        ~_on3,
    'Year-1 pre-treatment SCC count unknown/not reported':
        _pre_raw['yr1'].isna(),
    'Excluded by the study team on review (Study ID list)':
        df_raw['Study ID'].isin(EXCLUDE_PLAIN),
}
FLOW_ORDER = list(FLOW_CRITERIA)

_remaining = pd.Series(True, index=df_raw.index)
_flow_rows = [{'Step': 'Patients in the dataset', 'n excluded': '',
               'n excluded (criterion alone)': '', 'n remaining': len(df_raw),
               'Study IDs excluded at this step': ''}]
for _name in FLOW_ORDER:
    _hit = _remaining & FLOW_CRITERIA[_name]
    _remaining &= ~FLOW_CRITERIA[_name]
    _flow_rows.append({
        'Step': _name,
        'n excluded': int(_hit.sum()),
        'n excluded (criterion alone)': int(FLOW_CRITERIA[_name].sum()),
        'n remaining': int(_remaining.sum()),
        'Study IDs excluded at this step':
            ', '.join(str(int(v)) for v in sorted(df_raw.loc[_hit, 'Study ID'])),
    })
patient_flow = pd.DataFrame(_flow_rows)
FLOW_FINAL_N = int(_remaining.sum())
FLOW_COHORT_IDS = set(df_raw.loc[_remaining, 'Study ID'].astype(int))

print("\n" + "=" * 74)
print("  PATIENT FLOW")
print("=" * 74)
with pd.option_context('display.width', 200, 'display.max_colwidth', 60):
    print(patient_flow.drop(columns='Study IDs excluded at this step').to_string(index=False))
patient_flow.to_csv('./table_patient_flow.csv', index=False)
print("\n  ✓ Saved to ./table_patient_flow.csv")

# ── The file's own bookkeeping columns must agree with the rules ────────────
_flag_excl = set(df_raw.loc[df_raw['excluded_20260815'] == 'Yes', 'Study ID'].astype(int))
_flag_in   = set(df_raw.loc[df_raw['in_final_cohort'] == 'Yes', 'Study ID'].astype(int))
assert _flag_excl == set(EXCLUDE_PLAIN) | set(EXCLUDE_YR1_NA), \
    f"excluded_20260815 disagrees with the exclusion lists: {sorted(_flag_excl ^ (set(EXCLUDE_PLAIN) | set(EXCLUDE_YR1_NA)))}"
assert _flag_in == FLOW_COHORT_IDS, \
    f"in_final_cohort disagrees with the rules: {sorted(_flag_in ^ FLOW_COHORT_IDS)}"
print("✔ excluded_20260815 and in_final_cohort columns agree with the rules")

# ── Apply the by-ID exclusions ──────────────────────────────────────────────
_n_before = len(df_raw)
df_raw = df_raw.loc[~df_raw['Study ID'].isin(EXCLUDE_PLAIN)].copy().reset_index(drop=True)
print(f"\nDropped {_n_before - len(df_raw)} rows listed by the study team "
      f"({_n_before} → {len(df_raw)} rows); the Year-1 rule is applied in Cell 6")

In [ ]:
# %% Cell 3 — Remove incomplete surveys and DKBeta site
df = df_raw[
    (df_raw['Complete?'] == 'Complete') &
    (~df_raw['Site ID'].isin(['DKBeta', 'dkbeta']))
].copy().reset_index(drop=True)
print(f"After removing Incomplete & DKBeta: {df.shape[0]} rows")

In [ ]:
# %% Cell 4 — Parse columns
age_col = [c for c in df.columns if 'Age' in c][0]
wt_col  = [c for c in df.columns if 'Weight' in c][0]
nic_col = [c for c in df.columns if 'nicotinamide' in c and 'date' not in c.lower()][0]
ft_col  = [c for c in df.columns if 'field therapy used' in c][0]

df['age']    = pd.to_numeric(df[age_col], errors='coerce')
df['weight'] = pd.to_numeric(df[wt_col], errors='coerce')
df['sex']    = df['Sex '].str.strip()
df['immune'] = df['Immune status'].str.strip()
df['site']   = df['Site ID'].str.strip().str.upper()
df['nicotinamide']  = (df[nic_col].str.strip() == 'Yes').astype(int)
df['field_therapy'] = (df[ft_col].str.strip() == 'Yes').astype(int)
df['is_immunosuppressed'] = (df['immune'] == 'Immunosuppressed').astype(int)

# Immunosuppression subtypes
df['is_sotr'] = (df[[c for c in df.columns if 'SOTR)' in c]].values == 'Checked').any(axis=1)
df['is_cll']  = (df[[c for c in df.columns if 'CLL)' in c]].values == 'Checked').any(axis=1)

# Pre-treatment yearly SCC
pre_scc = pd.DataFrame(index=df.index)
for yr in range(1, 11):
    col = [c for c in df.columns if f'Year #{yr} before' in c and 'invasive SCCs' in c][0]
    pre_scc[f'yr{yr}'] = pd.to_numeric(df[col], errors='coerce')

# Monthly post-treatment dose columns
dose_raw_cols, dose_other_cols = {}, {}
for m in range(1, 25):
    dc = [c for c in df.columns if c.startswith(f'Acitretin average monthly dosing: Month {m}')][0]
    dose_raw_cols[m] = dc
    if m == 1:
        oc = [c for c in df.columns if 'If Other, round average dose' in c
              and not any(c.endswith(f'.{i}') for i in range(1, 24))][0]
    else:
        oc_list = [c for c in df.columns if 'If Other, round average dose' in c and c.endswith(f'.{m-1}')]
        oc = oc_list[0] if oc_list else None
    dose_other_cols[m] = oc

# Monthly post-treatment SCC
scc_month = pd.DataFrame(index=df.index)
for m in range(1, 25):
    sc = [c for c in df.columns if f'month #{m}' in c and 'invasive SCCs' in c]
    if sc:
        scc_month[f'm{m}'] = pd.to_numeric(df[sc[0]], errors='coerce')

# Parse dose values
def parse_dose(raw_val, other_val):
    if pd.isna(raw_val): return np.nan
    v = str(raw_val).strip()
    if v == '0': return 0.0
    if 'mg' in v: return float(v.replace(' mg', ''))
    if v == 'Other': return float(other_val) if pd.notna(other_val) else np.nan
    return np.nan

dose_mg = pd.DataFrame(index=df.index)
for m in range(1, 25):
    raw = df[dose_raw_cols[m]]
    oc = dose_other_cols[m]
    oth = pd.to_numeric(df[oc], errors='coerce') if oc else pd.Series(np.nan, index=df.index)
    dose_mg[f'm{m}'] = [parse_dose(r, o) for r, o in zip(raw, oth)]

In [ ]:
# %% Cell 5 — Censoring: censor at first month where dose is 0 OR NaN
first_off = pd.Series(np.nan, index=df.index, dtype=float)
for idx in df.index:
    for m in range(1, 25):
        v = dose_mg.loc[idx, f'm{m}']
        if v == 0 or pd.isna(v):
            first_off[idx] = m
            break

df['first_off_month'] = first_off
df['months_on_drug'] = first_off.fillna(25) - 1

dose_mg_cens = dose_mg.copy()
scc_month_cens = scc_month.copy()
for idx in df.index:
    fom = first_off[idx]
    if pd.notna(fom):
        for m in range(int(fom), 25):
            dose_mg_cens.loc[idx, f'm{m}'] = np.nan
            scc_month_cens.loc[idx, f'm{m}'] = np.nan

# Impute SCC NaN as 0 within the on-drug window
# (scattered missing SCC when dose is positive = no SCC occurred)
imputed = 0
for idx in df.index:
    last_on = int(df.loc[idx, 'months_on_drug'])
    for m in range(1, last_on + 1):
        if pd.isna(scc_month_cens.loc[idx, f'm{m}']):
            scc_month_cens.loc[idx, f'm{m}'] = 0
            imputed += 1

In [ ]:
# %% Cell 6 — Filter: (a) on drug for first 3 months, (b) had SCC in prior window,
#                     (c) Year-1 pre-treatment SCC count reported
# ── (a) On drug for the first 3 months ───────────────────────────────────
mask_3m_on = pd.Series(True, index=df.index)
for m in range(1, 4):
    mask_3m_on = mask_3m_on & (dose_mg_cens[f'm{m}'] > 0)

# ── (b) Exclude patients with 0 SCCs in the X years prior ────────────────
# X = 0 disables this exclusion (original behavior).
# Study team 2026-08-15: use the 10-year window — a patient is kept only if
# they had ≥1 biopsy-proven invasive SCC in the 10 years before acitretin.
EXCLUDE_ZERO_PRIOR_YEARS = 10
assert 0 <= EXCLUDE_ZERO_PRIOR_YEARS <= 10, "pre_scc only holds years 1–10"

if EXCLUDE_ZERO_PRIOR_YEARS > 0:
    _pre_cols = [f'yr{y}' for y in range(1, EXCLUDE_ZERO_PRIOR_YEARS + 1)]
    _pre_sum  = pre_scc[_pre_cols].sum(axis=1, min_count=1)   # NaN only if ALL years missing
    # keep: had ≥1 SCC in window, OR window entirely unobserved (can't assess → don't drop)
    mask_prior_scc = (_pre_sum > 0) | _pre_sum.isna()
    # STRICTER alternative (also drop anyone without a positive OBSERVED year, incl. all-missing):
    mask_prior_scc = (_pre_sum > 0)
else:
    mask_prior_scc = pd.Series(True, index=df.index)

# ── (c) Exclude "Unknown/not reported" Year-1 pre-treatment SCC ──────────
# Study team 2026-08-15. Written as a rule rather than a list of Study IDs;
# Cell 2b verifies it reproduces their red-highlighted list exactly.
# The 12 by-ID exclusions were already dropped in Cell 2b.
mask_yr1_known = pre_scc['yr1'].notna()

keep = mask_3m_on & mask_prior_scc & mask_yr1_known

n_before    = len(df)
n_fail_3m   = int((~mask_3m_on).sum())
n_zero_pre  = int((mask_3m_on & ~mask_prior_scc).sum())
n_yr1_unk   = int((mask_3m_on & mask_prior_scc & ~mask_yr1_known).sum())

df             = df.loc[keep].copy().reset_index(drop=True)
dose_mg_cens   = dose_mg_cens.loc[keep].copy().reset_index(drop=True)
scc_month_cens = scc_month_cens.loc[keep].copy().reset_index(drop=True)
pre_scc        = pre_scc.loc[keep].copy().reset_index(drop=True)

print(f"SCC NaN imputed as 0 within on-drug window: {imputed} patient-months")
print(f"Excluded — not on drug ≥3 months:            {n_fail_3m}")
if EXCLUDE_ZERO_PRIOR_YEARS > 0:
    print(f"Excluded — 0 SCCs in prior {EXCLUDE_ZERO_PRIOR_YEARS} year(s): {n_zero_pre}")
print(f"Excluded — Year-1 pre-tx SCC unknown:        {n_yr1_unk}")
print(f"After censoring + ≥3 months + prior-SCC + Year-1 filters: n={len(df)} patients "
      f"(from {n_before})")
print(f"Months on drug — median: {df['months_on_drug'].median():.0f}, "
      f"mean: {df['months_on_drug'].mean():.1f}, "
      f"range: {df['months_on_drug'].min():.0f}–{df['months_on_drug'].max():.0f}")

# Same n and same patients as the patient flow computed in Cell 2b
assert len(df) == FLOW_FINAL_N, (len(df), FLOW_FINAL_N)
assert set(df['Study ID'].astype(int)) == FLOW_COHORT_IDS

In [ ]:
# %% Cell 7 — Post-censoring diagnostics

print("=" * 70)
print("DIAGNOSTIC 1: Dataset overview")
print("=" * 70)
print(f"  Patients: {len(df)}")
print(f"  Months on drug — median: {df['months_on_drug'].median():.0f}, "
      f"mean: {df['months_on_drug'].mean():.1f}, "
      f"range: {df['months_on_drug'].min():.0f}–{df['months_on_drug'].max():.0f}")

print(f"\n  Duration distribution:")
for cutoff in [3, 6, 12, 18, 24]:
    n = (df['months_on_drug'] >= cutoff).sum()
    print(f"    ≥{cutoff:2d} months: {n:3d} ({100*n/len(df):.0f}%)")

print("\n" + "=" * 70)
print("DIAGNOSTIC 2: No NaN in on-drug window (SCC)")
print("=" * 70)
scc_nan_in_window = 0
for idx in df.index:
    last_on = int(df.loc[idx, 'months_on_drug'])
    for m in range(1, last_on + 1):
        if pd.isna(scc_month_cens.loc[idx, f'm{m}']):
            scc_nan_in_window += 1
print(f"  SCC NaN within on-drug window: {scc_nan_in_window}  (should be 0)")

print("\n" + "=" * 70)
print("DIAGNOSTIC 3: No NaN in on-drug window (dose)")
print("=" * 70)
dose_nan_in_window = 0
for idx in df.index:
    last_on = int(df.loc[idx, 'months_on_drug'])
    for m in range(1, last_on + 1):
        if pd.isna(dose_mg_cens.loc[idx, f'm{m}']):
            dose_nan_in_window += 1
print(f"  Dose NaN within on-drug window: {dose_nan_in_window}  (should be 0)")

print("\n" + "=" * 70)
print("DIAGNOSTIC 4: All on-drug doses are > 0")
print("=" * 70)
zero_dose_in_window = 0
for idx in df.index:
    last_on = int(df.loc[idx, 'months_on_drug'])
    for m in range(1, last_on + 1):
        if dose_mg_cens.loc[idx, f'm{m}'] == 0:
            zero_dose_in_window += 1
print(f"  Zero doses within on-drug window: {zero_dose_in_window}  (should be 0)")

print("\n" + "=" * 70)
print("DIAGNOSTIC 5: Everything is NaN AFTER on-drug window")
print("=" * 70)
non_nan_after = 0
for idx in df.index:
    last_on = int(df.loc[idx, 'months_on_drug'])
    for m in range(last_on + 1, 25):
        if pd.notna(scc_month_cens.loc[idx, f'm{m}']):
            non_nan_after += 1
        if pd.notna(dose_mg_cens.loc[idx, f'm{m}']):
            non_nan_after += 1
print(f"  Non-NaN values after on-drug window: {non_nan_after}  (should be 0)")

print("\n" + "=" * 70)
print("DIAGNOSTIC 6: Months 1-3 all have positive dose (filter check)")
print("=" * 70)
bad_m1_3 = 0
for m in range(1, 4):
    bad = (dose_mg_cens[f'm{m}'].isna() | (dose_mg_cens[f'm{m}'] <= 0)).sum()
    bad_m1_3 += bad
print(f"  Patients with missing/zero dose in months 1-3: {bad_m1_3}  (should be 0)")

print("\n" + "=" * 70)
print("DIAGNOSTIC 7: Dose sequence patterns (censored)")
print("=" * 70)
patterns = {}
for idx in df.index:
    seq = ''
    for m in range(1, 25):
        v = dose_mg_cens.loc[idx, f'm{m}']
        if pd.isna(v):
            seq += '.'
        elif v == 0:
            seq += '0'
        else:
            seq += '+'
    patterns[idx] = seq

pat_counts = Counter(patterns.values())
print(f"  Unique patterns: {len(pat_counts)}")
print(f"\n  Top 15 patterns (+ = on drug, . = censored/off):")
for pat, cnt in pat_counts.most_common(15):
    print(f"    {pat}  (n={cnt})")

# Check: no pattern should have a + after a .
bad_patterns = {p: c for p, c in pat_counts.items() if '.+' in p}
print(f"\n  Patterns with + after . (should be 0): {len(bad_patterns)}")
for p, c in bad_patterns.items():
    print(f"    {p}  (n={c})")

print("\n" + "=" * 70)
print("DIAGNOSTIC 8: SCC sanity checks")
print("=" * 70)
total_scc = scc_month_cens.sum(axis=1, min_count=1)
print(f"  Total on-drug SCCs: {total_scc.sum():.0f}")
print(f"  Patients with 0 SCCs: {(total_scc == 0).sum()} ({100*(total_scc == 0).sum()/len(df):.0f}%)")
print(f"  Max SCCs for one patient: {total_scc.max():.0f}")
print(f"  Mean SCCs per patient: {total_scc.mean():.2f}")
print(f"  SCC per patient-month: {total_scc.sum() / scc_month_cens.notna().sum().sum():.3f}")

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13.5, 11))
fig.subplots_adjust(hspace=0.50, wspace=0.32)

n_total = len(df)

def _clean(ax, ylabel='Count'):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')
    ax.spines['left'].set_linewidth(2.0)
    ax.spines['bottom'].set_linewidth(2.0)
    ax.tick_params(colors='#333333', width=1.5, labelsize=10)
    ax.grid(False)
    if ylabel:
        ax.set_ylabel(ylabel, fontweight='bold')

def _bar_labels(ax, bars, vals, total=None, fs=9):
    for b, v in zip(bars, vals):
        txt = f'{v}' + (f'\n({100*v/total:.0f}%)' if total else '')
        ax.text(b.get_x() + b.get_width()/2, v + max(vals)*0.03,
                txt, ha='center', va='bottom', fontsize=fs, fontweight='bold', color='#333333')

# ── Row 1, Col 1: Age ────────────────────────────────────────
ax = axes[0, 0]
ages = df['age'].dropna()
bins_a = np.arange(ages.min() // 5 * 5, ages.max() + 6, 5)
ax.hist(ages, bins=bins_a, color=C['mauve'], edgecolor='#2B2D42', lw=1.8, alpha=0.90)
kde_x = np.linspace(ages.min() - 5, ages.max() + 5, 200)
kde_a = stats.gaussian_kde(ages, bw_method=0.3)
ax.plot(kde_x, kde_a(kde_x) * len(ages) * (bins_a[1] - bins_a[0]),
        color=C['navy'], lw=2.5, alpha=0.6)
ax.axvline(ages.median(), color='k', ls='--', lw=1.8)
ax.text(ages.median() - 11, ax.get_ylim()[1] * 0.92,
        f'Median {ages.median():.0f}\nIQR {ages.quantile(.25):.0f}–{ages.quantile(.75):.0f}',
        fontsize=7.5, color='#333', va='top')
ax.set_xlabel('Age at acitretin initiation (years)', fontweight='bold')
ax.set_title('Age', fontweight='bold')
_clean(ax)

# ── Row 1, Col 2: Sex ────────────────────────────────────────
ax = axes[0, 1]
sex_order = ['M', 'F']
sex_vals = [df['sex'].value_counts().get(s, 0) for s in sex_order]
colors_sex = [C['pink'], C['mauve']]
bars = ax.bar(sex_order, sex_vals, width=0.55,
              color=colors_sex, edgecolor='#2B2D42', lw=1.8)
_bar_labels(ax, bars, sex_vals, total=n_total)
ax.set_ylim(0, max(sex_vals) * 1.18)
ax.set_title('Sex', fontweight='bold')
_clean(ax)

# ── Row 1, Col 3: Weight ─────────────────────────────────────
ax = axes[0, 2]
wts = df.loc[df['weight'] > 10, 'weight'].dropna()
bins_w = np.arange(40, wts.max() + 11, 10)
ax.hist(wts, bins=bins_w, color=C['mauve'], edgecolor='#2B2D42', lw=1.8, alpha=0.90)
kde_xw = np.linspace(wts.min() - 10, wts.max() + 10, 200)
kde_w = stats.gaussian_kde(wts, bw_method=0.3)
ax.plot(kde_xw, kde_w(kde_xw) * len(wts) * (bins_w[1] - bins_w[0]),
        color=C['navy'], lw=2.5, alpha=0.6)
ax.set_xlabel('Weight (kg)', fontweight='bold')
ax.set_title('Weight', fontweight='bold')
_clean(ax)

# ── Row 2, Col 1: Immune status ──────────────────────────────
ax = axes[1, 0]
imm_order = ['Immunocompetent', 'Immunosuppressed']
imm_vals = [df['immune'].value_counts().get(k, 0) for k in imm_order]
imm_display = ['Immuno-\ncompetent', 'Immuno-\ncompromised']
colors_imm = [C['pink'], C['mauve']]
bars = ax.bar(imm_display, imm_vals, width=0.55,
              color=colors_imm, edgecolor='#2B2D42', lw=1.8)
_bar_labels(ax, bars, imm_vals, total=n_total)
ax.set_ylim(0, max(imm_vals) * 1.18)
ax.set_title('Immune status', fontweight='bold')
_clean(ax)

# ── Row 2, Col 2: Nicotinamide × Field therapy ───────────────
ax = axes[1, 1]
df['_nic_ft'] = 'Neither'
df.loc[(df['nicotinamide'] == 1) & (df['field_therapy'] == 0), '_nic_ft'] = 'Nic only'
df.loc[(df['nicotinamide'] == 0) & (df['field_therapy'] == 1), '_nic_ft'] = 'FT only'
df.loc[(df['nicotinamide'] == 1) & (df['field_therapy'] == 1), '_nic_ft'] = 'Both'

cat_order  = ['Neither', 'Nic only', 'FT only', 'Both']
cat_colors = [C['pink'],  C['mauve'], C['plum'], C['purple']]
cat_counts = [df['_nic_ft'].value_counts().get(c, 0) for c in cat_order]

bars = ax.bar(cat_order, cat_counts, width=0.6,
              color=cat_colors, edgecolor='#2B2D42', lw=1.8)
_bar_labels(ax, bars, cat_counts, total=n_total)
ax.set_ylim(0, max(cat_counts) * 1.22)
ax.set_title('Adjunctive therapies')
ax.set_xlabel('Nicotinamide (Nic) / Field therapy (FT)')
_clean(ax)

# ── Row 2, Col 3: Treatment duration ─────────────────────────
ax = axes[1, 2]
dur = df['months_on_drug']
bins_d = np.arange(2.5, 25.5, 1)
ax.hist(dur, bins=bins_d, color=C['mauve'], edgecolor='#2B2D42', lw=1.5, alpha=0.90)
ax.set_xlabel('Months on acitretin', fontweight='bold')
ax.set_title('Treatment duration', fontweight='bold')
ax.set_xticks([3, 6, 9, 12, 15, 18, 21, 24])
_clean(ax)

# ── Row 3, Col 1: Immunosuppression type ─────────────────────
ax = axes[2, 0]
types = {'SOTR': df['is_sotr'].sum(),
         'CLL': df['is_cll'].sum(),
         'Other': (df[[c for c in df.columns if "Immunosuppression type" in c and "Other)" in c]].values == 'Checked').sum(),
         'Immuno-\ncompetent': (df['immune'] == 'Immunocompetent').sum()}
type_colors = [C['pink'], C['mauve'], C['plum'], C['purple']]
bars = ax.barh(list(types.keys()), list(types.values()),
               color=type_colors, edgecolor='#2B2D42', lw=1.8)
for b, v in zip(bars, types.values()):
    ax.text(v + 1, b.get_y() + b.get_height()/2, str(v), va='center', fontsize=10, fontweight='bold', color='#333333')
ax.set_xlabel('Number of patients', fontweight='bold')
ax.set_title('Immunocompromise type')
ax.invert_yaxis()
_clean(ax, ylabel=None)

# ── Row 3, Col 2: SOTR organ breakdown ───────────────────────
ax = axes[2, 1]
organs = {}
for label in ['Kidney', 'Liver', 'Heart', 'Lung', 'Small Bowel/GI/pancreas']:
    col = [c for c in df.columns if 'SOTR, what type' in c and label in c][0]
    organs[label] = (df[col] == 'Checked').sum()
organs = dict(sorted(organs.items(), key=lambda x: x[1], reverse=True))
organ_color_map = {'Lung': C['navy']}
organ_colors = [organ_color_map.get(k, [C['pink'], C['mauve'], C['plum'], C['purple']].pop(0))
                for k in organs.keys()]
# cleaner approach:
_palette_idx = 0
_palette = [C['pink'], C['mauve'], C['plum'], C['purple']]
organ_colors = []
for k in organs.keys():
    if k == 'Lung':
        organ_colors.append(C['navy'])
    else:
        organ_colors.append(_palette[_palette_idx])
        _palette_idx += 1

bars = ax.barh(list(organs.keys()), list(organs.values()),
               color=organ_colors, edgecolor='#2B2D42', lw=1.8, alpha=0.90)
for b, v in zip(bars, organs.values()):
    ax.text(v + 0.5, b.get_y() + b.get_height()/2, str(v), va='center', fontsize=10, fontweight='bold', color='#333333')
ax.set_xlabel('Number of patients', fontweight='bold')
ax.set_title('SOTR organ type')
ax.invert_yaxis()
_clean(ax, ylabel=None)

# ── Row 3, Col 3: Pre-treatment SCC burden (Year 1) ──────────
ax = axes[2, 2]
yr1 = pre_scc['yr1'].dropna()
max_scc = int(yr1.max())
bins_scc = np.arange(-0.5, max_scc + 1.5, 1)
ax.hist(yr1, bins=bins_scc, color=C['mauve'], edgecolor='#2B2D42', lw=1.8, alpha=0.90)
ax.axvline(yr1.median(), color='k', ls='--', lw=1.8)
ax.text(yr1.median() + 0.5, ax.get_ylim()[1] * 0.92,
        f'Median {yr1.median():.0f}\nIQR {yr1.quantile(.25):.0f}–{yr1.quantile(.75):.0f}\nMean {yr1.mean():.1f}',
        fontsize=7.5, color='#333', va='top')
ax.set_xlabel('Invasive SCCs in year before acitretin', fontweight='bold')
ax.set_title('Pre-treatment SCC burden (Year −1)')
ax.set_xticks(range(0, max_scc + 1, 2))
_clean(ax)

fig.suptitle(f'Cohort baseline characteristics  (n = {n_total})',
             fontsize=16, fontweight='bold', y=1.02)
plt.savefig(os.path.join(FIGURE_DIR, 'fig01_demographics.png'), bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.subplots_adjust(hspace=0.45, wspace=0.32)

n_total = len(df)

def _clean(ax, ylabel='Count'):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')
    ax.spines['left'].set_linewidth(2.0)
    ax.spines['bottom'].set_linewidth(2.0)
    ax.tick_params(colors='#333333', width=1.5, labelsize=10)
    ax.grid(False)
    if ylabel:
        ax.set_ylabel(ylabel, fontweight='bold')

def _bar_labels(ax, bars, vals, total=None, fs=9):
    for b, v in zip(bars, vals):
        txt = f'{v}' + (f'\n({100*v/total:.0f}%)' if total else '')
        ax.text(b.get_x() + b.get_width()/2, v + max(vals)*0.03,
                txt, ha='center', va='bottom', fontsize=fs, fontweight='bold', color='#333333')

# ── (0,0) Age ─────────────────────────────────────────────────
ax = axes[0, 0]
ages = df['age'].dropna()
bins_a = np.arange(ages.min() // 5 * 5, ages.max() + 6, 5)
ax.hist(ages, bins=bins_a, color=C['mauve'], edgecolor='#2B2D42', lw=1.8, alpha=0.90)
kde_x = np.linspace(ages.min() - 5, ages.max() + 5, 200)
kde_a = stats.gaussian_kde(ages, bw_method=0.3)
ax.plot(kde_x, kde_a(kde_x) * len(ages) * (bins_a[1] - bins_a[0]),
        color=C['navy'], lw=2.5, alpha=0.6)
ax.axvline(ages.median(), color='k', ls='--', lw=1.8)
ax.text(ages.median() - 11, ax.get_ylim()[1] * 0.92,
        f'Median {ages.median():.0f}\nIQR {ages.quantile(.25):.0f}–{ages.quantile(.75):.0f}',
        fontsize=7.5, color='#333', va='top')
ax.set_xlabel('Age at acitretin initiation (years)', fontweight='bold')
ax.set_title('Age', fontweight='bold')
_clean(ax)

# ── (0,1) Sex ─────────────────────────────────────────────────
ax = axes[0, 1]
sex_order = ['M', 'F']
sex_vals = [df['sex'].value_counts().get(s, 0) for s in sex_order]
colors_sex = [C['pink'], C['mauve']]
bars = ax.bar(sex_order, sex_vals, width=0.55,
              color=colors_sex, edgecolor='#2B2D42', lw=1.8)
_bar_labels(ax, bars, sex_vals, total=n_total)
ax.set_ylim(0, max(sex_vals) * 1.18)
ax.set_title('Sex', fontweight='bold')
_clean(ax)

# ── (1,0) Weight ──────────────────────────────────────────────
ax = axes[1, 0]
wts = df.loc[df['weight'] > 10, 'weight'].dropna()
bins_w = np.arange(40, wts.max() + 11, 10)
ax.hist(wts, bins=bins_w, color=C['mauve'], edgecolor='#2B2D42', lw=1.8, alpha=0.90)
kde_xw = np.linspace(wts.min() - 10, wts.max() + 10, 200)
kde_w = stats.gaussian_kde(wts, bw_method=0.3)
ax.plot(kde_xw, kde_w(kde_xw) * len(wts) * (bins_w[1] - bins_w[0]),
        color=C['navy'], lw=2.5, alpha=0.6)
ax.set_xlabel('Weight (kg)', fontweight='bold')
ax.set_title('Weight', fontweight='bold')
_clean(ax)

# ── (1,1) Treatment duration ─────────────────────────────────
ax = axes[1, 1]
dur = df['months_on_drug']
bins_d = np.arange(2.5, 25.5, 1)
ax.hist(dur, bins=bins_d, color=C['mauve'], edgecolor='#2B2D42', lw=1.5, alpha=0.90)
ax.set_xlabel('Months on acitretin', fontweight='bold')
ax.set_title('Treatment duration', fontweight='bold')
ax.set_xticks([3, 6, 9, 12, 15, 18, 21, 24])
_clean(ax)

fig.suptitle(f'Cohort baseline characteristics  (n = {n_total})',
             fontsize=16, fontweight='bold', y=1.05)
plt.savefig(os.path.join(FIGURE_DIR, 'fig01a_demographics.png'), bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.subplots_adjust(wspace=0.35)

# ── (0) Age by sex ───────────────────────────────────────────
ax = axes[0]
ages_m = df.loc[df['sex'] == 'M', 'age'].dropna()
ages_f = df.loc[df['sex'] == 'F', 'age'].dropna()
ages_all = df['age'].dropna()

bins_a = np.arange(ages_all.min() // 5 * 5, ages_all.max() + 6, 5)

counts_m, _ = np.histogram(ages_m, bins=bins_a)
counts_f, _ = np.histogram(ages_f, bins=bins_a)
bin_centers = (bins_a[:-1] + bins_a[1:]) / 2
width = (bins_a[1] - bins_a[0]) * 0.85

label_m = (f'Male (n={len(ages_m)})\n'
           f'  Mean {ages_m.mean():.1f}, Median {ages_m.median():.0f}, '
           f'IQR {ages_m.quantile(.25):.0f}–{ages_m.quantile(.75):.0f}')
label_f = (f'Female (n={len(ages_f)})\n'
           f'  Mean {ages_f.mean():.1f}, Median {ages_f.median():.0f}, '
           f'IQR {ages_f.quantile(.25):.0f}–{ages_f.quantile(.75):.0f}')

ax.bar(bin_centers, counts_m, width=width,
       color=C['mauve'], edgecolor='#2B2D42', lw=1.8, alpha=0.90, label=label_m)
ax.bar(bin_centers, counts_f, width=width, bottom=counts_m,
       color=C['pink'], edgecolor='#2B2D42', lw=1.8, alpha=0.90, label=label_f)

ax.axvline(ages_all.median(), color='k', ls='--', lw=1.8)
ax.text(ages_all.median() - 11, ax.get_ylim()[1] * 0.92,
        f'Median {ages_all.median():.0f}\nIQR {ages_all.quantile(.25):.0f}–{ages_all.quantile(.75):.0f}\nMean {ages_all.mean():.1f}',
        fontsize=7.5, color='#333', va='top')
ax.set_xlabel('Age at acitretin initiation (years)', fontweight='bold')
ax.set_ylabel('Number of patients', fontweight='bold')
ax.set_title('Age by sex', fontweight='bold')
ax.legend(frameon=True, fontsize=7, loc='upper left', fancybox=True, edgecolor='#cccccc')
_clean(ax, ylabel='Number of patients')

# ── (1) Immunosuppression type – SOTR split by organ ─────────
ax = axes[1]
from matplotlib.patches import Patch

imm_order = ['SOTR', 'CLL', 'Other', 'Immuno-\ncompetent']
imm_totals = {
    'SOTR': int(df['is_sotr'].sum()),
    'CLL': int(df['is_cll'].sum()),
    'Other': int((df[[c for c in df.columns if "Immunosuppression type" in c and "Other)" in c]].values == 'Checked').sum()),
    'Immuno-\ncompetent': int((df['immune'] == 'Immunocompetent').sum())
}

organ_labels = ['Kidney', 'Liver', 'Heart', 'Lung', 'Small Bowel/GI/pancreas']
_pal5 = [C['pink'], C['mauve'], C['plum'], C['purple'], C['navy']]
organ_colors = {lab: _pal5[i] for i, lab in enumerate(organ_labels)}

left = 0
for label in organ_labels:
    col = [c for c in df.columns if 'SOTR, what type' in c and label in c][0]
    n = int((df[col] == 'Checked').sum())
    if n > 0:
        ax.barh('SOTR', n, left=left, color=organ_colors[label],
                edgecolor='#2B2D42', lw=1.8, alpha=0.90)
        left += n
sotr_width = left

for cat in ['CLL', 'Other', 'Immuno-\ncompetent']:
    ax.barh(cat, imm_totals[cat], color=C['slate'],
            edgecolor='#2B2D42', lw=1.8, alpha=0.90)

bar_widths = {'SOTR': sotr_width}
bar_widths.update({cat: imm_totals[cat] for cat in ['CLL', 'Other', 'Immuno-\ncompetent']})
max_width = max(bar_widths.values())
for cat in imm_order:
    v = imm_totals[cat]
    ax.text(bar_widths[cat] + max_width * 0.03, cat,
            str(v), va='center', fontsize=10, fontweight='bold', color='#333333')

handles = [Patch(facecolor=organ_colors[lab], edgecolor='#2B2D42',
                 label=lab.replace('Small Bowel/GI/pancreas', 'Small Bowel/\nGI/Pancreas'))
           for lab in organ_labels]
handles.append(Patch(facecolor=C['slate'], edgecolor='#2B2D42', label='Non-SOTR'))
ax.legend(handles=handles, frameon=True, fontsize=7, loc='lower right',
          fancybox=True, edgecolor='#cccccc')
ax.set_xlim(0, max_width * 1.30)
ax.set_xlabel('Number of patients', fontweight='bold')
ax.set_title('Immunosuppression type', fontweight='bold')
ax.invert_yaxis()
_clean(ax, ylabel=None)

# ── (2) Adjunctive therapies with FT breakdown ───────────────
ax = axes[2]
df['_nic_ft'] = 'Neither'
df.loc[(df['nicotinamide'] == 1) & (df['field_therapy'] == 0), '_nic_ft'] = 'Nic only'
df.loc[(df['nicotinamide'] == 0) & (df['field_therapy'] == 1), '_nic_ft'] = 'FT only'
df.loc[(df['nicotinamide'] == 1) & (df['field_therapy'] == 1), '_nic_ft'] = 'Both'

cat_order = ['Both', 'FT only', 'Nic only', 'Neither']
cat_counts = {c: df['_nic_ft'].value_counts().get(c, 0) for c in cat_order}

ft_labels = ['5FU', '5FU/calcipotriene', 'Imiquimod', 'PDT', 'Chemical peel']
ft_cols = {}
for label in ft_labels:
    col = [c for c in df.columns if 'what type' in c and label in c]
    if col:
        ft_cols[label] = col[0]
ft_colors = {lab: _pal5[i] for i, lab in enumerate(ft_cols.keys())}

bar_widths = {}
for cat in ['Both', 'FT only']:
    mask = df['_nic_ft'] == cat
    left = 0
    for lab, col in ft_cols.items():
        n = (df.loc[mask, col] == 'Checked').sum()
        if n > 0:
            ax.barh(cat, n, left=left, color=ft_colors[lab],
                    edgecolor='#2B2D42', lw=1.8, alpha=0.90)
            left += n
    bar_widths[cat] = left

for cat in ['Nic only', 'Neither']:
    ax.barh(cat, cat_counts[cat], color=C['slate'],
            edgecolor='#2B2D42', lw=1.8, alpha=0.90)
    bar_widths[cat] = cat_counts[cat]

max_width = max(bar_widths.values())
for cat in cat_order:
    v = cat_counts[cat]
    pct = f' ({100*v/n_total:.0f}%)'
    ax.text(bar_widths[cat] + max_width * 0.03, cat,
            f'{v}{pct}', va='center', fontsize=9, fontweight='bold', color='#333333')

handles = [Patch(facecolor=ft_colors[lab], edgecolor='#2B2D42', label=lab) for lab in ft_cols.keys()]
handles.append(Patch(facecolor=C['slate'], edgecolor='#2B2D42', label='No field therapy'))
ax.legend(handles=handles, frameon=True, fontsize=7, loc='lower right',
          fancybox=True, edgecolor='#cccccc')
ax.set_xlim(0, max_width * 1.25)
ax.set_xlabel('Number of patients', fontweight='bold')
ax.set_title('Adjunctive therapies', fontweight='bold')
ax.invert_yaxis()
_clean(ax, ylabel=None)

fig.suptitle(f'Cohort baseline characteristics  (n = {n_total})',
             fontsize=16, fontweight='bold', y=1.07)
plt.savefig(os.path.join(FIGURE_DIR, 'fig_1x3_final.png'), bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 11))
fig.subplots_adjust(hspace=0.50, wspace=0.32)

# ── (0,0) Pre-treatment SCC burden (Year −1) ─────────────────
ax = axes[0, 0]
yr1 = pre_scc['yr1'].dropna()
max_scc = int(yr1.max())
bins_scc = np.arange(-0.5, max_scc + 1.5, 1)
ax.hist(yr1, bins=bins_scc, color=C['mauve'], edgecolor='#2B2D42', lw=1.8, alpha=0.90)
ax.axvline(yr1.median(), color='k', ls='--', lw=1.8)
ax.text(yr1.median() + 0.5, ax.get_ylim()[1] * 0.92,
        f'Median {yr1.median():.0f}\nIQR {yr1.quantile(.25):.0f}–{yr1.quantile(.75):.0f}\nMean {yr1.mean():.1f}',
        fontsize=7.5, color='#333', va='top')
ax.set_xlabel('Invasive SCCs in year before acitretin', fontweight='bold')
ax.set_title('Pre-treatment SCC burden (Year −1)')
ax.set_xticks(range(0, max_scc + 1, 2))
_clean(ax)

# ── (0,1) Immune status ──────────────────────────────────────
ax = axes[0, 1]
imm_order = ['Immunocompetent', 'Immunosuppressed']
imm_vals = [df['immune'].value_counts().get(k, 0) for k in imm_order]
imm_display = ['Immuno-\ncompetent', 'Immuno-\ncompromised']
colors_imm = [C['pink'], C['mauve']]
bars = ax.bar(imm_display, imm_vals, width=0.55,
              color=colors_imm, edgecolor='#2B2D42', lw=1.8)
_bar_labels(ax, bars, imm_vals, total=n_total)
ax.set_ylim(0, max(imm_vals) * 1.18)
ax.set_title('Immune status', fontweight='bold')
_clean(ax)

# ── (1,0) Immunosuppression type ─────────────────────────────
ax = axes[1, 0]
types = {'SOTR': df['is_sotr'].sum(),
         'CLL': df['is_cll'].sum(),
         'Other': (df[[c for c in df.columns if "Immunosuppression type" in c and "Other)" in c]].values == 'Checked').sum(),
         'Immuno-\ncompetent': (df['immune'] == 'Immunocompetent').sum()}
type_colors = [C['pink'], C['mauve'], C['plum'], C['purple']]
bars = ax.barh(list(types.keys()), list(types.values()),
               color=type_colors, edgecolor='#2B2D42', lw=1.8)
for b, v in zip(bars, types.values()):
    ax.text(v + 1, b.get_y() + b.get_height()/2, str(v), va='center', fontsize=10, fontweight='bold', color='#333333')
ax.set_xlabel('Number of patients', fontweight='bold')
ax.set_title('Immunosuppression type')
ax.invert_yaxis()
_clean(ax, ylabel=None)

# ── (1,1) SOTR organ breakdown ───────────────────────────────
ax = axes[1, 1]
organs = {}
for label in ['Kidney', 'Liver', 'Heart', 'Lung', 'Small Bowel/GI/pancreas']:
    col = [c for c in df.columns if 'SOTR, what type' in c and label in c][0]
    organs[label] = (df[col] == 'Checked').sum()
organs = dict(sorted(organs.items(), key=lambda x: x[1], reverse=True))
_pal5 = [C['pink'], C['mauve'], C['plum'], C['purple'], C['navy']]
organ_colors = [_pal5[i] for i in range(len(organs))]
bars = ax.barh(list(organs.keys()), list(organs.values()),
               color=organ_colors, edgecolor='#2B2D42', lw=1.8, alpha=0.90)
for b, v in zip(bars, organs.values()):
    ax.text(v + 0.5, b.get_y() + b.get_height()/2, str(v), va='center', fontsize=10, fontweight='bold', color='#333333')
ax.set_xlabel('Number of patients', fontweight='bold')
ax.set_title('SOTR organ type')
ax.invert_yaxis()
_clean(ax, ylabel=None)

# ── (2,0) Adjunctive therapies (horizontal) ──────────────────
ax = axes[2, 0]
df['_nic_ft'] = 'Neither'
df.loc[(df['nicotinamide'] == 1) & (df['field_therapy'] == 0), '_nic_ft'] = 'Nic only'
df.loc[(df['nicotinamide'] == 0) & (df['field_therapy'] == 1), '_nic_ft'] = 'FT only'
df.loc[(df['nicotinamide'] == 1) & (df['field_therapy'] == 1), '_nic_ft'] = 'Both'
cat_order  = ['Neither', 'Nic only', 'FT only', 'Both']
cat_colors = [C['pink'], C['mauve'], C['plum'], C['purple']]
cat_counts = [df['_nic_ft'].value_counts().get(c, 0) for c in cat_order]
bars = ax.barh(cat_order, cat_counts,
               color=cat_colors, edgecolor='#2B2D42', lw=1.8)
for b, v in zip(bars, cat_counts):
    pct = f' ({100*v/n_total:.0f}%)'
    ax.text(v + max(cat_counts)*0.03, b.get_y() + b.get_height()/2,
            f'{v}{pct}', va='center', fontsize=9, fontweight='bold', color='#333333')
ax.set_xlim(0, max(cat_counts) * 1.25)
ax.set_xlabel('Number of patients', fontweight='bold')
ax.set_title('Adjunctive therapies')
ax.invert_yaxis()
_clean(ax, ylabel=None)

# ── (2,1) Field therapy breakdown ─────────────────────────────
ax = axes[2, 1]
ft_types = {}
for label in ['5FU', '5FU/calcipotriene', 'Imiquimod', 'PDT', 'Chemical peel']:
    col = [c for c in df.columns if 'what type' in c and label in c]
    if col:
        ft_types[label] = (df[col[0]] == 'Checked').sum()
ft_types['None'] = (df['field_therapy'] == 0).sum()
ft_s = pd.Series(ft_types)

# Sort therapies by count descending, but force "None" to the bottom
none_val = ft_s.pop('None')
ft_s = ft_s.sort_values(ascending=False)
ft_s['None'] = none_val

_pal5 = [C['pink'], C['mauve'], C['plum'], C['purple'], C['navy']]
_pi = 0
ft_colors = []
for v in ft_s.index:
    if v == 'None':
        ft_colors.append(C['slate'])
    else:
        ft_colors.append(_pal5[_pi])
        _pi += 1
bars = ax.barh(ft_s.index, ft_s.values, color=ft_colors, edgecolor='#2B2D42', lw=1.8, alpha=0.90)
for b, v in zip(bars, ft_s.values):
    ax.text(v + 0.5, b.get_y() + b.get_height()/2, str(v), va='center', fontsize=10, fontweight='bold', color='#333333')
n_ft = (df['field_therapy'] == 1).sum()
ax.set_xlabel('Number of patients', fontweight='bold')
ax.set_title(f'Field therapy types\n({n_ft}/{n_total} = {n_ft/n_total*100:.0f}% received any)')
ax.invert_yaxis()
_clean(ax, ylabel=None)

fig.suptitle(f'Disease characteristics & adjunctive therapies  (n = {n_total})',
             fontsize=16, fontweight='bold', y=1.02)
plt.savefig(os.path.join(FIGURE_DIR, 'fig01b_disease.png'), bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.subplots_adjust(hspace=0.40, wspace=0.35)

from matplotlib.patches import Patch

# ═══════════════════════════════════════════════════════════════
# TOP ROW: Demographics
# ═══════════════════════════════════════════════════════════════

# ── (0,0) Age by sex ─────────────────────────────────────────
ax = axes[0, 0]
ages_m = df.loc[df['sex'] == 'M', 'age'].dropna()
ages_f = df.loc[df['sex'] == 'F', 'age'].dropna()
ages_all = df['age'].dropna()

bins_a = np.arange(ages_all.min() // 5 * 5, ages_all.max() + 6, 5)
counts_m, _ = np.histogram(ages_m, bins=bins_a)
counts_f, _ = np.histogram(ages_f, bins=bins_a)
bin_centers = (bins_a[:-1] + bins_a[1:]) / 2
width = (bins_a[1] - bins_a[0]) * 1

label_m = (f'Male (n={len(ages_m)})\n'
           f'  Mean {ages_m.mean():.1f},\n  Median {ages_m.median():.0f},\n'
           f'  IQR {ages_m.quantile(.25):.0f}–{ages_m.quantile(.75):.0f}')
label_f = (f'Female (n={len(ages_f)})\n'
           f'  Mean {ages_f.mean():.1f},\n  Median {ages_f.median():.0f},\n'
           f'  IQR {ages_f.quantile(.25):.0f}–{ages_f.quantile(.75):.0f}')

ax.bar(bin_centers, counts_m, width=width,
       color=C['mauve'], edgecolor='#2B2D42', lw=1.8, alpha=0.90, label=label_m)
ax.bar(bin_centers, counts_f, width=width, bottom=counts_m,
       color=C['pink'], edgecolor='#2B2D42', lw=1.8, alpha=0.90, label=label_f)
kde_xa = np.linspace(ages_all.min() - 10, ages_all.max() + 10, 200)
kde_a = stats.gaussian_kde(ages_all, bw_method=0.3)
ax.plot(kde_xa, kde_a(kde_xa) * len(ages_all) * (bins_a[1] - bins_a[0]),
        color=C['navy'], lw=2.5, alpha=0.6)
ax.axvline(ages_all.mean(), color='k', ls='--', lw=1.8)
ax.text(ages_all.mean() - 2, ax.get_ylim()[1] * 0.92,
        f'Mean {ages_all.mean():.1f}\n± {ages_all.std():.1f}\n',
        fontsize=7.5, color='#333', va='top', ha='right')
# ax.axvline(ages_all.median(), color='k', ls='--', lw=1.8)
# ax.text(ages_all.median() - 2, ax.get_ylim()[1] * 0.92,
#         f'Median {ages_all.median():.0f}\nIQR {ages_all.quantile(.25):.0f}–{ages_all.quantile(.75):.0f}',
#         fontsize=7.5, color='#333', va='top', ha='right')
ax.set_xlabel('Age at acitretin initiation (years)', fontweight='bold')
ax.set_ylabel('Number of patients', fontweight='bold')
ax.set_title('Age by sex', fontweight='bold')
ax.legend(frameon=True, fontsize=7, loc='upper left', fancybox=True, edgecolor='#cccccc')
_clean(ax, ylabel='Number of patients')

# ── (0,1) Weight ─────────────────────────────────────────────
ax = axes[0, 1]
wts = df.loc[df['weight'] > 10, 'weight'].dropna()
bins_w = np.arange(40, wts.max() + 11, 10)
ax.hist(wts, bins=bins_w, color=C['mauve'], edgecolor='#2B2D42', lw=1.8, alpha=0.90)
kde_xw = np.linspace(wts.min() - 10, wts.max() + 10, 200)
kde_w = stats.gaussian_kde(wts, bw_method=0.3)
ax.plot(kde_xw, kde_w(kde_xw) * len(wts) * (bins_w[1] - bins_w[0]),
        color=C['navy'], lw=2.5, alpha=0.6)
ax.axvline(wts.median(), color='k', ls='--', lw=1.8)
ax.text(wts.median() + 5, ax.get_ylim()[1] * 0.92,
        f'Mean {wts.mean():.1f}\n     ± {wts.std():.1f}',
        fontsize=7.5, color='#333', va='top')
# ax.axvline(wts.median(), color='k', ls='--', lw=1.8)
# ax.text(wts.median() + 5, ax.get_ylim()[1] * 0.92,
#         f'Median {wts.median():.0f}\nIQR {wts.quantile(.25):.0f}–{wts.quantile(.75):.0f}',
#         fontsize=7.5, color='#333', va='top')
ax.set_xlabel('Weight (kg)', fontweight='bold')
ax.set_ylabel('Number of patients', fontweight='bold')
ax.set_title('Weight', fontweight='bold')
_clean(ax, ylabel='Number of patients')

# ── (0,2) Treatment duration (no annotation) ────────────────
ax = axes[0, 2]
dur = df['months_on_drug']
bins_d = np.arange(2.5, 25.5, 1)
ax.hist(dur, bins=bins_d, color=C['mauve'], edgecolor='#2B2D42', lw=1.5, alpha=0.90)
ax.set_xlabel('Months on acitretin', fontweight='bold')
ax.set_ylabel('Number of patients', fontweight='bold')
ax.set_title('Treatment duration', fontweight='bold')
ax.set_xticks([3, 6, 9, 12, 15, 18, 21, 24])
_clean(ax, ylabel='Number of patients')

# ═══════════════════════════════════════════════════════════════
# BOTTOM ROW: Disease characteristics & adjunctive therapies
# ═══════════════════════════════════════════════════════════════

# ── (1,0) Pre-treatment SCC burden by immune status ──────────
ax = axes[1, 0]
yr1_comp = pre_scc.loc[df['immune'] == 'Immunocompetent', 'yr1'].dropna()
yr1_supp = pre_scc.loc[df['immune'] == 'Immunosuppressed', 'yr1'].dropna()
yr1_all = pre_scc['yr1'].dropna()
max_scc = int(yr1_all.max())
scc_range = range(0, max_scc + 1)
counts_comp = [int((yr1_comp == v).sum()) for v in scc_range]
counts_supp = [int((yr1_supp == v).sum()) for v in scc_range]

label_supp = (f'Immunocompromised (n={len(yr1_supp)})\n'
              f'  Mean {yr1_supp.mean():.1f},\n  Median {yr1_supp.median():.0f},\n'
              f'  IQR {yr1_supp.quantile(.25):.0f}–{yr1_supp.quantile(.75):.0f}')
label_comp = (f'Immunocompetent (n={len(yr1_comp)})\n'
              f'  Mean {yr1_comp.mean():.1f},\n  Median {yr1_comp.median():.0f},\n'
              f'  IQR {yr1_comp.quantile(.25):.0f}–{yr1_comp.quantile(.75):.0f}')

ax.bar(scc_range, counts_supp, width=0.85,
       color=C['mauve'], edgecolor='#2B2D42', lw=1.8, alpha=0.90, label=label_supp)
ax.bar(scc_range, counts_comp, width=0.85, bottom=counts_supp,
       color=C['pink'], edgecolor='#2B2D42', lw=1.8, alpha=0.90, label=label_comp)
ax.axvline(yr1_all.median(), color='k', ls='--', lw=1.8)
ax.text(yr1_all.median() + 0.5, ax.get_ylim()[1] * 0.92,
        f'Median {yr1_all.median():.0f}\nIQR {yr1_all.quantile(.25):.0f}–{yr1_all.quantile(.75):.0f}',
        fontsize=7.5, color='#333', va='top')
ax.set_xlabel('Invasive SCCs in year before acitretin', fontweight='bold')
ax.set_ylabel('Number of patients', fontweight='bold')
ax.set_title('Pre-treatment SCC burden by immune status', fontweight='bold')
ax.set_xticks(range(0, max_scc + 1, 2))
ax.legend(frameon=True, fontsize=7, loc='upper right', fancybox=True, edgecolor='#cccccc')
_clean(ax, ylabel='Number of patients')

# ── (1,1) Immunosuppression type – SOTR split by organ ──────
ax = axes[1, 1]
imm_order = ['SOTR', 'CLL', 'Other', 'Immuno-\ncompetent']
imm_totals = {
    'SOTR': int(df['is_sotr'].sum()),
    'CLL': int(df['is_cll'].sum()),
    'Other': int((df[[c for c in df.columns if "Immunosuppression type" in c and "Other)" in c]].values == 'Checked').sum()),
    'Immuno-\ncompetent': int((df['immune'] == 'Immunocompetent').sum())
}

organ_labels = ['Kidney', 'Liver', 'Heart', 'Lung', 'Small Bowel/GI/pancreas']
_pal5 = [C['pink'], C['mauve'], C['plum'], C['purple'], C['navy']]
organ_colors = {lab: _pal5[i] for i, lab in enumerate(organ_labels)}

left = 0
for label in organ_labels:
    col = [c for c in df.columns if 'SOTR, what type' in c and label in c][0]
    n = int((df[col] == 'Checked').sum())
    if n > 0:
        ax.barh('SOTR', n, left=left, color=organ_colors[label],
                edgecolor='#2B2D42', lw=1.8, alpha=0.90)
        left += n
sotr_width = left

for cat in ['CLL', 'Other', 'Immuno-\ncompetent']:
    ax.barh(cat, imm_totals[cat], color=C['slate'],
            edgecolor='#2B2D42', lw=1.8, alpha=0.90)

bar_widths = {'SOTR': sotr_width}
bar_widths.update({cat: imm_totals[cat] for cat in ['CLL', 'Other', 'Immuno-\ncompetent']})
max_width = max(bar_widths.values())
for cat in imm_order:
    v = imm_totals[cat]
    pct = f' ({100*v/n_total:.0f}%)'
    ax.text(bar_widths[cat] + max_width * 0.03, cat,
            f'{v}{pct}', va='center', fontsize=9, fontweight='bold', color='#333333')

handles = [Patch(facecolor=organ_colors[lab], edgecolor='#2B2D42',
                 label=lab.replace('Small Bowel/GI/pancreas', 'Small Bowel/\nGI/Pancreas'))
           for lab in organ_labels]
handles.append(Patch(facecolor=C['slate'], edgecolor='#2B2D42', label='Non-SOTR'))
ax.legend(handles=handles, frameon=True, fontsize=7, loc='center right',
          fancybox=True, edgecolor='#cccccc')
ax.set_xlim(0, max_width * 1.45)
ax.set_xlabel('Number of patients', fontweight='bold')
ax.set_title('Immunosuppression type', fontweight='bold')
ax.invert_yaxis()
_clean(ax, ylabel=None)

# ── (1,2) Adjunctive therapies with FT breakdown ────────────
ax = axes[1, 2]
df['_nic_ft'] = 'Neither'
df.loc[(df['nicotinamide'] == 1) & (df['field_therapy'] == 0), '_nic_ft'] = 'Nic only'
df.loc[(df['nicotinamide'] == 0) & (df['field_therapy'] == 1), '_nic_ft'] = 'FT only'
df.loc[(df['nicotinamide'] == 1) & (df['field_therapy'] == 1), '_nic_ft'] = 'Both'

cat_order = ['Both', 'FT only', 'Nic only', 'Neither']
cat_counts = {c: df['_nic_ft'].value_counts().get(c, 0) for c in cat_order}

ft_labels = ['5FU', '5FU/calcipotriene', 'Imiquimod', 'PDT', 'Chemical peel']
ft_cols = {}
for label in ft_labels:
    col = [c for c in df.columns if 'what type' in c and label in c]
    if col:
        ft_cols[label] = col[0]
ft_colors = {lab: _pal5[i] for i, lab in enumerate(ft_cols.keys())}

bar_widths = {}
for cat in ['Both', 'FT only']:
    mask = df['_nic_ft'] == cat
    left = 0
    for lab, col in ft_cols.items():
        n = (df.loc[mask, col] == 'Checked').sum()
        if n > 0:
            ax.barh(cat, n, left=left, color=ft_colors[lab],
                    edgecolor='#2B2D42', lw=1.8, alpha=0.90)
            left += n
    bar_widths[cat] = left

for cat in ['Nic only', 'Neither']:
    ax.barh(cat, cat_counts[cat], color=C['slate'],
            edgecolor='#2B2D42', lw=1.8, alpha=0.90)
    bar_widths[cat] = cat_counts[cat]

max_width = max(bar_widths.values())
for cat in cat_order:
    v = cat_counts[cat]
    pct = f' ({100*v/n_total:.0f}%)'
    ax.text(bar_widths[cat] + max_width * 0.03, cat,
            f'{v}{pct}', va='center', fontsize=9, fontweight='bold', color='#333333')

handles = [Patch(facecolor=ft_colors[lab], edgecolor='#2B2D42', label=lab) for lab in ft_cols.keys()]
handles.append(Patch(facecolor=C['slate'], edgecolor='#2B2D42', label='No field therapy'))
ax.legend(handles=handles, frameon=True, fontsize=7, loc='center right',
          fancybox=True, edgecolor='#cccccc')
ax.set_xlim(0, max_width * 1.45)
ax.set_xlabel('Number of patients', fontweight='bold')
ax.set_title('Adjunctive therapies', fontweight='bold')
ax.invert_yaxis()
_clean(ax, ylabel=None)

fig.suptitle(f'Cohort characteristics  (n = {n_total})',
             fontsize=16, fontweight='bold', y=1.06)
plt.savefig(os.path.join(FIGURE_DIR, 'fig01_combined.png'), bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# %% NEW CELL — Acitretin side effects (requested 2026-08-15)
# ===========================================================================
# Left  : percentage of the whole cohort reporting each side effect.
# Right : the same percentages stratified by immune status.
#
# Source = the REDCap checkbox block "Did the patient experience acitretin
# side effects? (check those that apply)". Denominator is the full analysis
# cohort. "Any side effect" = ≥1 specific box ticked; it is computed from the
# specific boxes and NOT from the "None" box, because a few records have
# "None" ticked alongside a specific side effect (those are listed below).
# ===========================================================================

SE_LABELS = ['Abnormal LFTs', 'Abnormal lipids', 'Retinoid dermatitis',
             'Skeletal abnormalities', 'Xerosis/Pruritis', 'Hair loss', 'Other']

se_cols = {}
for lab in SE_LABELS + ['None']:
    _c = [c for c in df.columns
          if 'acitretin side effects' in c and f'choice={lab})' in c]
    assert _c, f"side-effect column not found for {lab!r}"
    se_cols[lab] = _c[0]

se_flag = pd.DataFrame({lab: (df[col] == 'Checked').astype(int)
                        for lab, col in se_cols.items()}, index=df.index)

any_se  = se_flag[SE_LABELS].sum(axis=1) > 0
none_se = se_flag['None'] == 1

print("=" * 74)
print(f"  ACITRETIN SIDE EFFECTS  (N = {len(df)})")
print("=" * 74)
_both    = int((any_se & none_se).sum())
_neither = int((~any_se & ~none_se).sum())
print(f"  ≥1 specific side effect ticked : {int(any_se.sum())} ({100*any_se.mean():.1f}%)")
print(f"  'None' ticked                  : {int(none_se.sum())} ({100*none_se.mean():.1f}%)")
print(f"  'None' ticked AND ≥1 specific  : {_both}   ← inconsistent records")
print(f"  Nothing ticked at all          : {_neither}")
if _both:
    print("\n  Records with 'None' plus a specific side effect (worth a data check):")
    _bad = pd.DataFrame({
        'Study ID': df.loc[any_se & none_se, 'Study ID'].values,
        'Site':     df.loc[any_se & none_se, 'site'].values,
        'Reported': [', '.join([l for l in SE_LABELS if se_flag.loc[i, l]])
                     for i in df.index[any_se & none_se]],
    })
    print(_bad.to_string(index=False))

# ── Summary table ──────────────────────────────────────────────────────────
groups = [('All patients',      pd.Series(True, index=df.index)),
          ('Immunocompetent',   df['immune'] == 'Immunocompetent'),
          ('Immunocompromised', df['immune'] == 'Immunosuppressed')]

rows = []
for lab in SE_LABELS + ['Any side effect', 'No side effect reported']:
    if   lab == 'Any side effect':         flag = any_se
    elif lab == 'No side effect reported': flag = ~any_se
    else:                                  flag = se_flag[lab] == 1
    rec = {'Side effect': lab}
    for gname, gmask in groups:
        n, tot = int((flag & gmask).sum()), int(gmask.sum())
        rec[f'{gname} (n={tot})'] = f'{n} ({100*n/tot:.1f}%)'
        rec[f'_pct_{gname}'] = 100 * n / tot
        rec[f'_n_{gname}']   = n
    rows.append(rec)
se_table = pd.DataFrame(rows)

# Order: specific side effects by overall frequency, then the two summary rows
_spec_order = (se_table[se_table['Side effect'].isin(SE_LABELS)]
               .sort_values('_pct_All patients', ascending=False)['Side effect'].tolist())
order = _spec_order + ['Any side effect', 'No side effect reported']
se_table = se_table.set_index('Side effect').loc[order].reset_index()

_show_cols = [c for c in se_table.columns if not c.startswith('_')]
print()
with pd.option_context('display.width', 160, 'display.max_columns', None):
    print(se_table[_show_cols].to_string(index=False))
se_table[_show_cols].to_csv('./table_side_effects.csv', index=False)
print("\n  ✓ Saved to ./table_side_effects.csv")

# ── Figure ─────────────────────────────────────────────────────────────────
fig, (axA, axB) = plt.subplots(1, 2, figsize=(15, 6),
                                gridspec_kw={'width_ratios': [1, 1.15]})

def _clean_h(a):
    a.spines['top'].set_visible(False)
    a.spines['right'].set_visible(False)
    a.spines['left'].set_color('#333333')
    a.spines['bottom'].set_color('#333333')
    a.spines['left'].set_linewidth(2.0)
    a.spines['bottom'].set_linewidth(2.0)
    a.tick_params(colors='#333333', width=1.5, labelsize=10)
    a.grid(False)
    a.set_ylim(len(order) - 0.35, -0.65)

y = np.arange(len(order))

# ── Panel A: whole cohort ──────────────────────────────────────────────────
colors_A = [C['mauve']] * len(SE_LABELS) + [C['plum'], C['slate']]
vals_A = se_table['_pct_All patients'].values
ns_A   = se_table['_n_All patients'].values

axA.barh(y, vals_A, height=0.68, color=colors_A,
         edgecolor='#2B2D42', lw=1.6, alpha=0.90)
for yi, v, n in zip(y, vals_A, ns_A):
    axA.text(v + max(vals_A) * 0.02, yi, f'{n} ({v:.1f}%)',
             va='center', fontsize=9, fontweight='bold', color='#333333')
axA.axhline(len(SE_LABELS) - 0.5, color='#CCCCCC', lw=1.0, ls='-')
axA.set_yticks(y)
axA.set_yticklabels(order, fontsize=10)
axA.set_xlim(0, max(vals_A) * 1.30)
axA.set_xlabel('Percentage of patients (%)', fontweight='bold')
axA.set_title(f'A.  Entire cohort  (n = {len(df)})',
              fontsize=12, fontweight='bold', loc='left')
_clean_h(axA)

# ── Panel B: by immune status ──────────────────────────────────────────────
bar_h = 0.34
for i, (gname, col) in enumerate([('Immunocompetent',   C['pink']),
                                   ('Immunocompromised', C['plum'])]):
    gmask = dict(groups)[gname]
    vals = se_table[f'_pct_{gname}'].values
    ns   = se_table[f'_n_{gname}'].values
    offs = (i - 0.5) * bar_h
    axB.barh(y + offs, vals, height=bar_h, color=col,
             edgecolor='#2B2D42', lw=1.4, alpha=0.90,
             label=f'{gname} (n={int(gmask.sum())})')
    for yi, v, n in zip(y, vals, ns):
        axB.text(v + 1.2, yi + offs, f'{n} ({v:.0f}%)',
                 va='center', fontsize=7.5, color='#333333')

axB.axhline(len(SE_LABELS) - 0.5, color='#CCCCCC', lw=1.0, ls='-')
axB.set_yticks(y)
axB.set_yticklabels(order, fontsize=10)
_bmax = max(se_table['_pct_Immunocompetent'].max(),
            se_table['_pct_Immunocompromised'].max())
axB.set_xlim(0, _bmax * 1.32)
axB.set_xlabel('Percentage of patients within group (%)', fontweight='bold')
axB.set_title('B.  Stratified by immune status',
              fontsize=12, fontweight='bold', loc='left')
axB.legend(fontsize=9, loc='upper right', framealpha=0.95, edgecolor='#cccccc')
_clean_h(axB)

fig.suptitle('Acitretin side effects reported during the 24-month study period',
             fontsize=15, fontweight='bold', y=1.09)
plt.savefig(os.path.join(FIGURE_DIR, 'fig_side_effects.png'), bbox_inches='tight', dpi=600)
plt.show()
print("  ✓ Saved to figures/fig_side_effects.png")

# ── What was written in the free-text "Other" box ──────────────────────────
_oth_free = [c for c in df.columns if c.strip() == 'If Other, please specify.']
if _oth_free:
    _txt = df.loc[se_flag['Other'] == 1, _oth_free[0]].dropna()
    print(f"\n  Free-text entries behind 'Other' "
          f"(n={len(_txt)} recorded of {int((se_flag['Other'] == 1).sum())} who ticked it):")
    for t in _txt:
        print(f"    · {str(t).strip()}")

In [ ]:
# %% NEW CELL — Discontinuation attributed to adverse effects (requested 2026-08-15)
# ===========================================================================
# Source = the single REDCap field
#   "If the patient had >2 consecutive months OFF acitretin during the 24
#    month study period, why was acitretin stopped or paused?"
# with fixed options {Abnormal LFTs, Abnormal lipids, Retinoid dermatitis,
# Cost, Other} plus a free-text box for "Other".
#
# Scope note for the manuscript: this field is only completed when a patient
# was off drug for >2 consecutive months, so the numerator is "stopped or
# paused for >2 months, attributed to an adverse effect" and the denominator
# is the whole cohort. A patient who stopped for a shorter period, or who was
# never off drug, contributes a blank and is counted in the denominator only.
#
# Classification rule
#   adverse effect  = the three toxicity options above, or a free-text entry
#                     describing a symptom / laboratory abnormality attributed
#                     to acitretin.
#   not an adverse effect = cost, death, intercurrent illness or a competing
#                     diagnosis, lack of efficacy, non-adherence or
#                     administrative reasons, loss to follow-up, unknown.
# Every free-text entry is coded explicitly in discontinuation_reason_coding.csv
# (loaded into FREETEXT_AE below) and the per-patient calls are exported
# to table_discontinuation_freetext_coding.csv so any single call can be
# checked and overridden.
# ===========================================================================

_stop_col = [c for c in df.columns
             if c.startswith('If the patient had >2 consecutive months OFF')][0]
_stop_oth = [c for c in df.columns if c.strip() == 'If Other, please specify.2'][0]

STRUCTURED_AE = {'Abnormal LFTs', 'Abnormal lipids', 'Retinoid dermatitis'}

# Verbatim free text  →  adverse effect Yes / No, plus the reason category used
# in panel B. The coding lives in a CSV next to the dataset, because the text
# is patient-level clinical data and stays out of the code repository. Every
# entry was coded individually; change a call by editing that file.
REASON_CODING_FILE = os.path.join(PROJECT_DIR, 'discontinuation_reason_coding.csv')
_coding = pd.read_csv(REASON_CODING_FILE, keep_default_na=False)
assert _coding['free_text'].is_unique
assert set(_coding['adverse_effect']) <= {'Yes', 'No'}
FREETEXT_AE       = dict(zip(_coding['free_text'], _coding['adverse_effect'] == 'Yes'))
FREETEXT_CATEGORY = dict(zip(_coding['free_text'], _coding['category']))

_reason  = df[_stop_col]
_free    = df[_stop_oth].astype(str).str.strip()

stop_ae      = pd.Series(False, index=df.index)
stop_recorded = _reason.notna()
_unmapped = []

for i in df.index:
    r = _reason[i]
    if pd.isna(r):
        continue
    if r in STRUCTURED_AE:
        stop_ae[i] = True
    elif r == 'Other':
        t = _free[i]
        if t in ('', 'nan'):
            _unmapped.append((df.loc[i, 'Study ID'], '(blank)'))
        elif t in FREETEXT_AE:
            stop_ae[i] = FREETEXT_AE[t]
        else:
            _unmapped.append((df.loc[i, 'Study ID'], t))

print("=" * 74)
print(f"  DISCONTINUATION / PAUSE >2 CONSECUTIVE MONTHS  (N = {len(df)})")
print("=" * 74)
print(f"  Reason recorded (i.e. off drug >2 months) : "
      f"{int(stop_recorded.sum())} ({100*stop_recorded.mean():.1f}%)")
print(f"  Attributed to an adverse effect           : "
      f"{int(stop_ae.sum())} ({100*stop_ae.mean():.1f}% of cohort, "
      f"{100*stop_ae.sum()/max(int(stop_recorded.sum()),1):.1f}% of those with a reason)")
if _unmapped:
    print(f"\n  ⚠  {len(_unmapped)} free-text entries not covered by FREETEXT_AE "
          f"(counted as NOT an adverse effect):")
    for sid, t in _unmapped:
        print(f"      Study ID {sid}: {t!r}")

# ── Auditable coding table ─────────────────────────────────────────────────
_code_rows = []
for i in df.index[stop_recorded]:
    r = _reason[i]
    t = _free[i] if r == 'Other' else ''
    _code_rows.append({
        'Study ID':        df.loc[i, 'Study ID'],
        'Site':            df.loc[i, 'site'],
        'Immune status':   'Immunocompromised' if df.loc[i, 'immune'] == 'Immunosuppressed'
                           else 'Immunocompetent',
        'REDCap reason':   r,
        'Free text':       '' if t in ('nan',) else t,
        'Coded as adverse effect': 'Yes' if stop_ae[i] else 'No',
    })
disc_coding = pd.DataFrame(_code_rows).sort_values(
    ['Coded as adverse effect', 'REDCap reason', 'Study ID'])
disc_coding.to_csv('./table_discontinuation_freetext_coding.csv', index=False)
with pd.option_context('display.max_rows', None, 'display.width', 200,
                       'display.max_colwidth', 70):
    print("\n" + disc_coding.to_string(index=False))
print("\n  ✓ Saved to ./table_discontinuation_freetext_coding.csv")

# ── Summary table ──────────────────────────────────────────────────────────
disc_groups = [('All patients',      pd.Series(True, index=df.index)),
               ('Immunocompetent',   df['immune'] == 'Immunocompetent'),
               ('Immunocompromised', df['immune'] == 'Immunosuppressed')]

disc_rows = []
for gname, gmask in disc_groups:
    tot = int(gmask.sum())
    n_ae = int((stop_ae & gmask).sum())
    n_rec = int((stop_recorded & gmask).sum())
    disc_rows.append({
        'Group': gname, 'n': tot,
        'Off drug >2 months (reason recorded)': f'{n_rec} ({100*n_rec/tot:.1f}%)',
        'Stopped/paused for an adverse effect': f'{n_ae} ({100*n_ae/tot:.1f}%)',
        '_pct_ae': 100 * n_ae / tot, '_n_ae': n_ae, '_tot': tot,
    })
disc_table = pd.DataFrame(disc_rows)
print()
print(disc_table[[c for c in disc_table.columns if not c.startswith('_')]]
      .to_string(index=False))
disc_table[[c for c in disc_table.columns if not c.startswith('_')]].to_csv(
    './table_discontinuation_ae.csv', index=False)
print("\n  ✓ Saved to ./table_discontinuation_ae.csv")

# ── Reason categories (panel B) ────────────────────────────────────────────
def _reason_category(i):
    r = _reason[i]
    if pd.isna(r):
        return 'Not off drug >2 months / no reason recorded'
    if r in STRUCTURED_AE:
        return 'Adverse effect'
    if r == 'Cost':
        return 'Cost'
    return FREETEXT_CATEGORY.get(_free[i], 'Other non-toxicity reason')

reason_cat = pd.Series([_reason_category(i) for i in df.index], index=df.index)
cat_order = ['Adverse effect', 'Death', 'Lack of efficacy / course completed',
             'Cost', 'Other non-toxicity reason',
             'Not off drug >2 months / no reason recorded']
cat_counts = [int((reason_cat == c).sum()) for c in cat_order]
pd.DataFrame({'Reason category': cat_order, 'n': cat_counts,
              '% of cohort': [100 * c / len(df) for c in cat_counts]}).to_csv(
    './table_discontinuation_reasons.csv', index=False)

# ── Figure ─────────────────────────────────────────────────────────────────
fig, (axA, axB) = plt.subplots(1, 2, figsize=(14, 5.5),
                                gridspec_kw={'width_ratios': [1, 1.25]})

def _clean_v(a):
    a.spines['top'].set_visible(False)
    a.spines['right'].set_visible(False)
    a.spines['left'].set_color('#333333')
    a.spines['bottom'].set_color('#333333')
    a.spines['left'].set_linewidth(2.0)
    a.spines['bottom'].set_linewidth(2.0)
    a.tick_params(colors='#333333', width=1.5, labelsize=10)
    a.grid(False)

# Panel A — the requested figure
xs = np.arange(3)
vals = disc_table['_pct_ae'].values
bars = axA.bar(xs, vals, width=0.58,
               color=[C['mauve'], C['pink'], C['plum']],
               edgecolor='#2B2D42', lw=1.8, alpha=0.90)
for b, v, n, tot in zip(bars, vals, disc_table['_n_ae'], disc_table['_tot']):
    axA.text(b.get_x() + b.get_width() / 2, v + max(vals) * 0.04,
             f'{v:.1f}%\n{n}/{tot}', ha='center', va='bottom',
             fontsize=10, fontweight='bold', color='#333333')
axA.set_xticks(xs)
axA.set_xticklabels(['All\npatients', 'Immuno-\ncompetent', 'Immuno-\ncompromised'],
                    fontsize=10)
axA.set_ylim(0, max(vals) * 1.35)
axA.set_ylabel('Percentage of patients (%)', fontweight='bold')
axA.set_title('A.  Acitretin stopped or paused >2 months\n     because of an adverse effect',
              fontsize=12, fontweight='bold', loc='left')
_clean_v(axA)

# Panel B — every recorded reason, for context
yb = np.arange(len(cat_order))
cat_colors = [C['plum'], C['navy'], C['purple'], C['coral'], C['mauve'], C['slate']]
axB.barh(yb, [100 * c / len(df) for c in cat_counts], height=0.66,
         color=cat_colors, edgecolor='#2B2D42', lw=1.5, alpha=0.90)
for yi, c in zip(yb, cat_counts):
    axB.text(100 * c / len(df) + 1.0, yi, f'{c} ({100*c/len(df):.1f}%)',
             va='center', fontsize=9, fontweight='bold', color='#333333')
axB.set_yticks(yb)
axB.set_yticklabels([c.replace(' / ', ' /\n') for c in cat_order], fontsize=9)
axB.set_xlim(0, max(100 * c / len(df) for c in cat_counts) * 1.30)
axB.set_xlabel('Percentage of patients (%)', fontweight='bold')
axB.set_title('B.  Reason recorded for stopping or pausing\n     (entire cohort)',
              fontsize=12, fontweight='bold', loc='left')
axB.invert_yaxis()
_clean_v(axB)

fig.suptitle(f'Acitretin discontinuation attributed to adverse effects  (n = {len(df)})',
             fontsize=15, fontweight='bold', y=1.09)
plt.savefig(os.path.join(FIGURE_DIR, 'fig_discontinuation_ae.png'), bbox_inches='tight', dpi=600)
plt.show()
print("  ✓ Saved to figures/fig_discontinuation_ae.png")

In [ ]:
# ============================================================================
# FIGURE — Multi-site structure
# ============================================================================
# Left:  Total patients per site
# Right: Diverging bars — pre-treatment monthly SCC rate (up) vs.
#        on-acitretin monthly SCC rate (down), ordered by pre rate
#        Both normalized to SCCs per patient per month.
# ============================================================================

# ── Compute site-level rates ────────────────────────────────
site_stats = pd.DataFrame({'site': df['site'].unique()}).set_index('site')

for s in site_stats.index:
    mask = df['site'] == s
    site_stats.loc[s, 'n'] = mask.sum()

    # Pre-treatment: Year −1 SCC ÷ 12 → monthly rate
    pre_vals = pre_scc.loc[mask, 'yr1'].dropna()
    site_stats.loc[s, 'pre_monthly'] = (pre_vals.mean() / 12) if len(pre_vals) > 0 else np.nan

    # On-drug: pooled SCC per patient-month (censored)
    patient_totals = []
    patient_months = []
    for idx in df.index[mask]:
        last_on = int(df.loc[idx, 'months_on_drug'])
        vals = [scc_month_cens.loc[idx, f'm{m}'] for m in range(1, last_on + 1)]
        vals = [v for v in vals if pd.notna(v)]
        if vals:
            patient_totals.append(sum(vals))
            patient_months.append(len(vals))
    if patient_months:
        site_stats.loc[s, 'post_monthly'] = sum(patient_totals) / sum(patient_months)
    else:
        site_stats.loc[s, 'post_monthly'] = np.nan

site_stats = site_stats.sort_values('pre_monthly', ascending=False)

# ── Figure ──────────────────────────────────────────────────
fig, (ax_n, ax_div) = plt.subplots(1, 2, figsize=(15, 5.5),
                                    gridspec_kw={'width_ratios': [1, 2.2]})
fig.subplots_adjust(wspace=0.30)

site_order = site_stats.index.tolist()
x = np.arange(len(site_order))

# ── LEFT: Patients per site (same order) ────────────────────
counts = site_stats['n'].values.astype(int)
bars = ax_n.bar(x, counts, color=C['mauve'], edgecolor='#2B2D42', lw=1.5, alpha=0.90)
for i, v in enumerate(counts):
    ax_n.text(i, v + 0.4, str(v), ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='#333')
ax_n.set_xticks(x)
ax_n.set_xticklabels(site_order, rotation=45, ha='right', fontsize=8)
ax_n.set_ylabel('Number of patients', fontweight='bold')
ax_n.set_title('Patients per site')
_clean(ax_n, ylabel='Number of patients')

# ── RIGHT: Diverging pre (up) / post (down) ────────────────
pre = site_stats['pre_monthly'].values
post = site_stats['post_monthly'].values

bar_w = 0.6
bars_pre = ax_div.bar(x, pre, width=bar_w,
                       color=C['plum'], edgecolor='#2B2D42', lw=1.5, alpha=0.90,
                       label='Pre-treatment (Year −1 ÷ 12)')
bars_post = ax_div.bar(x, -post, width=bar_w,
                        color=C['pink'], edgecolor='#2B2D42', lw=1.5, alpha=0.90,
                        label='On acitretin')

# Value labels
for i in range(len(site_order)):
    if pd.notna(pre[i]):
        ax_div.text(i, pre[i] + 0.008, f'{pre[i]:.2f}',
                    ha='center', va='bottom', fontsize=7.5, fontweight='bold', color='#333')
    if pd.notna(post[i]):
        ax_div.text(i, -post[i] - 0.008, f'{post[i]:.2f}',
                    ha='center', va='top', fontsize=7.5, fontweight='bold', color='#333')

ax_div.axhline(0, color='#333', lw=1.5)
ax_div.set_xticks(x)
ax_div.set_xticklabels(site_order, rotation=45, ha='right', fontsize=8)
ax_div.set_ylabel('Mean SCCs per patient per month', fontweight='bold')
ax_div.set_title('Pre-treatment vs. on-acitretin SCC rate (monthly)')

# Show absolute values on y-axis (FuncFormatter avoids tick/label desync)
ax_div.yaxis.set_major_formatter(mticker.FuncFormatter(lambda t, _: f'{abs(t):.2f}'))

ax_div.legend(fontsize=8.5, loc='upper right', framealpha=0.9)
_clean(ax_div, ylabel='Mean SCCs per patient per month')

fig.suptitle('Multi-site structure: enrolment & pre/post SCC rate comparison',
             fontsize=15, fontweight='bold', y=1.07)
plt.savefig(os.path.join(FIGURE_DIR, 'fig_sites.png'), bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# ============================================================================
# FIGURE 4 (revised) — Pre-treatment + On-treatment annualized SCC rate
#                       Two-panel: full history (left) + ±2 yr zoom (right)
# ============================================================================
# Paste after data-prep cells (1–6).
# Uses: pre_scc, scc_month_cens, df, C

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15, 5),
                               gridspec_kw={'width_ratios': [3, 1.4]})

# ── PRE-TREATMENT: mean SCC per year ──────────────────────────────────────
pre_means, pre_lo, pre_hi, pre_ns = [], [], [], []
for yr in range(10, 0, -1):
    vals = pre_scc[f'yr{yr}'].dropna()
    mu = vals.mean()
    se = vals.std() / np.sqrt(len(vals))
    pre_means.append(mu)
    pre_lo.append(mu - 1.96 * se)
    pre_hi.append(mu + 1.96 * se)
    pre_ns.append(len(vals))

x_pre = list(range(-9, 1))  # -9, -8, ..., -1, 0

# ── ON-TREATMENT: annualized SCC for Year 1 (m1–12) & Year 2 (m13–24) ────
on_means, on_lo, on_hi, on_ns = [], [], [], []
for m_start, m_end in [(1, 12), (13, 24)]:
    cols = [f'm{m}' for m in range(m_start, m_end + 1)]
    n_months = scc_month_cens[cols].notna().sum(axis=1)
    total_scc = scc_month_cens[cols].sum(axis=1, min_count=1)

    mask = n_months >= 3
    weights = n_months[mask].values.astype(float)
    annualized = (total_scc[mask] / n_months[mask]) * 12

    mu = np.average(annualized, weights=weights)
    w_var = np.average((annualized - mu) ** 2, weights=weights)
    # Reliability-weighted SE: accounts for unequal weights
    # (Cochran 1977, Sampling Techniques, ch. 5)
    sum_w  = np.sum(weights)
    sum_w2 = np.sum(weights ** 2)
    n_eff  = sum_w ** 2 / sum_w2          # effective sample size
    se = np.sqrt(w_var / n_eff)

    on_means.append(mu)
    on_lo.append(mu - 1.96 * se)
    on_hi.append(mu + 1.96 * se)
    on_ns.append(mask.sum())

x_on = [1, 2]


# ═══════════════════════════════════════════════════════════════════════════
# HELPER: draw a panel
# ═══════════════════════════════════════════════════════════════════════════
def draw_panel(a, x_pre_sub, pre_m, pre_l, pre_h, pre_n,
               x_on_sub, on_m, on_l, on_h, on_n,
               title, show_legend=True):
    # Pre segment
    a.fill_between(x_pre_sub, pre_l, pre_h, alpha=0.18, color=C['plum'])
    a.plot(x_pre_sub, pre_m, 'o-', color=C['plum'], markersize=5, lw=2,
           label='Pre (SCC/year)')

    # Connecting bridge from last pre to first on
    a.plot([x_pre_sub[-1], x_on_sub[0]],
           [pre_m[-1], on_m[0]], '-', color=C['pink'], lw=2, alpha=1.0)
    a.fill_between([x_pre_sub[-1], x_on_sub[0]],
                   [pre_l[-1], on_l[0]],
                   [pre_h[-1], on_h[0]],
                   alpha=0.12, color=C['pink'])

    # On segment
    a.fill_between(x_on_sub, on_l, on_h, alpha=0.18, color=C['pink'])
    a.plot(x_on_sub, on_m, 'o-', color=C['pink'], markersize=5, lw=2,
           label='On acitretin (annualised)')

    # n labels
    for xi, mu, n in zip(x_pre_sub, pre_m, pre_n):
        if xi == -1:
            a.text(xi, mu - 0.18, f'n={n}', ha='center', fontsize=6.5,
                   color='black', va='top')
        else:
            a.text(xi, mu + 0.12, f'n={n}', ha='center', fontsize=6.5,
                   color='black')
    for xi, mu, n in zip(x_on_sub, on_m, on_n):
        a.text(xi, mu + 0.12, f'n={n}', ha='center', fontsize=6.5,
               color='black')

    # Acitretin-start divider
    a.axvline(0, color=C['navy'], ls='-', lw=1.5, alpha=0.35)
    a.text(0, a.get_ylim()[1] if a.get_ylim()[1] > 2.5 else 2.8,
           'Acitretin start', ha='center', fontsize=8.5, color=C['navy'],
           alpha=0.7, va='bottom')

    # Tick labels
    all_x = list(x_pre_sub) + list(x_on_sub)
    tick_labels = [f'Pre {abs(xi)+1}' if xi < 0
                   else ('Pre 1' if xi == 0 else f'On {xi}')
                   for xi in all_x]
    a.set_xticks(all_x)
    a.set_xticklabels(tick_labels, fontsize=7.5, rotation=45, ha='right')

    a.set_xlabel('Years relative to acitretin start', fontweight='bold')
    a.set_ylabel('Mean SCC count per year', fontweight='bold')
    a.set_title(title, fontsize=12, pad=10)
    if show_legend:
        a.legend(fontsize=8, loc='upper left')

    a.spines['top'].set_visible(False)
    a.spines['right'].set_visible(False)
    a.spines['left'].set_linewidth(2.0)
    a.spines['bottom'].set_linewidth(2.0)
    a.spines['left'].set_color('#333333')
    a.spines['bottom'].set_color('#333333')


# ═══════════════════════════════════════════════════════════════════════════
# LEFT PANEL — full history (Pre 10 → On 2)
# ═══════════════════════════════════════════════════════════════════════════
draw_panel(ax, x_pre, pre_means, pre_lo, pre_hi, pre_ns,
           x_on, on_means, on_lo, on_hi, on_ns,
           'Pre-treatment SCC history & on-treatment rate',
           show_legend=True)

# ═══════════════════════════════════════════════════════════════════════════
# RIGHT PANEL — ±2 yr zoom (Pre 2, Pre 1, On 1, On 2)
# ═══════════════════════════════════════════════════════════════════════════
# Pre 2 = x=-1, Pre 1 = x=0  →  last 2 entries of the pre arrays
zoom_pre_x = x_pre[-2:]            # [-1, 0]
zoom_pre_m = pre_means[-2:]
zoom_pre_l = pre_lo[-2:]
zoom_pre_h = pre_hi[-2:]
zoom_pre_n = pre_ns[-2:]

draw_panel(ax2, zoom_pre_x, zoom_pre_m, zoom_pre_l, zoom_pre_h, zoom_pre_n,
           x_on, on_means, on_lo, on_hi, on_ns,
           '±2 years around start',
           show_legend=False)

plt.savefig(os.path.join(FIGURE_DIR, 'fig04_pre_on.png'), bbox_inches='tight', dpi=600)
plt.show()

# ── Companion table: exactly the numbers plotted above ────────────────────
# Pre points  : per-year arithmetic mean of the recorded SCC count, complete
#               cases for that year, CI = mean ± 1.96 · SD/√n.
# On points   : per-patient annualised rate ((SCCs ÷ months observed) × 12)
#               among patients with ≥3 observed months in the window; the
#               point is the mean weighted by months observed and the CI uses
#               the reliability-weighted SE with n_eff = (Σw)² / Σw².
_rows = [{'Period': f'Pre {yr}', 'n': n, 'Mean SCC/yr': mu, 'CI low': lo, 'CI high': hi}
         for yr, mu, lo, hi, n in zip(range(10, 0, -1), pre_means, pre_lo, pre_hi, pre_ns)]
_rows += [{'Period': f'On {k}', 'n': n, 'Mean SCC/yr': mu, 'CI low': lo, 'CI high': hi}
          for k, mu, lo, hi, n in zip([1, 2], on_means, on_lo, on_hi, on_ns)]
tbl_pre_on = pd.DataFrame(_rows)
tbl_pre_on.to_csv('./table_pre_on_overall.csv', index=False)
with pd.option_context('display.max_rows', None, 'display.width', 120):
    print(tbl_pre_on.to_string(index=False, float_format='%.3f'))
print("  ✓ Saved to ./table_pre_on_overall.csv")

In [ ]:
# %% NEW CELL — Figure 4 (stratified) — Pre + On annualized SCC rate by on-drug SCC status
# Two trajectories (zero vs any on-drug SCC). Two-panel: full history (left) + ±2yr zoom (right).
# Run after df['is_zero_scc'] exists. Uses: pre_scc, scc_month_cens, df, C, np, pd, plt
# on-drug SCC status derived locally so this cell can sit right after the unstratified fig,
# before df['is_zero_scc'] is created in the Zero-SCC analysis cell (same definition as there)
_zero_on_drug = scc_month_cens.sum(axis=1, min_count=1).fillna(0) == 0

strata = [
    ('Zero on-drug SCC', _zero_on_drug,  C['plum']),
    ('Any on-drug SCC',  ~_zero_on_drug, C['coral']),
]
def compute_pre(mask):
    means, lo, hi, ns = [], [], [], []
    for yr in range(10, 0, -1):
        vals = pre_scc.loc[mask, f'yr{yr}'].dropna()
        n = len(vals)
        if n == 0:
            means.append(np.nan); lo.append(np.nan); hi.append(np.nan); ns.append(0); continue
        mu = vals.mean()
        se = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0.0
        means.append(mu); lo.append(mu - 1.96 * se); hi.append(mu + 1.96 * se); ns.append(n)
    return means, lo, hi, ns

def compute_on(mask):
    means, lo, hi, ns = [], [], [], []
    for m_start, m_end in [(1, 12), (13, 24)]:
        cols = [f'm{m}' for m in range(m_start, m_end + 1)]
        n_months  = scc_month_cens.loc[mask, cols].notna().sum(axis=1)
        total_scc = scc_month_cens.loc[mask, cols].sum(axis=1, min_count=1)
        sub = n_months >= 3
        if sub.sum() == 0:
            means.append(np.nan); lo.append(np.nan); hi.append(np.nan); ns.append(0); continue
        weights    = n_months[sub].values.astype(float)
        annualized = (total_scc[sub] / n_months[sub]) * 12
        mu = np.average(annualized, weights=weights)
        w_var = np.average((annualized - mu) ** 2, weights=weights)
        sum_w, sum_w2 = weights.sum(), (weights ** 2).sum()
        n_eff = sum_w ** 2 / sum_w2
        se = np.sqrt(w_var / n_eff) if n_eff > 0 else 0.0
        means.append(mu); lo.append(mu - 1.96 * se); hi.append(mu + 1.96 * se); ns.append(int(sub.sum()))
    return means, lo, hi, ns

x_pre = list(range(-9, 1))   # -9…0  → Pre 10 … Pre 1
x_on  = [1, 2]               #  1, 2 → On 1, On 2

strat_data = {label: {'color': color, 'pre': compute_pre(mask), 'on': compute_on(mask)}
              for label, mask, color in strata}

def draw_stratum(a, xpre, pre, xon, on, color, label, n_side):
    pm, pl, ph, pn = pre
    om, ol, oh, on_n = on
    a.fill_between(xpre, pl, ph, alpha=0.15, color=color)
    a.plot(xpre, pm, 'o-', color=color, markersize=4, lw=2, label=label)
    a.plot([xpre[-1], xon[0]], [pm[-1], om[0]], '-', color=color, lw=2, alpha=0.9)
    a.fill_between([xpre[-1], xon[0]], [pl[-1], ol[0]], [ph[-1], oh[0]], alpha=0.10, color=color)
    a.fill_between(xon, ol, oh, alpha=0.15, color=color)
    a.plot(xon, om, 'o-', color=color, markersize=4, lw=2)
    dy, va = 0.17 * n_side, ('bottom' if n_side > 0 else 'top')
    for xi, mu, n in list(zip(xpre, pm, pn)) + list(zip(xon, om, on_n)):
        if n > 0 and np.isfinite(mu):
            a.text(xi, mu + dy, f'{n}', ha='center', fontsize=6, color=color, va=va,
                   bbox=dict(boxstyle='square,pad=0.12', facecolor='white',
                             edgecolor='none', alpha=0.72))

def finish_panel(a, all_x, title, show_legend):
    a.axvline(0, color=C['navy'], ls='-', lw=1.5, alpha=0.35)
    ymax = max(a.get_ylim()[1], 2.8)
    a.text(0, ymax, 'Acitretin start', ha='center', fontsize=8.5,
           color=C['navy'], alpha=0.7, va='bottom')
    tick_labels = [f'Pre {abs(xi)+1}' if xi < 0 else ('Pre 1' if xi == 0 else f'On {xi}')
                   for xi in all_x]
    a.set_xticks(all_x); a.set_xticklabels(tick_labels, fontsize=7.5, rotation=45, ha='right')
    a.set_xlabel('Years relative to acitretin start', fontweight='bold')
    a.set_ylabel('Mean SCC count per year', fontweight='bold')
    a.set_title(title, fontsize=12, pad=10)
    if show_legend:
        a.legend(fontsize=8, loc='upper left', framealpha=0.9, edgecolor='#666666')
    for sp in ['top', 'right']:  a.spines[sp].set_visible(False)
    for sp in ['left', 'bottom']:
        a.spines[sp].set_linewidth(2.0); a.spines[sp].set_color('#333333')

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [3, 1.4]})

for i, (label, mask, color) in enumerate(strata):
    d = strat_data[label]
    draw_stratum(ax, x_pre, d['pre'], x_on, d['on'], color,
                 f"{label} (n={int(mask.sum())})", n_side=+1 if i == 0 else -1)
finish_panel(ax, x_pre + x_on,
             'Pre-treatment SCC history & on-treatment rate\nby on-drug SCC status', True)

for i, (label, mask, color) in enumerate(strata):
    d = strat_data[label]
    zoom_pre = tuple(arr[-2:] for arr in d['pre'])
    draw_stratum(ax2, x_pre[-2:], zoom_pre, x_on, d['on'], color, label,
                 n_side=+1 if i == 0 else -1)
finish_panel(ax2, x_pre[-2:] + x_on, '±2 years around start', False)

plt.savefig(os.path.join(FIGURE_DIR, 'fig04_pre_on_by_zerostatus.png'), bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# %% NEW CELL — Figure 4 (stratified) — Pre + On annualized SCC rate by IMMUNE STATUS
# Requested 2026-08-15: same construction as the by-on-drug-SCC-status figure
# above, restratified into immunocompetent vs immunocompromised.
# Two-panel: full history (left) + ±2 yr zoom (right).
# Uses the compute_pre / compute_on / draw_stratum / finish_panel helpers
# defined in the preceding cell, so the two figures are identical apart from
# how patients are split.

strata_imm = [
    ('Immunocompetent',   df['immune'] == 'Immunocompetent',  C['plum']),
    ('Immunocompromised', df['immune'] == 'Immunosuppressed', C['coral']),
]

x_pre = list(range(-9, 1))   # -9…0  → Pre 10 … Pre 1
x_on  = [1, 2]               #  1, 2 → On 1, On 2

strat_imm_data = {label: {'color': color, 'pre': compute_pre(mask), 'on': compute_on(mask)}
                  for label, mask, color in strata_imm}

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [3, 1.4]})

for i, (label, mask, color) in enumerate(strata_imm):
    d = strat_imm_data[label]
    draw_stratum(ax, x_pre, d['pre'], x_on, d['on'], color,
                 f"{label} (n={int(mask.sum())})", n_side=+1 if i == 0 else -1)
finish_panel(ax, x_pre + x_on,
             'Pre-treatment SCC history & on-treatment rate\nby immune status', True)

for i, (label, mask, color) in enumerate(strata_imm):
    d = strat_imm_data[label]
    zoom_pre = tuple(arr[-2:] for arr in d['pre'])
    draw_stratum(ax2, x_pre[-2:], zoom_pre, x_on, d['on'], color, label,
                 n_side=+1 if i == 0 else -1)
finish_panel(ax2, x_pre[-2:] + x_on, '±2 years around start', False)

plt.savefig(os.path.join(FIGURE_DIR, 'fig04_pre_on_by_immune.png'), bbox_inches='tight', dpi=600)
plt.show()
print("  ✓ Saved to figures/fig04_pre_on_by_immune.png")

# ── Companion table: the plotted numbers, so they can be quoted in text ────
_rows = []
for label, mask, _ in strata_imm:
    d = strat_imm_data[label]
    pm, pl, ph, pn = d['pre']
    om, ol, oh, on_n = d['on']
    for yr, mu, lo, hi, n in zip(range(10, 0, -1), pm, pl, ph, pn):
        _rows.append({'Stratum': label, 'Period': f'Pre {yr}', 'n': n,
                      'Mean SCC/yr': mu, 'CI low': lo, 'CI high': hi})
    for k, mu, lo, hi, n in zip([1, 2], om, ol, oh, on_n):
        _rows.append({'Stratum': label, 'Period': f'On {k}', 'n': n,
                      'Mean SCC/yr': mu, 'CI low': lo, 'CI high': hi})
tbl_imm_traj = pd.DataFrame(_rows)
tbl_imm_traj.to_csv('./table_pre_on_by_immune.csv', index=False)
with pd.option_context('display.max_rows', None, 'display.width', 140):
    print(tbl_imm_traj.to_string(index=False, float_format='%.3f'))
print("  ✓ Saved to ./table_pre_on_by_immune.csv")

In [ ]:
# ============================================================================
# FIGURE — Within-patient pre vs on-acitretin (connected dots / spaghetti)
# ============================================================================
# Uses: pre_scc, scc_month_cens, df, C
#
# Readability notes. With ~170 lines the limiting factor is overplotting, not
# hue, so three things do the work here:
#   · direction is carried by two colours far apart in both hue and lightness
#     (purple down, coral up) at an alpha high enough to see a single line but
#     low enough that crossings still read as density;
#   · both endpoints are jittered horizontally, because pre-treatment counts
#     are integers and a third of the on-drug rates are exactly 0, so without
#     jitter every line lands on one of a handful of points;
#   · a box at each margin shows the distribution the spaghetti cannot.

# ── On-acitretin annualized rate (all on-drug months, per patient) ──────────
on_total = scc_month_cens.sum(axis=1, min_count=1)
on_months = scc_month_cens.notna().sum(axis=1)
on_annual = (on_total / on_months) * 12
on_annual[on_months < 3] = np.nan

# ── Pre-treatment baseline (Year −1) ───────────────────────────────────────
pre_yr1 = pre_scc['yr1']

both = pre_yr1.notna() & on_annual.notna()
n_down = int((on_annual[both] < pre_yr1[both]).sum())
n_up   = int((on_annual[both] >= pre_yr1[both]).sum())

C_DOWN, C_UP = C['plum'], C['coral']

# ── Figure ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9.2, 6.0))

# Margin boxes: the distribution at each timepoint, behind everything
bp = ax.boxplot([pre_yr1[both].values, on_annual[both].values],
                positions=[-0.20, 1.20], widths=0.11, vert=True,
                showfliers=False, patch_artist=True, zorder=1)
for patch in bp['boxes']:
    patch.set(facecolor='#EDEAF0', edgecolor='#9A93A3', linewidth=1.2)
for part in ('whiskers', 'caps'):
    for ln in bp[part]:
        ln.set(color='#9A93A3', linewidth=1.2)
for ln in bp['medians']:
    ln.set(color='#4A4458', linewidth=2.0)

# Spaghetti, jittered at both ends so the integer pile-ups fan out
rng = np.random.default_rng(0)
idx_down = [k for k in df.index[both] if on_annual[k] <  pre_yr1[k]]
idx_up   = [k for k in df.index[both] if on_annual[k] >= pre_yr1[k]]

for group, color, z in ((idx_down, C_DOWN, 2), (idx_up, C_UP, 3)):
    for k in group:
        x0, x1 = rng.normal(0, 0.030), rng.normal(1, 0.030)
        y0, y1 = pre_yr1[k], on_annual[k]
        ax.plot([x0, x1], [y0, y1], '-', color=color, alpha=0.34, lw=0.9,
                zorder=z, solid_capstyle='round')
        ax.plot([x0, x1], [y0, y1], '.', color=color, alpha=0.40, ms=3.0,
                zorder=z)

# Mean
ax.plot([0, 1], [pre_yr1[both].mean(), on_annual[both].mean()],
        'D-', color=C['navy'], markersize=10, lw=3.0, zorder=6,
        markeredgecolor='white', markeredgewidth=1.4)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Year −1 pre-treatment',
                    'On acitretin\n(all months, annualized)'], fontsize=10)
ax.set_ylabel('Annualized SCC rate (SCCs / year)', fontweight='bold')
ax.set_title(f'Within-patient paired comparison\n'
             f'{n_down} decreased · {n_up} increased or same',
             fontsize=11)

from matplotlib.lines import Line2D
from matplotlib.patches import Patch
legend_els = [
    Line2D([0], [0], color=C_DOWN, lw=2.2, alpha=0.85, label=f'Decreased (n={n_down})'),
    Line2D([0], [0], color=C_UP,   lw=2.2, alpha=0.85, label=f'Increased / same (n={n_up})'),
    Line2D([0], [0], color=C['navy'], lw=3.0, marker='D', markersize=7,
           markeredgecolor='white', label=f'Mean (n={int(both.sum())})'),
    Patch(facecolor='#EDEAF0', edgecolor='#9A93A3', label='Median and IQR'),
]
ax.legend(handles=legend_els, fontsize=8, loc='upper left',
          framealpha=0.95, edgecolor='#666666')

ax.grid(axis='y', color='#DDDDDD', lw=0.6, alpha=0.7, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(2.0)
ax.spines['bottom'].set_linewidth(2.0)
ax.spines['left'].set_color('#333333')
ax.spines['bottom'].set_color('#333333')
ax.set_xlim(-0.30, 1.30)

plt.savefig(os.path.join(FIGURE_DIR, 'fig_spaghetti_paired.png'), bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# %% FIGURE — Patient flow between SCC-rate bands (pre → on acitretin)
from matplotlib.path import Path
from matplotlib.patches import PathPatch, Patch

on_total  = scc_month_cens.sum(axis=1, min_count=1)
on_months = scc_month_cens.notna().sum(axis=1)
on_annual = (on_total / on_months) * 12
on_annual[on_months < 3] = np.nan
pre_yr1 = pre_scc['yr1']
both = pre_yr1.notna() & on_annual.notna()

def rate_cat(r):
    if r <= 0:  return 0
    if r <= 1:  return 1
    if r <= 3:  return 2
    return 3
CAT = ['x = 0', '0 < x ≤ 1', '1 < x ≤ 3', '3 < x']
NCAT = len(CAT)

pc = pre_yr1[both].map(rate_cat).astype(int).values
oc = on_annual[both].map(rate_cat).astype(int).values
M  = np.zeros((NCAT, NCAT), int)
for a, b in zip(pc, oc):
    M[a, b] += 1
N = int(M.sum())

left_tot, right_tot = M.sum(axis=1), M.sum(axis=0)
gap = 0.02 * N
def stack_tops(totals):
    tops, y = {}, N + gap * (NCAT - 1)
    for i, t in enumerate(totals):
        tops[i] = y; y -= t + gap
    return tops
left_top, right_top = stack_tops(left_tot), stack_tops(right_tot)

fig, ax = plt.subplots(figsize=(8, 6))
xL, xR, bw = 0.0, 1.0, 0.04
band_colors = [C['slate'], C['mauve'], C['plum'], C['coral']]
for i in range(NCAT):
    ax.add_patch(plt.Rectangle((xL - bw, left_top[i] - left_tot[i]), bw, left_tot[i],
                               color=band_colors[i], alpha=0.9, lw=0))
    ax.add_patch(plt.Rectangle((xR, right_top[i] - right_tot[i]), bw, right_tot[i],
                               color=band_colors[i], alpha=0.9, lw=0))
    if left_tot[i]:
        ax.text(xL - bw - 0.015, left_top[i] - left_tot[i]/2, f'{CAT[i]}  (n={left_tot[i]})',
                ha='right', va='center', fontsize=8)
    if right_tot[i]:
        ax.text(xR + bw + 0.015, right_top[i] - right_tot[i]/2, f'{CAT[i]}  (n={right_tot[i]})',
                ha='left', va='center', fontsize=8)

def ribbon(y0a, y0b, y1a, y1b, color):
    verts = [(xL, y0a), (0.5, y0a), (0.5, y1a), (xR, y1a),
             (xR, y1b), (0.5, y1b), (0.5, y0b), (xL, y0b), (xL, y0a)]
    codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4,
             Path.LINETO, Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.CLOSEPOLY]
    ax.add_patch(PathPatch(Path(verts, codes), facecolor=color, edgecolor='none', alpha=0.45))

left_used  = {i: 0.0 for i in range(NCAT)}
right_used = {j: 0.0 for j in range(NCAT)}
for i in range(NCAT):
    for j in range(NCAT):
        c = M[i, j]
        if not c: continue
        y0a = left_top[i]  - left_used[i]
        y1a = right_top[j] - right_used[j]
        col = C['plum'] if j < i else (C['coral'] if j > i else C['slate'])
        ribbon(y0a, y0a - c, y1a, y1a - c, col)
        left_used[i]  += c
        right_used[j] += c

top_y = N + gap * (NCAT - 1)
ax.set_xlim(-0.30, 1.30); ax.set_ylim(-gap, top_y + gap)
ax.axis('off')
ax.text(xL - bw/2, top_y + gap*0.5, 'Pre-treatment\n(Year −1)',
        ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.text(xR + bw/2, top_y + gap*0.5, 'On acitretin\n(annualized)',
        ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title(f'Patient flow between SCC-rate bands (n={N})', fontsize=12, pad=24)
ax.legend(handles=[Patch(facecolor=C['plum'],  alpha=0.6, label='To lower band'),
                   Patch(facecolor=C['slate'], alpha=0.6, label='Same band'),
                   Patch(facecolor=C['coral'], alpha=0.6, label='To higher band')],
          fontsize=8, loc='lower center', ncol=3, framealpha=0.9,
          edgecolor='#666666', bbox_to_anchor=(0.5, -0.06))
plt.savefig(os.path.join(FIGURE_DIR, 'fig_flow_bands.png'), bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# ============================================================================
# FIGURE 9 — Pre-treatment baseline vs on-acitretin SCC rate
# ============================================================================
# Paste after data-prep cells (1–6).
# Uses: pre_scc, scc_month_cens, df, C

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# ── Compute per-patient pre-treatment annual rates ─────────────────────────
# A patient enters a window only if every year in it is recorded (methods:
# "patients were excluded from that specific analysis if any year within the
# window lacked complete data"), so n falls as the window lengthens.
def _complete_window_rate(years):
    cols = [f'yr{y}' for y in years]
    return pre_scc[cols].mean(axis=1).where(pre_scc[cols].notna().all(axis=1))

pre_1_10 = _complete_window_rate(range(1, 11))
pre_1_5  = _complete_window_rate(range(1, 6))
pre_1_2  = _complete_window_rate([1, 2])
pre_1    = pre_scc['yr1']

# ── On-acitretin: per-patient annualized rate, weighted by months observed ─
on_total  = scc_month_cens.sum(axis=1, min_count=1)
on_months = scc_month_cens.notna().sum(axis=1)
on_annual = (on_total / on_months) * 12
on_mask   = on_months >= 3          # same ≥3-month threshold as Fig 4


# ═══════════════════════════════════════════════════════════════════════════
# LEFT PANEL — Overall
# ═══════════════════════════════════════════════════════════════════════════
ax = axes[0]

means = [pre_1_10.mean(), pre_1_5.mean(), pre_1_2.mean(),
         pre_1.mean(),
         np.average(on_annual[on_mask], weights=on_months[on_mask])]
ns    = [pre_1_10.notna().sum(), pre_1_5.notna().sum(),
         pre_1_2.notna().sum(), pre_1.notna().sum(),
         on_mask.sum()]
labels = ['Pre 1–10', 'Pre 1–5', 'Pre 1–2', 'Pre 1',
          'On\nacitretin']
colors_bar = [C['plum'], C['plum'], C['plum'], C['plum'], C['pink']]

bars = ax.bar(range(len(labels)), means,
              color=colors_bar, edgecolor='#2B2D42', lw=1.8, alpha=0.90)

for b, v, n in zip(bars, means, ns):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.05,
            f'{v:.2f}\nn={n}', ha='center', fontsize=8.5, color='#444')

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Annualized SCC rate (SCCs / year)', fontweight='bold')
ax.set_ylim([0, 3.5])
ax.grid(False)
ax.set_title('Pre-treatment baseline vs on-acitretin rate')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(2.0)
ax.spines['bottom'].set_linewidth(2.0)
ax.spines['left'].set_color('#333333')
ax.spines['bottom'].set_color('#333333')


# ═══════════════════════════════════════════════════════════════════════════
# RIGHT PANEL — Stratified by immune status
# ═══════════════════════════════════════════════════════════════════════════
ax = axes[1]

strat_labels = ['Pre 1–10', 'Pre 1–5', 'Pre 1–2', 'Pre 1',
                'On\nacitretin']
x = np.arange(len(strat_labels))
width = 0.38
display_name = {'Immunosuppressed': 'Immunocompromised', 'Immunocompetent': 'Immunocompetent'}

for i, (status, pre_color, on_color) in enumerate(zip(
        ['Immunosuppressed', 'Immunocompetent'],
        [C['plum'], C['mauve']],
        [C['pink'], C['purple']])):
    mask = df['immune'] == status

    s_means, s_ns = [], []
    for series in [pre_1_10, pre_1_5, pre_1_2, pre_1]:
        vals = series[mask].dropna()
        s_means.append(vals.mean())
        s_ns.append(len(vals))

    # On-drug (weighted)
    on_m = on_mask & mask
    if on_m.sum() > 0:
        s_means.append(np.average(on_annual[on_m],
                                  weights=on_months[on_m]))
    else:
        s_means.append(np.nan)
    s_ns.append(on_m.sum())

    # Per-bar colors: pre bars in group color, on-acitretin bar shifts
    bar_colors = [pre_color] * 4 + [on_color]

    bars = ax.bar(x + i * width - width / 2, s_means, width,
                  color=bar_colors, alpha=0.8, edgecolor='#2B2D42', lw=1.8)

    # Legend entries (one pre + one on per group)
    bars[0].set_label(f'{display_name[status]}, pre-treatment')
    bars[4].set_label(f'{display_name[status]}, on acitretin')

    for b, v, n in zip(bars, s_means, s_ns):
        if pd.notna(v):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.05,
                    f'{v:.2f}\nn={n}', ha='center', fontsize=7, color='#444')

ax.set_xticks(x)
ax.set_xticklabels(strat_labels, fontsize=9)
ax.set_ylabel('Annualized SCC rate (SCCs / year)', fontweight='bold')
ax.set_ylim([0, 3.5])
ax.grid(False)
ax.set_title('Stratified by immune status')
ax.legend(fontsize=8, loc='upper left', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(2.0)
ax.spines['bottom'].set_linewidth(2.0)
ax.spines['left'].set_color('#333333')
ax.spines['bottom'].set_color('#333333')

fig.suptitle('Pre-treatment baseline comparison & on-acitretin SCC rate',
             fontsize=15, fontweight='bold', y=1.07)
plt.savefig(os.path.join(FIGURE_DIR, 'fig09_pre_post.png'), bbox_inches='tight', dpi=600)
plt.show()

print("Figure 9 windows (complete pre-treatment years only):")
for _lab, _ser in [('Pre 1-10', pre_1_10), ('Pre 1-5', pre_1_5),
                   ('Pre 1-2', pre_1_2), ('Pre 1', pre_1)]:
    _parts = [f"{_lab:9s} all n={_ser.notna().sum():3d} mean={_ser.mean():.2f}"]
    for _st in ['Immunosuppressed', 'Immunocompetent']:
        _v = _ser[df['immune'] == _st].dropna()
        _parts.append(f"{_st[:10]} n={len(_v):3d} mean={_v.mean():.2f}")
    print("   " + "   ".join(_parts))
print(f"   On acitretin  n={int(on_mask.sum())}  "
      f"weighted mean={np.average(on_annual[on_mask], weights=on_months[on_mask]):.2f}")

In [ ]:
# %% Final figure: MCF + trajectories (top), swimmer (bottom, full width)

fig = plt.figure(figsize=(12, 18))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 2])

ax_mcf = fig.add_subplot(gs[0, 0])
ax_traj = fig.add_subplot(gs[0, 1])
ax_swim = fig.add_subplot(gs[1, :])

months = list(range(1, 25))

# Build per-patient cumulative SCC
cum_scc = pd.DataFrame(index=df.index, columns=months, dtype=float)
for m in months:
    if m == 1:
        cum_scc[m] = scc_month_cens['m1']
    else:
        cum_scc[m] = cum_scc[m - 1].where(
            scc_month_cens[f'm{m}'].isna(),
            cum_scc[m - 1].fillna(0) + scc_month_cens[f'm{m}'].fillna(0))
        cum_scc.loc[scc_month_cens[f'm{m}'].isna(), m] = np.nan

mcf_vals, mcf_lo, mcf_hi, mcf_ns = [], [], [], []
for m in months:
    vals = cum_scc[m].dropna()
    n = len(vals)
    mu = vals.mean() if n > 0 else np.nan
    se = vals.std() / np.sqrt(n) if n > 1 else 0
    mcf_vals.append(mu)
    mcf_lo.append(mu - 1.96 * se)
    mcf_hi.append(mu + 1.96 * se)
    mcf_ns.append(n)

pre_yr1_month = pre_scc['yr1'].mean() / 12
ref_yr1 = [pre_yr1_month * m for m in months]

# ============================================================
# TOP LEFT: MCF
# ============================================================
ax = ax_mcf
ax.fill_between(months, mcf_lo, mcf_hi, alpha=0.2, color=C['mauve'])
ax.plot(months, mcf_vals, 'o-', color=C['plum'], markersize=4, lw=2.5, label='MCF (on-drug)')
ax.plot(months, ref_yr1, '--', color=C['pink'], lw=2,
        label=f'Expected if Year-1 rate\n({pre_scc["yr1"].mean():.2f}/yr)')

for i, m in enumerate(months):
    if m in [1, 6, 12, 18, 24]:
        ax.text(m, mcf_vals[i] + 0.2, f'n={mcf_ns[i]}', ha='center', fontsize=7, color=C['slate'])

ax.set_xlabel('Months after acitretin start')
ax.set_ylabel('Mean cumulative SCC per patient')
ax.set_title('Mean Cumulative Function')
ax.legend(fontsize=8, loc='upper left')
ax.set_xticks([1, 3, 6, 9, 12, 15, 18, 21, 24])
ax.set_xlim([1, 24])

# ============================================================
# TOP RIGHT: Individual cumulative trajectories
# ============================================================
ax = ax_traj

for idx in cum_scc.index:
    row = cum_scc.loc[idx].dropna()
    if len(row) >= 3:
        ax.plot(row.index, row.values, color=C['mauve'], alpha=0.2, lw=1.0)

ax.plot(months, mcf_vals, '-', color=C['plum'], lw=2.5, zorder=5, label='MCF (mean)')
ax.plot(months, ref_yr1, '--', color=C['pink'], lw=2, zorder=5,
        label='Expected if Year-1 rate')

ax.set_xlabel('Months after acitretin start')
ax.set_ylabel('Cumulative SCC count')
ax.set_title('Individual cumulative trajectories')
ax.legend(fontsize=8, loc='upper left')
ax.set_xticks([1, 3, 6, 9, 12, 15, 18, 21, 24])
ax.set_xlim([1, 24])

# ============================================================
# BOTTOM: Swimmer plot
# ============================================================
ax = ax_swim

n_patients = len(df)
sort_order = df.sort_values(['months_on_drug', 'first_off_month'], ascending=[True, True]).index

ax.grid(False)
for m in range(1, 25):
    if m % 2 == 0:
        ax.add_patch(mpatches.Rectangle(
            (m - 0.5, -0.5), 1, n_patients,
            facecolor=C['slate'], alpha=0.06, edgecolor='none', zorder=0))

for y in np.arange(-0.5, n_patients + 0.5, 1):
    ax.axhline(y, color='grey', alpha=0.2, lw=0.3, zorder=0)

for rank, idx in enumerate(sort_order):
    dur = df.loc[idx, 'months_on_drug']

    rect = mpatches.Rectangle(
        (0.5, rank - 0.5), dur, 1.0,
        facecolor=C['mauve'], alpha=0.25, edgecolor='none', zorder=1)
    ax.add_patch(rect)

    for m in range(1, int(dur) + 1):
        val = int(scc_month_cens.loc[idx, f'm{m}'])
        if val > 0:
            if val == 1:
                offsets = [0]
            elif val == 2:
                offsets = [-0.2, 0.2]
            else:
                offsets = np.linspace(-0.35, 0.35, val)
            for off in offsets:
                ax.plot(m + off, rank, 'o', color=C['purple'], markersize=4,
                        markeredgecolor='white', markeredgewidth=0.4, zorder=3)

    if dur < 24:
        ax.plot(dur + 0.5, rank, 's', color=C['navy'], markersize=3,
                markeredgecolor='none', zorder=2, alpha=0.7)

from matplotlib.lines import Line2D
legend_els = [
    mpatches.Patch(color=C['mauve'], alpha=0.25, label='On drug'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C['purple'],
           markersize=8, label='SCC event (each dot = 1)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=C['navy'],
           markersize=5, alpha=0.7, label='Censored (stopped drug)'),
    mpatches.Patch(facecolor=C['slate'], alpha=0.06, label='Even months (shading)'),
]
ax.legend(handles=legend_els, fontsize=9, loc='lower right',
          framealpha=0.9, edgecolor='#666666')
ax.set_xlabel('Months after acitretin start', fontsize=11)
ax.set_ylabel(f'Patients (n={n_patients})', fontsize=11)
ax.set_yticks([])
ax.set_xlim(0.5, 24.5)
ax.set_ylim(-0.5, n_patients - 0.5)
ax.set_xticks(range(1, 25))
ax.set_xticklabels([str(m) for m in range(1, 25)], fontsize=9)
ax.set_title('Individual patient timelines', fontsize=12)

fig.suptitle('Post-treatment SCC accumulation (censored at first discontinuation)',
             fontsize=14, y=1.01)
plt.savefig(os.path.join(FIGURE_DIR, 'fig_mcf_swimmer.png'), bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
# %% Final figure: MCF + trajectories (top), swimmer (bottom)
# ------------------------------------------------------------------
# REPLACES the previous cell. Key change: the MCF panel now uses the
# formal Nelson-Aalen estimator with Lawless-Nadeau variance, rather
# than averaging per-patient running totals over the at-risk set.
#
# Nelson-Aalen MCF:
#     M̂(t) = Σ_{m ≤ t}  dN(m) / Y(m)
#   where Y(m) = # patients on drug at month m
#         dN(m) = total SCCs at month m across the at-risk set
#
# Lawless-Nadeau robust pointwise variance:
#     var(M̂(t)) = Σ_i [ Σ_{m ≤ t} δ_i(m) · (n_i(m) - dM̂(m)) / Y(m) ]²
#   where δ_i(m) = 1 if patient i is on drug at month m, n_i(m) = SCCs that
#   month. Summing each patient's residuals over months before squaring keeps
#   the within-patient correlation of SCC counts; no Poisson assumption.
#   (Corrected 2026-09-21: the earlier version summed per-month variances,
#   which treats a patient's months as independent and understates the band.)
#
# Reference: Lawless JF, Nadeau C. Technometrics 1995; 37(2):158-168.
# Equivalent to lifelines.NelsonAalenFitter if you prefer a named dep.
# ------------------------------------------------------------------

fig = plt.figure(figsize=(12, 18))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 2])

ax_mcf  = fig.add_subplot(gs[0, 0])
ax_traj = fig.add_subplot(gs[0, 1])
ax_swim = fig.add_subplot(gs[1, :])

months = list(range(1, 25))

# ============================================================
# Nelson-Aalen MCF + Lawless-Nadeau variance
# ============================================================
at_risk   = np.zeros(25)   # Y(m);  index 0 unused, 1..24 are months
dN        = np.zeros(25)   # total events at month m
dM_hat    = np.zeros(25)   # event rate increment at month m
ln_resid  = pd.DataFrame(0.0, index=df.index, columns=months)  # δ_i(m)(n_i(m) - dM̂(m))/Y(m)

for m in months:
    contributing = scc_month_cens[f'm{m}'].notna()
    Y_m = int(contributing.sum())
    at_risk[m] = Y_m
    if Y_m > 0:
        vals = scc_month_cens.loc[contributing, f'm{m}'].values
        dN[m]     = vals.sum()
        dM_hat[m] = vals.mean()                                   # = dN(m)/Y(m)
        ln_resid.loc[contributing, m] = (vals - dM_hat[m]) / Y_m

mcf     = np.cumsum(dM_hat[1:])            # length 24, indexed to months 1..24
mcf_var = (ln_resid.cumsum(axis=1) ** 2).sum(axis=0).values   # Lawless-Nadeau robust
mcf_se  = np.sqrt(mcf_var)
mcf_lo  = mcf - 1.96 * mcf_se
mcf_hi  = mcf + 1.96 * mcf_se
mcf_ns  = at_risk[1:].astype(int)          # at-risk set size per month

# ============================================================
# Per-patient cumulative SCC (for individual-trajectories panel
# and for the swimmer-plot event dots — unchanged from before)
# ============================================================
cum_scc = pd.DataFrame(index=df.index, columns=months, dtype=float)
for m in months:
    if m == 1:
        cum_scc[m] = scc_month_cens['m1']
    else:
        cum_scc[m] = cum_scc[m - 1].where(
            scc_month_cens[f'm{m}'].isna(),
            cum_scc[m - 1].fillna(0) + scc_month_cens[f'm{m}'].fillna(0))
        cum_scc.loc[scc_month_cens[f'm{m}'].isna(), m] = np.nan

# Reference line: expected accrual if Year-1 rate persisted linearly.
# NOTE: this is the same reference as the original figure; consider
# using a longer pre-treatment window (e.g. mean of Years 1-5) to
# reduce indication bias — see separate discussion.
pre_yr1_month = pre_scc['yr1'].mean() / 12
ref_yr1 = [pre_yr1_month * m for m in months]

# ============================================================
# TOP LEFT: MCF (Nelson-Aalen)
# ============================================================
ax = ax_mcf
ax.fill_between(months, mcf_lo, mcf_hi, alpha=0.2, color=C['mauve'])
ax.plot(months, mcf, 'o-', color=C['plum'], markersize=4, lw=2.5,
        label='Nelson-Aalen MCF')
ax.plot(months, ref_yr1, '--', color=C['pink'], lw=2,
        label=f'Expected if Year-1 rate\n({pre_scc["yr1"].mean():.2f}/yr)')

for i, m in enumerate(months):
    if m in [1, 6, 12, 18, 24]:
        # below the band (the reference line runs above it); edge labels
        # anchored inward so the axes frame does not cut them
        ax.text(m + {1: 0.2, 24: -0.2}.get(m, 0), mcf_lo[i] - 0.08, f'Y={mcf_ns[i]}',
                ha={1: 'left', 24: 'right'}.get(m, 'center'), va='top',
                fontsize=7, color=C['slate'])
ax.set_ylim(bottom=-0.45)

ax.set_xlabel('Months after acitretin start')
ax.set_ylabel('Mean cumulative SCC per patient')
ax.set_title('Mean Cumulative Function (Nelson-Aalen)')
ax.legend(fontsize=8, loc='upper left')
ax.set_xticks([1, 3, 6, 9, 12, 15, 18, 21, 24])
ax.set_xlim([1, 24])

# ============================================================
# TOP RIGHT: Individual cumulative trajectories
# ============================================================
ax = ax_traj
for idx in cum_scc.index:
    row = cum_scc.loc[idx].dropna()
    if len(row) >= 3:
        ax.plot(row.index, row.values, color=C['mauve'], alpha=0.2, lw=1.0)

ax.plot(months, mcf, '-', color=C['plum'], lw=2.5, zorder=5,
        label='Nelson-Aalen MCF')
ax.plot(months, ref_yr1, '--', color=C['pink'], lw=2, zorder=5,
        label='Expected if Year-1 rate')

ax.set_xlabel('Months after acitretin start')
ax.set_ylabel('Cumulative SCC count')
ax.set_title('Individual cumulative trajectories')
ax.legend(fontsize=8, loc='upper left')
ax.set_xticks([1, 3, 6, 9, 12, 15, 18, 21, 24])
ax.set_xlim([1, 24])

# ============================================================
# BOTTOM: Swimmer plot (unchanged — raw data display)
# ============================================================
ax = ax_swim

n_patients = len(df)
sort_order = df.sort_values(['months_on_drug', 'first_off_month'],
                             ascending=[True, True]).index

ax.grid(False)
for m in range(1, 25):
    if m % 2 == 0:
        ax.add_patch(mpatches.Rectangle(
            (m - 0.5, -0.5), 1, n_patients,
            facecolor=C['slate'], alpha=0.06, edgecolor='none', zorder=0))

for y in np.arange(-0.5, n_patients + 0.5, 1):
    ax.axhline(y, color='grey', alpha=0.2, lw=0.3, zorder=0)

for rank, idx in enumerate(sort_order):
    dur = df.loc[idx, 'months_on_drug']
    ax.add_patch(mpatches.Rectangle(
        (0.5, rank - 0.5), dur, 1.0,
        facecolor=C['mauve'], alpha=0.25, edgecolor='none', zorder=1))

    for m in range(1, int(dur) + 1):
        val = int(scc_month_cens.loc[idx, f'm{m}'])
        if val > 0:
            if val == 1:
                offsets = [0]
            elif val == 2:
                offsets = [-0.2, 0.2]
            else:
                offsets = np.linspace(-0.35, 0.35, val)
            for off in offsets:
                ax.plot(m + off, rank, 'o', color=C['purple'], markersize=4,
                        markeredgecolor='white', markeredgewidth=0.4, zorder=3)

    if dur < 24:
        ax.plot(dur + 0.5, rank, 's', color=C['navy'], markersize=3,
                markeredgecolor='none', zorder=2, alpha=0.7)

from matplotlib.lines import Line2D
legend_els = [
    mpatches.Patch(color=C['mauve'], alpha=0.25, label='On drug'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C['purple'],
           markersize=8, label='SCC event (each dot = 1)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=C['navy'],
           markersize=5, alpha=0.7, label='Censored (stopped drug)'),
    mpatches.Patch(facecolor=C['slate'], alpha=0.06, label='Even months (shading)'),
]
ax.legend(handles=legend_els, fontsize=9, loc='lower right',
          framealpha=0.9, edgecolor='#666666')
ax.set_xlabel('Months after acitretin start', fontsize=11)
ax.set_ylabel(f'Patients (n={n_patients})', fontsize=11)
ax.set_yticks([])
ax.set_xlim(0.5, 24.5)
ax.set_ylim(-0.5, n_patients - 0.5)
ax.set_xticks(range(1, 25))
ax.set_xticklabels([str(m) for m in range(1, 25)], fontsize=9)
ax.set_title('Individual patient timelines', fontsize=12)

fig.suptitle('Post-treatment SCC accumulation (censored at first discontinuation)',
             fontsize=14, y=1.01)
plt.savefig(os.path.join(FIGURE_DIR, 'fig_mcf_swimmer.png'), bbox_inches='tight', dpi=600)
plt.show()

# ------------------------------------------------------------------
# Quick sanity print — compare final-month MCF to mean-of-running-totals
# ------------------------------------------------------------------
naive_m24 = cum_scc[24].dropna().mean()
print(f"Nelson-Aalen MCF at month 24 : {mcf[-1]:.3f}  "
      f"(95% CI {mcf_lo[-1]:.3f}–{mcf_hi[-1]:.3f}, Y={int(mcf_ns[-1])})")
print(f"Old naive estimator at m24   : {naive_m24:.3f}   "
      f"(among {cum_scc[24].notna().sum()} patients still on drug)")
print(f"Difference                   : {mcf[-1] - naive_m24:+.3f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

months = list(range(1, 25))

# ============================================================
# TOP LEFT: Raw monthly means
# ============================================================
ax = axes[0, 0]
means, lows, highs, ns = [], [], [], []
for m in months:
    vals = scc_month_cens[f'm{m}'].dropna()
    n = len(vals)
    mu = vals.mean() if n > 0 else np.nan
    se = vals.std() / np.sqrt(n) if n > 1 else 0
    means.append(mu)
    lows.append(mu - 1.96 * se)
    highs.append(mu + 1.96 * se)
    ns.append(n)

ax.fill_between(months, lows, highs, alpha=0.2, color=C['mauve'])
ax.plot(months, means, 'o-', color=C['plum'], markersize=5, lw=2, label='Mean SCC/month')

for i, m in enumerate(months):
    if m % 3 == 0 or m == 1:
        ax.text(m, means[i] + 0.03, f'n={ns[i]}', ha='center', fontsize=6.5, color=C['slate'])

ax.set_xlabel('Months after acitretin start')
ax.set_ylabel('Mean SCC count per month')
ax.set_title('Monthly SCC rate')
ax.legend(fontsize=8, loc='upper right')
ax.set_xticks([1, 3, 6, 9, 12, 15, 18, 21, 24])

# ============================================================
# TOP RIGHT: Quarterly — exposure-weighted pooled rate
# ============================================================
ax = axes[0, 1]
q_labels = ['Q1\n(1–3)', 'Q2\n(4–6)', 'Q3\n(7–9)', 'Q4\n(10–12)',
            'Q5\n(13–15)', 'Q6\n(16–18)', 'Q7\n(19–21)', 'Q8\n(22–24)']
q_means, q_lows, q_highs, q_ns, q_pm = [], [], [], [], []

for q_idx, start_m in enumerate(range(1, 25, 3)):
    cols = [f'm{m}' for m in range(start_m, start_m + 3)]
    valid_pp   = scc_month_cens[cols].notna().sum(axis=1)   # valid months per patient
    total_pp   = scc_month_cens[cols].sum(axis=1, min_count=1)

    contrib    = valid_pp > 0
    total_scc  = total_pp[contrib].sum()
    total_mos  = valid_pp[contrib].sum()

    if total_mos > 0:
        # Pooled monthly rate, scaled to 3-month quarter
        rate_monthly = total_scc / total_mos
        mu           = rate_monthly * 3
        # Poisson-based SE for the pooled rate
        se_monthly   = np.sqrt(total_scc) / total_mos
        se           = se_monthly * 3
        lo, hi       = mu - 1.96 * se, mu + 1.96 * se
    else:
        mu = lo = hi = np.nan

    q_means.append(mu)
    q_lows.append(lo)
    q_highs.append(hi)
    q_ns.append(int(contrib.sum()))        # patients contributing any months
    q_pm.append(int(total_mos))            # total patient-months pooled

x = list(range(8))
ax.fill_between(x, q_lows, q_highs, alpha=0.2, color=C['mauve'])
ax.plot(x, q_means, 'o-', color=C['plum'], markersize=7, lw=2.5,
        label='Pooled SCC rate × 3')

for i in range(8):
    ax.text(i, q_means[i] + 0.06,
            f'n={q_ns[i]}\n({q_pm[i]} pt-mo)',
            ha='center', fontsize=6.5, color=C['slate'])

ax.set_xticks(x)
ax.set_xticklabels(q_labels, fontsize=8)
ax.set_xlabel('Quarter after acitretin start')
ax.set_ylabel('SCC count per quarter')
ax.set_title('Quarterly SCC rate (exposure-weighted)')
ax.legend(fontsize=8, loc='upper right')
ax.legend(fontsize=8, loc='upper right')

# ============================================================
# BOTTOM LEFT: Individual monthly trajectories
# ============================================================
ax = axes[1, 0]

valid_months_per_pt = scc_month_cens.notna().sum(axis=1)

sub = scc_month_cens.copy()
sub.columns = months

for idx in sub.index:
    row = sub.loc[idx].dropna()
    ax.plot(row.index, row.values, color=C['mauve'], alpha=0.25, lw=0.8)

mean_traj = sub.mean()
n_traj = sub.notna().sum()
mean_traj[n_traj < 5] = np.nan
ax.plot(months, mean_traj.values, 'o-', color=C['plum'], lw=2.5,
        markersize=5, zorder=5, label=f'Mean')

ax.set_xlabel('Months after acitretin start')
ax.set_ylabel('SCC count per month')
ax.set_title('Individual monthly trajectories')
ax.legend(fontsize=8)
ax.set_xticks([1, 3, 6, 9, 12, 15, 18, 21, 24])

# ============================================================
# BOTTOM RIGHT: Individual quarterly trajectories
# ============================================================
ax = axes[1, 1]

q_df = pd.DataFrame(index=scc_month_cens.index)
for q_idx, start_m in enumerate(range(1, 25, 3)):
    cols = [f'm{m}' for m in range(start_m, start_m + 3)]
    valid = scc_month_cens[cols].notna().sum(axis=1)
    total = scc_month_cens[cols].sum(axis=1, min_count=1)
    q_df[q_idx] = total / valid * 3
    q_df.loc[valid < 1, q_idx] = np.nan

valid_qs = q_df.notna().sum(axis=1)

sub_q = q_df.copy()

for idx in sub_q.index:
    row = sub_q.loc[idx].dropna()
    ax.plot(row.index, row.values, color=C['mauve'], alpha=0.25, lw=0.8)

pooled_q = []
for q_idx, start_m in enumerate(range(1, 25, 3)):
    cols      = [f'm{m}' for m in range(start_m, start_m + 3)]
    valid_pp  = scc_month_cens[cols].notna().sum(axis=1)
    total_pp  = scc_month_cens[cols].sum(axis=1, min_count=1)
    contrib   = valid_pp > 0
    total_scc = total_pp[contrib].sum()
    total_mos = valid_pp[contrib].sum()
    pooled_q.append((total_scc / total_mos * 3) if total_mos > 0 else np.nan)

ax.plot(x, pooled_q, 'o-', color=C['plum'], lw=2.5,
        markersize=7, zorder=5,
        label=f'Pooled mean (weighted by pt-mo)')

ax.set_xticks(x)
ax.set_xticklabels(q_labels, fontsize=8)
ax.set_xlabel('Quarter after acitretin start')
ax.set_ylabel('SCC count per quarter')
ax.set_title('Individual quarterly trajectories')
ax.legend(fontsize=8)

plt.savefig(os.path.join(FIGURE_DIR, 'fig_rate_plots_monthly_quarterly.png'), bbox_inches='tight')
plt.show()

In [ ]:
# %% Cell — Zero-SCC Patient Analysis
# ============================================================================
#  ZERO-SCC PATIENT ANALYSIS
#  Paste this cell into your notebook AFTER cells 1–6 (data prep + censoring).
#  Requires: df, scc_month_cens, pre_scc, dose_mg_cens, C
# ============================================================================


# ── 1.  Identify the patient-ID column ────────────────────────────────────
#     REDCap exports typically call it 'Record ID' or 'Study ID'
id_candidates = [c for c in df_raw.columns if any(k in c.lower() for k in ['record', 'study id', 'patient'])]
if id_candidates:
    id_col = id_candidates[0]
else:
    id_col = df_raw.columns[0]          # fallback: first column is usually Record ID
    print(f"⚠  Could not auto-detect a Record-ID column; falling back to '{id_col}'")

# Carry it through to the filtered df if it isn't there already
if 'patient_id' not in df.columns:
    # We re-derive it from the filtered rows (df was reset_index'd, but we
    # kept the same row content)
    # Safest route: re-run the same filter chain on df_raw to get aligned IDs
    _filtered = df_raw[
        (df_raw['Complete?'] == 'Complete') &
        (~df_raw['Site ID'].isin(['DKBeta', 'dkbeta']))
    ].copy().reset_index(drop=True)

    # Apply the ≥3-month mask exactly as the original script did
    _dose_mg_full = dose_mg.copy()          # pre-censored dose_mg before the 3-month filter
    # Since dose_mg was already filtered in the main script, we reconstruct
    # the patient_id from the *filtered* df_raw rows.  The trick: df still
    # has columns from df_raw, so id_col should still be present.
    if id_col in df.columns:
        df['patient_id'] = df[id_col].astype(str).str.strip()
    else:
        # If the column was dropped, number them
        df['patient_id'] = [f'PT-{i+1}' for i in range(len(df))]
        print(f"⚠  '{id_col}' not in filtered df — using sequential IDs")

print(f"Patient ID column: '{id_col}'")
print(f"Sample IDs: {df['patient_id'].head(3).tolist()}")

# ── 2.  Compute total on-drug SCCs per patient ───────────────────────────
df['total_on_scc'] = scc_month_cens.sum(axis=1, min_count=1).fillna(0).astype(int)
df['is_zero_scc']  = (df['total_on_scc'] == 0).astype(int)

n_total      = len(df)
n_zero       = df['is_zero_scc'].sum()
n_any_scc    = n_total - n_zero
pct_zero     = 100 * n_zero / n_total

print("\n" + "=" * 70)
print("  ZERO-SCC ON ACITRETIN — OVERVIEW")
print("=" * 70)
print(f"  Total patients (≥3 months on drug): {n_total}")
print(f"  Zero SCCs while on acitretin:       {n_zero}  ({pct_zero:.1f}%)")
print(f"  ≥1 SCC while on acitretin:          {n_any_scc} ({100 - pct_zero:.1f}%)")

# ── 3.  Stratify by immune status ─────────────────────────────────────────
print("\n" + "=" * 70)
print("  STRATIFICATION BY IMMUNE STATUS")
print("=" * 70)

for grp in ['Immunocompetent', 'Immunosuppressed']:
    mask_grp = df['immune'] == grp
    n_grp    = mask_grp.sum()
    n_zero_g = (df.loc[mask_grp, 'is_zero_scc'] == 1).sum()
    pct_g    = 100 * n_zero_g / n_grp if n_grp > 0 else 0

    dur_zero = df.loc[mask_grp & (df['is_zero_scc'] == 1), 'months_on_drug']
    dur_any  = df.loc[mask_grp & (df['is_zero_scc'] == 0), 'months_on_drug']

    print(f"\n  {grp} (n={n_grp}):")
    print(f"    Zero-SCC: {n_zero_g} ({pct_g:.1f}%)")
    if len(dur_zero) > 0:
        print(f"    Duration (zero-SCC) — median {dur_zero.median():.0f} mo, "
              f"mean {dur_zero.mean():.1f}, range {dur_zero.min():.0f}–{dur_zero.max():.0f}")
    if len(dur_any) > 0:
        print(f"    Duration (≥1 SCC)  — median {dur_any.median():.0f} mo, "
              f"mean {dur_any.mean():.1f}, range {dur_any.min():.0f}–{dur_any.max():.0f}")

# Sub-stratify immunosuppressed by type
print("\n  Immunocompromised sub-types (zero-SCC):")
for label, col in [('SOTR', 'is_sotr'), ('CLL', 'is_cll')]:
    mask_sub   = (df['immune'] == 'Immunosuppressed') & (df[col] == True)
    n_sub      = mask_sub.sum()
    n_zero_sub = (df.loc[mask_sub, 'is_zero_scc'] == 1).sum()
    pct_sub    = 100 * n_zero_sub / n_sub if n_sub > 0 else 0
    print(f"    {label}: {n_zero_sub}/{n_sub} zero-SCC ({pct_sub:.1f}%)")

# ── 4.  Duration breakdown for zero-SCC patients ─────────────────────────
print("\n" + "=" * 70)
print("  DURATION ON DRUG — ZERO-SCC PATIENTS")
print("=" * 70)

dur_zero_all = df.loc[df['is_zero_scc'] == 1, 'months_on_drug']
print(f"  Median: {dur_zero_all.median():.0f} months")
print(f"  Mean:   {dur_zero_all.mean():.1f} months")
print(f"  IQR:    {dur_zero_all.quantile(.25):.0f}–{dur_zero_all.quantile(.75):.0f}")
print(f"  Range:  {dur_zero_all.min():.0f}–{dur_zero_all.max():.0f}")

for cutoff in [6, 12, 18, 24]:
    n_c = (dur_zero_all >= cutoff).sum()
    print(f"    ≥{cutoff:2d} months & zero SCC: {n_c:3d} ({100*n_c/n_zero:.0f}% of zero-SCC group)")

# ── 5.  Build the patient-level export table ──────────────────────────────
# Pre-treatment SCC burden (Year -1)
df['pre_yr1_scc'] = pre_scc['yr1'].values

# Mean on-drug dose
on_drug_mean_dose = []
for idx in df.index:
    last_on = int(df.loc[idx, 'months_on_drug'])
    doses = [dose_mg_cens.loc[idx, f'm{m}'] for m in range(1, last_on + 1)]
    doses = [d for d in doses if pd.notna(d)]
    on_drug_mean_dose.append(np.mean(doses) if doses else np.nan)
df['mean_dose_mg'] = on_drug_mean_dose

zero_df = df.loc[df['is_zero_scc'] == 1].copy()

export_cols = ['patient_id', 'site', 'age', 'sex', 'immune',
               'months_on_drug', 'mean_dose_mg', 'pre_yr1_scc',
               'nicotinamide', 'field_therapy', 'is_sotr', 'is_cll']

# Keep only columns that exist
export_cols = [c for c in export_cols if c in zero_df.columns]
export_table = zero_df[export_cols].copy()
export_table = export_table.sort_values(['immune', 'months_on_drug'], ascending=[True, False])
export_table = export_table.reset_index(drop=True)

# Prettify column names for export
rename_map = {
    'patient_id':      'Patient ID',
    'site':            'Site',
    'age':             'Age',
    'sex':             'Sex',
    'immune':          'Immune Status',
    'months_on_drug':  'Months on Drug',
    'mean_dose_mg':    'Mean Dose (mg)',
    'pre_yr1_scc':     'Pre-Tx Year-1 SCCs',
    'nicotinamide':    'Nicotinamide',
    'field_therapy':   'Field Therapy',
    'is_sotr':         'SOTR',
    'is_cll':          'CLL',
}
export_table = export_table.rename(columns=rename_map)

# Replace 0/1 and True/False with Yes/No for readability
# (np.where is type-agnostic — works for int 0/1, Python bool, and numpy.bool_)
for col in ['Nicotinamide', 'Field Therapy', 'SOTR', 'CLL']:
    if col in export_table.columns:
        export_table[col] = np.where(export_table[col].astype(bool), 'Yes', 'No')

export_table['Mean Dose (mg)'] = export_table['Mean Dose (mg)'].round(1)

print("\n" + "=" * 70)
print(f"  ZERO-SCC PATIENT ROSTER  (n = {len(export_table)})")
print("=" * 70)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 140):
    print(export_table.to_string(index=False))

# Save to file
export_table.to_csv('./zero_scc_patients.csv', index=False)
print(f"\n  ✓ Saved to ./zero_scc_patients.csv")

# ── 6.  FIGURE — Zero-SCC deep dive (2 × 2 panel) ────────────────────────

fig = plt.figure(figsize=(14, 13))
gs = fig.add_gridspec(2, 2)

ax_bar      = fig.add_subplot(gs[0, 0])
ax_dur      = fig.add_subplot(gs[0, 1])
ax_strip_ic = fig.add_subplot(gs[1, 0])
ax_strip_is = fig.add_subplot(gs[1, 1])

# ── Panel A: Proportion zero-SCC by immune status ────────────────────────
ax = ax_bar

categories = ['All', 'Immuno-\ncompetent', 'Immuno-\ncompromised']
masks = [
    pd.Series(True, index=df.index),
    df['immune'] == 'Immunocompetent',
    df['immune'] == 'Immunosuppressed',
]
bar_colors_zero = [C['mauve'], C['mauve'], C['mauve']]
bar_colors_scc  = [C['plum'],  C['plum'],  C['plum']]

x_pos = np.arange(len(categories))
bar_w = 0.55

zeros, sccs, totals = [], [], []
for m in masks:
    n_grp    = m.sum()
    n_zero_g = (df.loc[m, 'is_zero_scc'] == 1).sum()
    zeros.append(n_zero_g)
    sccs.append(n_grp - n_zero_g)
    totals.append(n_grp)

bars_z = ax.bar(x_pos, zeros, bar_w, label='Zero SCC', color=C['mauve'],
                edgecolor='#2B2D42', lw=1.8, alpha=0.9)
bars_s = ax.bar(x_pos, sccs, bar_w, bottom=zeros, label='≥1 SCC', color=C['plum'],
                edgecolor='#2B2D42', lw=1.8, alpha=0.8)

for i in range(len(categories)):
    pct = 100 * zeros[i] / totals[i]
    ax.text(i, zeros[i] / 2, f'{zeros[i]}\n({pct:.0f}%)',
            ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.text(i, zeros[i] + sccs[i] / 2, f'{sccs[i]}',
            ha='center', va='center', fontsize=9, color='white', alpha=0.9)
    ax.text(i, totals[i] + 0.5, f'n={totals[i]}', ha='center', fontsize=8, color='#555')

ax.set_xticks(x_pos)
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylabel('Number of patients')
ax.set_title('A.  Zero-SCC patients by immune status', fontsize=11, fontweight='bold', loc='left')
ax.legend(fontsize=8, loc='upper right', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, max(totals) * 1.12)

# ── Panel B: Duration distribution — zero-SCC vs ≥1 SCC ──────────────────
ax = ax_dur

dur_zero = df.loc[df['is_zero_scc'] == 1, 'months_on_drug']
dur_scc  = df.loc[df['is_zero_scc'] == 0, 'months_on_drug']

bins_d = np.arange(2.5, 25.5, 1)

ax.hist([dur_scc, dur_zero], bins=bins_d,
        color=[C['plum'], C['mauve']],
        edgecolor='#2B2D42', lw=1.5,
        alpha=0.85, stacked=True,
        label=[f'≥1 SCC (n={len(dur_scc)})', f'Zero SCC (n={len(dur_zero)})'],
        zorder=3)

# Median lines
ax.axvline(dur_zero.median(), color=C['mauve'], ls='--', lw=1.5, alpha=0.8)
ax.axvline(dur_scc.median(),  color=C['plum'],  ls='--', lw=1.5, alpha=0.8)

y_top = ax.get_ylim()[1]
ax.text(dur_zero.median() + 0.3, y_top * 0.93,
        f'Zero-SCC\nmedian {dur_zero.median():.0f} mo',
        fontsize=7.5, color=C['mauve'], va='top', fontweight='bold')
ax.text(dur_scc.median() - 0.3, y_top * 0.75,
        f'≥1 SCC\nmedian {dur_scc.median():.0f} mo',
        fontsize=7.5, color=C['plum'], va='top', ha='right', fontweight='bold')

ax.set_xlabel('Months on acitretin', fontweight='bold')
ax.set_ylabel('Number of patients')
ax.set_title('B.  Treatment duration by SCC outcome', fontsize=11, fontweight='bold', loc='left')
ax.set_xticks([3, 6, 9, 12, 15, 18, 21, 24])
ax.legend(fontsize=8, loc='upper left')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(2.0)
ax.spines['bottom'].set_linewidth(2.0)
ax.spines['left'].set_color('#333333')
ax.spines['bottom'].set_color('#333333')

# ── Panel C & D: Strip charts — zero-SCC patients, side by side ───────────

# Separate by immune status
zero_ic = df.loc[(df['is_zero_scc'] == 1) & (df['immune'] == 'Immunocompetent')].copy()
zero_is = df.loc[(df['is_zero_scc'] == 1) & (df['immune'] == 'Immunosuppressed')].copy()

# Sort each group by duration
zero_ic = zero_ic.sort_values('months_on_drug', ascending=True).reset_index(drop=True)
zero_is = zero_is.sort_values('months_on_drug', ascending=True).reset_index(drop=True)

n_ic = len(zero_ic)
n_is = len(zero_is)

# Shared x-axis limit (room for labels)
x_max = 31

bar_h = 0.7

def _draw_strip(ax, sub_df, color, show_subtype=False):
    """Draw horizontal bars with patient ID / site labels for one group."""
    for i, (_, row) in enumerate(sub_df.iterrows()):
        ax.barh(i, row['months_on_drug'], height=bar_h,
                left=0, color=color, edgecolor='#2B2D42', lw=1.2, alpha=0.90)
        suffix = ''
        if show_subtype:
            if row.get('is_sotr', False): suffix += ' ◆SOTR'
            if row.get('is_cll', False):  suffix += ' ●CLL'
        label_txt = f"{row['patient_id']}  ({row['site']}){suffix}"
        ax.text(row['months_on_drug'] + 0.3, i, label_txt,
                va='center', fontsize=5.5, color='#555', family='monospace')

    n = len(sub_df)
    # Milestone lines
    for milestone in [6, 12, 18, 24]:
        ax.axvline(milestone, color='#BBBBBB', ls=':', lw=0.7, alpha=0.6, zorder=0)
        ax.text(milestone, n - 0.3, f'{milestone} mo',
                ha='center', va='bottom', fontsize=6.5, color='#999')

    ax.set_xlabel('Months on acitretin', fontsize=9)
    ax.set_yticks([])
    ax.set_xlim(-0.5, x_max)
    ax.set_ylim(-0.8, max(n - 0.2, 0.5))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_linewidth(2.0)

# ── Left: Immunocompetent ─────────────────────────────────────────────────
_draw_strip(ax_strip_ic, zero_ic, C['mauve'], show_subtype=False)
ax_strip_ic.set_title(f'C.  Immunocompetent  (n={n_ic})',
                       fontsize=11, fontweight='bold', loc='left')

# Duration summary annotation
if n_ic > 0:
    dur_ic = zero_ic['months_on_drug']
    txt_ic = (f'Median {dur_ic.median():.0f} mo\n'
              f'IQR {dur_ic.quantile(.25):.0f}–{dur_ic.quantile(.75):.0f}\n'
              f'≥12 mo: {(dur_ic >= 12).sum()}/{n_ic}')
    ax_strip_ic.text(0.97, 0.04, txt_ic, transform=ax_strip_ic.transAxes,
                     fontsize=7.5, ha='right', va='bottom', color=C['mauve'],
                     fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.4', facecolor='#F5F0F0',
                               edgecolor='#CCBBBB', alpha=0.85))

# ── Right: Immunosuppressed ──────────────────────────────────────────────
_draw_strip(ax_strip_is, zero_is, C['plum'], show_subtype=True)
ax_strip_is.set_title(f'D.  Immunocompromised  (n={n_is})',
                       fontsize=11, fontweight='bold', loc='left')

if n_is > 0:
    dur_is = zero_is['months_on_drug']
    txt_is = (f'Median {dur_is.median():.0f} mo\n'
              f'IQR {dur_is.quantile(.25):.0f}–{dur_is.quantile(.75):.0f}\n'
              f'≥12 mo: {(dur_is >= 12).sum()}/{n_is}')
    ax_strip_is.text(0.97, 0.04, txt_is, transform=ax_strip_is.transAxes,
                     fontsize=7.5, ha='right', va='bottom', color=C['plum'],
                     fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.4', facecolor='#F5F0F0',
                               edgecolor='#CCBBBB', alpha=0.85))

# Overall summary
txt = f'Zero-SCC patients: {n_zero}/{n_total} ({pct_zero:.0f}%)'

fig.suptitle(f'Patients with zero invasive SCCs on acitretin  —  {n_zero}/{n_total} ({pct_zero:.0f}%)',
             fontsize=14, fontweight='bold', y=1.01)

# Labelled by Study ID and site, i.e. patient-level: saved with the tables
# in OUTPUT_DIR, not in FIGURE_DIR.
plt.savefig('./fig_zero_scc_analysis.png', bbox_inches='tight', dpi=600)
plt.show()
print("  ✓ Figure saved to ./fig_zero_scc_analysis.png")


# ── 7.  Compact summary table for manuscript ─────────────────────────────
print("\n" + "=" * 70)
print("  TABLE — For manuscript")
print("=" * 70)

rows = []
for grp_label, grp_mask in [('All',               pd.Series(True, index=df.index)),
                              ('Immunocompetent',   df['immune'] == 'Immunocompetent'),
                              ('Immunosuppressed',  df['immune'] == 'Immunosuppressed'),
                              ('  — SOTR',          (df['immune'] == 'Immunosuppressed') & (df['is_sotr'])),
                              ('  — CLL',           (df['immune'] == 'Immunosuppressed') & (df['is_cll']))]:
    n_grp    = grp_mask.sum()
    zero_m   = grp_mask & (df['is_zero_scc'] == 1)
    n_zero_g = zero_m.sum()
    pct_g    = f'{100 * n_zero_g / n_grp:.1f}' if n_grp > 0 else '—'
    dur_g    = df.loc[zero_m, 'months_on_drug']

    rows.append({
        'Group':            grp_label,
        'n (total)':        n_grp,
        'n (zero SCC)':     n_zero_g,
        '% zero SCC':       pct_g,
        'Duration median':  f'{dur_g.median():.0f}' if len(dur_g) > 0 else '—',
        'Duration IQR':     f'{dur_g.quantile(.25):.0f}–{dur_g.quantile(.75):.0f}' if len(dur_g) > 0 else '—',
        'Duration range':   f'{dur_g.min():.0f}–{dur_g.max():.0f}' if len(dur_g) > 0 else '—',
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
summary_df.to_csv('./zero_scc_summary_table.csv', index=False)
print("\n  ✓ Saved to ./zero_scc_summary_table.csv")


In [ ]:
# ============================================================================
# CELL 1 — Pre-computations for paired offset analysis
# ============================================================================
# Paste after cells 1–6 (data prep + censoring).
# Requires: df, scc_month_cens, pre_scc, C, np, pd

import statsmodels.api as sm
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Poisson as PoissonFamily
from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.discrete.discrete_model import NegativeBinomial as NB2
from statsmodels.discrete.discrete_model import Poisson as PoissonDisc
from scipy.stats import wilcoxon, chi2

# ── On-treatment aggregates ──────────────────────────────────────────────
df['on_scc']     = scc_month_cens.sum(axis=1, min_count=1).fillna(0).astype(int)
df['on_time_mo'] = df['months_on_drug'].astype(float)

# ── Pre-treatment windows (require complete years) ───────────────────────
# By default compare on-drug only against Pre-1 and Pre-1–2.
# Flip to True to also include the longer 1–5 and 1–10 windows.
IRR_INCLUDE_LONG_WINDOWS = False

pre_windows = [
    ('Pre 1',    [1]),
    ('Pre 1–2',  [1, 2]),
]
if IRR_INCLUDE_LONG_WINDOWS:
    pre_windows += [
        ('Pre 1–5',  list(range(1, 6))),
        ('Pre 1–10', list(range(1, 11))),
    ]

pre_comp = {}

print("=" * 70)
print("  PRE-COMPUTATION: PAIRED ANALYSIS DATA")
print("=" * 70)

for label, years in pre_windows:
    cols = [f'yr{y}' for y in years]
    mask = pre_scc[cols].notna().all(axis=1)           # require all years present
    scc  = pre_scc.loc[mask, cols].sum(axis=1).astype(int)
    time_mo = float(len(years) * 12)                   # same for every patient in window

    pre_comp[label] = {'scc': scc, 'time_mo': time_mo, 'mask': mask}

    ann_rate = scc.sum() / (mask.sum() * time_mo) * 12
    print(f"  {label:12s}  n={mask.sum():3d}  "
          f"total_SCC={scc.sum():5d}  "
          f"pt-yr={mask.sum() * time_mo / 12:7.0f}  "
          f"ann. rate={ann_rate:.3f}/yr")

ann_on = df['on_scc'].sum() / df['on_time_mo'].sum() * 12
print(f"  {'On-drug':12s}  n={len(df):3d}  "
      f"total_SCC={df['on_scc'].sum():5d}  "
      f"pt-yr={df['on_time_mo'].sum() / 12:7.0f}  "
      f"ann. rate={ann_on:.3f}/yr")
print()

In [ ]:
# ============================================================================
# CELL 2 (UPDATED) — All-patients paired analysis, NB GEE primary
# ============================================================================

from statsmodels.genmod.families import NegativeBinomial as NBFamily

results = []

print("=" * 70)
print("  PAIRED ANALYSIS: ON-ACITRETIN vs PRE-TREATMENT WINDOWS")
print("  Primary model : NB GEE, exchangeable correlation,")
print("                  log(person-months) offset, robust sandwich SEs")
print("  Sensitivity 1 : Poisson GEE (same spec)")
print("  Sensitivity 2 : Wilcoxon signed-rank on annualized rates")
print("=" * 70)

for label, info in pre_comp.items():
    mask = info['mask']
    n_pairs = mask.sum()

    print(f"\n{'─' * 70}")
    print(f"  {label}  vs  On-acitretin   (n = {n_pairs} paired patients)")
    print(f"{'─' * 70}")

    # ── Stack pre + on rows ───────────────────────────────────────────────
    pre_df = pd.DataFrame({
        'scc':      info['scc'].values,
        'on_drug':  0,
        'log_time': np.log(info['time_mo']),
        'patient':  np.arange(n_pairs),
    })
    on_df = pd.DataFrame({
        'scc':      df.loc[mask, 'on_scc'].values,
        'on_drug':  1,
        'log_time': np.log(df.loc[mask, 'on_time_mo'].values),
        'patient':  np.arange(n_pairs),
    })
    stacked = pd.concat([pre_df, on_df], ignore_index=True)
    stacked = stacked.sort_values('patient').reset_index(drop=True)

    X = sm.add_constant(stacked['on_drug'])

    # ══════════════════════════════════════════════════════════════════════
    # A.  NB GEE — PRIMARY (iterative alpha estimation)
    # ══════════════════════════════════════════════════════════════════════
    irr_nb_gee = ci_lo_nb_gee = ci_hi_nb_gee = p_nb_gee = scale_nb_gee = np.nan
    try:
        # Step 1: estimate alpha from NB2 GLM (MLE)
        _nb_glm = NB2(stacked['scc'].values, X.values,
                       offset=stacked['log_time'].values)
        _nb_glm_res = _nb_glm.fit(disp=0, maxiter=300)
        try:
            alpha_hat = np.exp(_nb_glm_res.lnalpha)
        except AttributeError:
            alpha_hat = 1.0  # fallback if estimation fails

        # Step 2: plug estimated alpha into GEE
        nb_gee_mod = GEE(
            stacked['scc'], X,
            groups=stacked['patient'],
            family=NBFamily(alpha=alpha_hat),
            cov_struct=Exchangeable(),
            offset=stacked['log_time'],
        )
        nb_gee_res = nb_gee_mod.fit(maxiter=200)

        print(f"    NB2 GLM α̂ = {alpha_hat:.4f} (plugged into GEE)")
        nb_gee_res = nb_gee_mod.fit(maxiter=200)

        irr_nb_gee  = np.exp(nb_gee_res.params.iloc[1])
        ci_arr      = nb_gee_res.conf_int().iloc[1]
        ci_lo_nb_gee = np.exp(ci_arr.iloc[0])
        ci_hi_nb_gee = np.exp(ci_arr.iloc[1])
        p_nb_gee    = nb_gee_res.pvalues.iloc[1]
        scale_nb_gee = nb_gee_res.scale

        # Overflow guard: baseline rate ≈ 0 → IRR explodes
        if irr_nb_gee > 1e6 or np.isinf(irr_nb_gee):
            irr_nb_gee = ci_lo_nb_gee = ci_hi_nb_gee = p_nb_gee = np.nan

        print(f"\n  NB GEE (primary):")
        print(f"    IRR  = {irr_nb_gee:.3f}   95% CI [{ci_lo_nb_gee:.3f}, {ci_hi_nb_gee:.3f}]   "
              f"p = {p_nb_gee:.2e}")
        print(f"    Scale = {scale_nb_gee:.2f}")
    except Exception as e:
        print(f"\n  NB GEE FAILED: {e}")

    # ══════════════════════════════════════════════════════════════════════
    # B.  Poisson GEE — Sensitivity 1
    # ══════════════════════════════════════════════════════════════════════
    irr_pois_gee = ci_lo_pois = ci_hi_pois = p_pois_gee = np.nan
    try:
        pois_gee_mod = GEE(
            stacked['scc'], X,
            groups=stacked['patient'],
            family=PoissonFamily(),
            cov_struct=Exchangeable(),
            offset=stacked['log_time'],
        )
        pois_gee_res = pois_gee_mod.fit()

        irr_pois_gee = np.exp(pois_gee_res.params.iloc[1])
        ci_arr_p     = pois_gee_res.conf_int().iloc[1]
        ci_lo_pois   = np.exp(ci_arr_p.iloc[0])
        ci_hi_pois   = np.exp(ci_arr_p.iloc[1])
        p_pois_gee   = pois_gee_res.pvalues.iloc[1]

        if irr_pois_gee > 1e6 or np.isinf(irr_pois_gee):
            irr_pois_gee = ci_lo_pois = ci_hi_pois = p_pois_gee = np.nan

        print(f"\n  Poisson GEE (sensitivity 1):")
        print(f"    IRR  = {irr_pois_gee:.3f}   95% CI [{ci_lo_pois:.3f}, {ci_hi_pois:.3f}]   "
              f"p = {p_pois_gee:.2e}")
    except Exception as e:
        print(f"\n  Poisson GEE FAILED: {e}")

    # ══════════════════════════════════════════════════════════════════════
    # C.  NB2 vs Poisson GLM — Dispersion diagnostic (LRT)
    # ══════════════════════════════════════════════════════════════════════
    alpha_nb = ll_nb = ll_pois = p_lrt = np.nan
    try:
        mod_nb = NB2(stacked['scc'].values, X.values,
                     offset=stacked['log_time'].values)
        res_nb = mod_nb.fit(disp=0, maxiter=300)
        ll_nb = res_nb.llf
        try:
            alpha_nb = np.exp(res_nb.lnalpha)
        except AttributeError:
            pass

        mod_pois = PoissonDisc(stacked['scc'].values, X.values,
                               offset=stacked['log_time'].values)
        res_pois = mod_pois.fit(disp=0, maxiter=300)
        ll_pois = res_pois.llf

        lrt_stat = -2 * (ll_pois - ll_nb)
        p_lrt    = chi2.sf(lrt_stat, 1) / 2

        print(f"\n  Dispersion diagnostic:")
        print(f"    NB2 α = {alpha_nb:.4f}   (0 ≈ Poisson)")
        print(f"    LRT χ² = {lrt_stat:.1f}   p(one-sided) = {p_lrt:.2e}   "
              f"→ {'NB preferred' if p_lrt < 0.05 else 'Poisson adequate'}")
    except Exception as e:
        print(f"\n  Dispersion test FAILED: {e}")

    # ══════════════════════════════════════════════════════════════════════
    # D.  Wilcoxon signed-rank — Sensitivity 2
    # ══════════════════════════════════════════════════════════════════════
    pre_rate_ann = info['scc'].values / info['time_mo'] * 12
    on_rate_ann  = df.loc[mask, 'on_scc'].values / df.loc[mask, 'on_time_mo'].values * 12
    diff = on_rate_ann - pre_rate_ann

    n_decreased = (diff < 0).sum()
    n_same      = (diff == 0).sum()
    n_increased = (diff > 0).sum()

    nonzero = diff != 0
    if nonzero.sum() > 0:
        stat_w, p_w = wilcoxon(diff[nonzero])
    else:
        stat_w, p_w = np.nan, np.nan

    median_pre = np.median(pre_rate_ann)
    median_on  = np.median(on_rate_ann)
    print(f"\n  Wilcoxon signed-rank (sensitivity 2):")
    print(f"    Median pre = {median_pre:.2f}/yr  →  on = {median_on:.2f}/yr")
    print(f"    Decreased: {n_decreased}   Same: {n_same}   Increased: {n_increased}")
    print(f"    W = {stat_w:.0f}   p = {p_w:.2e}")

    # ── Store results (primary = NB GEE) ─────────────────────────────────
    results.append({
        'label':       label,
        'n':           n_pairs,
        'irr':         irr_nb_gee,
        'ci_lo':       ci_lo_nb_gee,
        'ci_hi':       ci_hi_nb_gee,
        'p_gee':       p_nb_gee,
        'scale':       scale_nb_gee,
        'irr_pois':    irr_pois_gee,
        'ci_lo_pois':  ci_lo_pois,
        'ci_hi_pois':  ci_hi_pois,
        'p_pois':      p_pois_gee,
        'alpha_nb':    alpha_nb,
        'p_lrt':       p_lrt,
        'p_wilcoxon':  p_w,
        'median_pre':  median_pre,
        'median_on':   median_on,
        'n_decreased': n_decreased,
        'n_increased': n_increased,
    })

results_df = pd.DataFrame(results)

# ── Diagnostic: check how alpha varies across windows ────────────────
print("\n" + "=" * 70)
print("  DIAGNOSTIC: Estimated NB alpha per pre-treatment window")
print("  (α ≈ 0 means Poisson is adequate; α >> 1 means heavy overdispersion)")
print("=" * 70)
for label, info in pre_comp.items():
    mask = info['mask']
    pre_df_tmp = pd.DataFrame({
        'scc':      info['scc'].values,
        'on_drug':  0,
        'log_time': np.log(info['time_mo']),
        'patient':  np.arange(mask.sum()),
    })
    on_df_tmp = pd.DataFrame({
        'scc':      df.loc[mask, 'on_scc'].values,
        'on_drug':  1,
        'log_time': np.log(df.loc[mask, 'on_time_mo'].values),
        'patient':  np.arange(mask.sum()),
    })
    stk = pd.concat([pre_df_tmp, on_df_tmp], ignore_index=True).sort_values('patient')
    Xtmp = sm.add_constant(stk['on_drug'])
    try:
        _m = NB2(stk['scc'].values, Xtmp.values, offset=stk['log_time'].values)
        _r = _m.fit(disp=0, maxiter=300)
        a = np.exp(_r.lnalpha)
        print(f"  {label:12s}  α̂ = {a:.4f}")
    except Exception as e:
        print(f"  {label:12s}  estimation failed: {e}")


print("\n\n" + "=" * 70)
print("  RESULTS SUMMARY (PRIMARY = NB GEE)")
print("=" * 70)
with pd.option_context('display.max_columns', None, 'display.width', 160):
    print(results_df.to_string(index=False, float_format='%.4f'))

In [ ]:
# ============================================================================
# CELL 3a — Clean forest plot, NB GEE only (for manuscript main figure)
# ============================================================================

fig, ax = plt.subplots(figsize=(10, 3.8))

n_comp = len(results_df)
y_pos  = np.arange(n_comp)[::-1]

for i, row in results_df.iterrows():
    y = y_pos[i]
    if pd.notna(row['irr']):
        ax.plot([row['ci_lo'], row['ci_hi']], [y, y],
                color=C['plum'], lw=3, solid_capstyle='round', zorder=3)
        ax.plot(row['irr'], y, 'D', color=C['plum'], markersize=11,
                markeredgecolor='white', markeredgewidth=1.5, zorder=5)

ax.axvline(1.0, color=C['navy'], ls='--', lw=1.2, alpha=0.5, zorder=0)

ax.set_yticks(y_pos)
ax.set_yticklabels(
    [f"{r['label']}  (n={int(r['n'])})" for _, r in results_df.iterrows()],
    fontsize=10, fontweight='bold')

x_text = results_df['ci_hi'].max() * 1.08
for i, row in results_df.iterrows():
    y = y_pos[i]

    p = row['p_gee']
    if pd.isna(p):
        continue
    if   p < 0.001: p_str = "p < 0.001"
    elif p < 0.05:  p_str = f"p = {p:.3f}"
    else:           p_str = f"p = {p:.2f}"

    txt = f"IRR {row['irr']:.2f}  [{row['ci_lo']:.2f}–{row['ci_hi']:.2f}]   {p_str}"
    ax.text(x_text, y, txt, va='center', fontsize=9, color='#333',
            family='monospace')

ymin = -0.9
ax.text(0.55, ymin, '← Favours acitretin',
        ha='center', fontsize=8.5, color=C['plum'], style='italic')
ax.text(1.55, ymin, 'Favours pre-treatment →',
        ha='center', fontsize=8.5, color=C['coral'], style='italic')

ax.set_xlabel('Incidence Rate Ratio  (on-acitretin / pre-treatment)', fontsize=10)
ax.set_title('Within-patient SCC rate comparison\n'
             'Negative binomial GEE · exchangeable correlation · '
             'log(person-months) offset · robust SEs',
             fontsize=11, fontweight='bold', pad=12)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_linewidth(2.0)

xmin = min(results_df['ci_lo'].dropna().min() * 0.8, 0.25)
xmax = x_text + 0.55
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin - 0.3, y_pos[0] + 0.8)

plt.savefig(os.path.join(FIGURE_DIR, 'fig_forest_paired_clean.png'), bbox_inches='tight', dpi=600)
plt.show()
print("  ✓ Saved to figures/fig_forest_paired_clean.png")


In [ ]:
# ============================================================================
# CELL 3b — Forest plot with sensitivity analyses (NB + Poisson + Wilcoxon)
# ============================================================================
# Two-zone layout: CI bars on the left, text annotations on the right

from matplotlib.lines import Line2D

def _fmt_p(p):
    if pd.isna(p): return ''
    if   p < 0.001: return "p < 0.001"
    elif p < 0.01:  return f"p = {p:.3f}"
    elif p < 0.05:  return f"p = {p:.3f}"
    else:           return f"p = {p:.2f}"

fig, (ax_ci, ax_txt) = plt.subplots(
    1, 2, figsize=(15, 8),
    gridspec_kw={'width_ratios': [1, 1.3], 'wspace': 0.02},
    sharey=True,
)

n_comp = len(results_df)
group_gap = 1.3
row_h = 0.9
y = 0

# ── Build position list ──────────────────────────────────────────────────
all_rows = []  # (y, window, model, irr, ci_lo, ci_hi, p, color, marker, ms, txt_right)

for i, row in results_df.iterrows():
    if i > 0:
        y += group_gap

    # NB GEE
    p = row['p_gee']
    p_str = _fmt_p(p) if pd.notna(p) else ''
    irr_txt = (f"IRR {row['irr']:.2f}  [{row['ci_lo']:.2f}–{row['ci_hi']:.2f}]   {p_str}"
               if pd.notna(row['irr']) else 'IRR undefined')
    all_rows.append((y, row['label'], 'NB GEE (primary)',
                     row['irr'], row['ci_lo'], row['ci_hi'], p,
                     C['plum'], 'D', 11, irr_txt))
    y += row_h

    # Poisson GEE
    if pd.notna(row.get('irr_pois')):
        p2 = row['p_pois']
        p2_str = _fmt_p(p2) if pd.notna(p2) else ''
        pois_txt = (f"IRR {row['irr_pois']:.2f}  [{row['ci_lo_pois']:.2f}–"
                    f"{row['ci_hi_pois']:.2f}]   {p2_str}")
        all_rows.append((y, row['label'], 'Poisson GEE',
                         row['irr_pois'], row['ci_lo_pois'], row['ci_hi_pois'], p2,
                         C['mauve'], 'o', 8, pois_txt))
    y += row_h

    # Wilcoxon
    pw = row['p_wilcoxon']
    pw_str = _fmt_p(pw) if pd.notna(pw) else ''
    w_txt = (f"Median {row['median_pre']:.1f} → {row['median_on']:.1f}/yr    "
             f"↓{int(row['n_decreased'])}  ↑{int(row['n_increased'])}    {pw_str}")
    all_rows.append((y, row['label'], 'Wilcoxon signed-rank',
                     np.nan, np.nan, np.nan, pw,
                     C['slate'], '|', 8, w_txt))
    y += row_h

max_y = y + 0.5

# ── CI panel (left) ─────────────────────────────────────────────────────
for (ypos, window, model, irr, ci_lo, ci_hi, p, color, marker, ms, _) in all_rows:
    yp = max_y - ypos

    if marker == '|':
        ax_ci.plot(1.0, yp, '|', color=color, markersize=7,
                   markeredgewidth=1.8, zorder=3, alpha=0.6)
    elif pd.notna(irr):
        ax_ci.plot([ci_lo, ci_hi], [yp, yp],
                   color=color, lw=2.8, solid_capstyle='round', zorder=3, alpha=0.85)
        ax_ci.plot(irr, yp, marker, color=color, markersize=ms,
                   markeredgecolor='white', markeredgewidth=1.3, zorder=5)

ax_ci.axvline(1.0, color=C['navy'], ls='--', lw=1.3, alpha=0.35, zorder=0)

# Group labels on left
y_reset = 0
for i, row in results_df.iterrows():
    if i > 0:
        y_reset += group_gap
    y_center = y_reset + row_h
    yp = max_y - y_center
    ax_ci.text(-0.05, yp, f"{row['label']}\n(n={int(row['n'])})",
               va='center', ha='right', fontsize=11, fontweight='bold',
               color=C['navy'], transform=ax_ci.get_yaxis_transform())
    y_reset += 3 * row_h

# Dividers
y_reset = 0
for i in range(n_comp):
    if i > 0:
        y_reset += group_gap
        yp = max_y - y_reset + group_gap / 2
        ax_ci.axhline(yp, color='#DDDDDD', lw=0.7, ls='-', zorder=0)
        ax_txt.axhline(yp, color='#DDDDDD', lw=0.7, ls='-', zorder=0)
    y_reset += 3 * row_h

# Direction arrows
y_bottom = max_y - y - 0.5
ax_ci.set_xlabel('Incidence Rate Ratio', fontsize=10)
ax_ci.set_yticks([])
ax_ci.spines['top'].set_visible(False)
ax_ci.spines['right'].set_visible(False)
ax_ci.spines['left'].set_visible(False)

valid = results_df.dropna(subset=['ci_lo', 'ci_hi'])
x_lo = min(valid['ci_lo'].min(), valid['ci_lo_pois'].dropna().min()) * 0.7
x_hi = max(valid['ci_hi'].max(), valid['ci_hi_pois'].dropna().max()) * 1.15
ax_ci.set_xlim(max(x_lo, 0.2), min(max(x_hi, 1.5), 4.0))
ax_ci.set_ylim(y_bottom - 1.2, max_y + 0.3)

# Direction labels, centred on each side of the null line
_xl, _xr = ax_ci.get_xlim()
ax_ci.text((_xl + 1.0) / 2, y_bottom - 0.7, '← Favours acitretin',
           ha='center', fontsize=8.5, color=C['plum'], style='italic')
ax_ci.text((1.0 + _xr) / 2, y_bottom - 0.7, 'Favours pre-treatment →',
           ha='center', fontsize=8.5, color=C['coral'], style='italic')

# Legend inside CI panel
legend_els = [
    Line2D([0], [0], marker='D', color='w', markerfacecolor=C['plum'],
           markersize=10, markeredgecolor='white', markeredgewidth=1,
           label='NB GEE (primary)', lw=0),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=C['mauve'],
           markersize=7, markeredgecolor='white', markeredgewidth=1,
           label='Poisson GEE (sensitivity)', lw=0),
    Line2D([0], [0], marker='|', color=C['slate'],
           markersize=7, markeredgewidth=1.8,
           label='Wilcoxon signed-rank', lw=0),
]
_ci_legend = legend_els   # drawn on the text panel below, where there is room

# ── Text panel (right) ──────────────────────────────────────────────────
ax_txt.set_xlim(0, 1)
ax_txt.axis('off')

# Column headers
header_y = max_y + 0.1
ax_txt.text(0.02, header_y, 'Model', fontsize=8.5, fontweight='bold',
            color='#555', va='bottom')
ax_txt.text(0.42, header_y, 'Estimate', fontsize=8.5, fontweight='bold',
            color='#555', va='bottom')

for (ypos, window, model, irr, ci_lo, ci_hi, p, color, marker, ms, txt_right) in all_rows:
    yp = max_y - ypos - 0.1

    # Model label
    ax_txt.text(0.02, yp, model, va='center', fontsize=8.5,
                color=color, fontweight='bold')

    # Estimate text
    ax_txt.text(0.42, yp, txt_right, va='center', fontsize=8.5,
                color='#333', family='monospace')

ax_txt.legend(handles=_ci_legend, fontsize=8, loc='lower left',
              framealpha=0.9, edgecolor='#666666')

fig.suptitle('Within-patient SCC rate comparison — all patients\n'
             'Primary: NB GEE · Sensitivity: Poisson GEE & Wilcoxon signed-rank',
             fontsize=12, fontweight='bold', y=1.05)

plt.savefig(os.path.join(FIGURE_DIR, 'fig_forest_paired_sens.png'), bbox_inches='tight', dpi=600)
plt.show()
print("  ✓ Saved to figures/fig_forest_paired_sens.png")


In [ ]:
# %% NEW CELL 3c — Covariate-adjusted sensitivity analysis
# ===========================================================================
# Added 2026-08-15 to answer "was the primary GEE adjusted for
# immunosuppression type, age and adjunctive field therapy?"
#
# It was NOT. The primary model in this notebook is the paired, self-
# controlled specification: two rows per patient (pre-treatment window and
# on-drug window), a single on_drug indicator, log(person-months) offset,
# patient-level cluster, exchangeable working correlation, robust SEs. No
# covariates enter it.
#
# The reason is structural. Every covariate named — age at initiation,
# immunosuppression type, field therapy, nicotinamide — takes one value per
# patient, so it is identical on that patient's pre row and on row. Such a
# term is orthogonal to the within-patient pre-vs-on contrast and cannot
# confound it; the design already conditions on everything fixed about the
# patient, measured or not. Covariates can only change the IRR through the
# working-correlation weighting, or if entered as covariate × period
# interactions (effect modification), which is what the stratified forest
# plots examine descriptively.
#
# This cell re-fits the primary model with those covariates as main effects
# so the manuscript can state the adjusted estimate and show it is unchanged.
# ===========================================================================

# ── Build the covariate block (one row per patient, cohort order) ──────────
_imm_type = np.where(df['immune'] != 'Immunosuppressed', 'Immunocompetent',
             np.where(df['is_sotr'], 'SOTR',
              np.where(df['is_cll'], 'CLL', 'Other immunosuppression')))
df['imm_type'] = _imm_type

_cov = pd.DataFrame({
    'age_per10':          (df['age'] - df['age'].mean()) / 10.0,
    'is_sotr':            (df['imm_type'] == 'SOTR').astype(float),
    'is_cll':             (df['imm_type'] == 'CLL').astype(float),
    'is_other_immuno':    (df['imm_type'] == 'Other immunosuppression').astype(float),
    'field_therapy':      df['field_therapy'].astype(float),
    'nicotinamide':       df['nicotinamide'].astype(float),
}, index=df.index)
COV_NAMES = list(_cov.columns)
print("  Covariates (reference level = immunocompetent, no field therapy,")
print("  no nicotinamide, mean age):")
for c in COV_NAMES:
    print(f"    · {c}")
print(f"  Age: mean {df['age'].mean():.1f} y, modelled per 10-year increment.")
print(f"  Immunosuppression type: {df['imm_type'].value_counts().to_dict()}")

# Patients with a missing covariate cannot enter the adjusted fit
_cov_complete = _cov.notna().all(axis=1)
print(f"  Complete covariates: {int(_cov_complete.sum())}/{len(df)}")


def _fit_nb_gee(y, X, groups, offset):
    """NB GEE with alpha from an NB2 GLM, matching the primary specification."""
    _glm = NB2(y.values, X.values, offset=offset.values).fit(disp=0, maxiter=300)
    try:
        alpha_hat = float(np.exp(_glm.lnalpha))
    except AttributeError:
        alpha_hat = 1.0
    res = GEE(y, X, groups=groups, family=NBFamily(alpha=alpha_hat),
              cov_struct=Exchangeable(), offset=offset).fit(maxiter=200)
    return res, alpha_hat


adj_rows = []
print("\n" + "=" * 74)
print("  UNADJUSTED vs COVARIATE-ADJUSTED — paired NB GEE")
print("=" * 74)

for label, info in pre_comp.items():
    mask = info['mask'] & _cov_complete
    n_pairs = int(mask.sum())

    stacked = pd.concat([
        pd.DataFrame({'scc': info['scc'].loc[mask[mask].index].values, 'on_drug': 0.0,
                      'log_time': np.log(info['time_mo']),
                      'patient': np.arange(n_pairs)}),
        pd.DataFrame({'scc': df.loc[mask, 'on_scc'].values, 'on_drug': 1.0,
                      'log_time': np.log(df.loc[mask, 'on_time_mo'].values),
                      'patient': np.arange(n_pairs)}),
    ], ignore_index=True)
    cov_stacked = pd.concat([_cov.loc[mask], _cov.loc[mask]], ignore_index=True)
    stacked = pd.concat([stacked, cov_stacked], axis=1)
    stacked = stacked.sort_values('patient').reset_index(drop=True)

    rec = {'window': label, 'n': n_pairs}
    for tag, cols in [('unadj', ['on_drug']), ('adj', ['on_drug'] + COV_NAMES)]:
        X = sm.add_constant(stacked[cols])
        try:
            res, alpha_hat = _fit_nb_gee(stacked['scc'], X,
                                         stacked['patient'], stacked['log_time'])
            ci = res.conf_int().loc['on_drug']
            rec[f'irr_{tag}']   = float(np.exp(res.params['on_drug']))
            rec[f'lo_{tag}']    = float(np.exp(ci.iloc[0]))
            rec[f'hi_{tag}']    = float(np.exp(ci.iloc[1]))
            rec[f'p_{tag}']     = float(res.pvalues['on_drug'])
            rec[f'alpha_{tag}'] = alpha_hat
            if tag == 'adj':
                rec['_res'] = res
        except Exception as e:
            print(f"    {label} / {tag}: FAILED — {e}")
            for k in ('irr', 'lo', 'hi', 'p', 'alpha'):
                rec[f'{k}_{tag}'] = np.nan

    adj_rows.append(rec)
    print(f"\n  {label}   (n = {n_pairs} paired patients)")
    print(f"    Unadjusted  IRR {rec['irr_unadj']:.3f}  "
          f"[{rec['lo_unadj']:.3f}–{rec['hi_unadj']:.3f}]   p = {rec['p_unadj']:.3g}")
    print(f"    Adjusted    IRR {rec['irr_adj']:.3f}  "
          f"[{rec['lo_adj']:.3f}–{rec['hi_adj']:.3f}]   p = {rec['p_adj']:.3g}")
    if '_res' in rec:
        print("    Covariate effects in the adjusted model (IRR per unit):")
        for c in COV_NAMES:
            b = rec['_res'].params[c]
            ci_c = rec['_res'].conf_int().loc[c]
            print(f"      {c:18s} {np.exp(b):6.3f}  "
                  f"[{np.exp(ci_c.iloc[0]):.3f}–{np.exp(ci_c.iloc[1]):.3f}]  "
                  f"p = {rec['_res'].pvalues[c]:.3g}")

adjusted_df = pd.DataFrame([{k: v for k, v in r.items() if k != '_res'}
                            for r in adj_rows])
adjusted_df.to_csv('./table_adjusted_vs_unadjusted_gee.csv', index=False)
print("\n  ✓ Saved to ./table_adjusted_vs_unadjusted_gee.csv")

# ── Forest plot: unadjusted vs adjusted, side by side ─────────────────────
fig, ax = plt.subplots(figsize=(11, 1.15 * len(adjusted_df) + 2.2))

row_h, group_gap = 0.85, 0.75
y, ypos = 0.0, []
for _, r in adjusted_df.iterrows():
    ypos.append((y, r, 'adj'));   y += row_h
    ypos.append((y, r, 'unadj')); y += row_h + group_gap
max_y = y

style = {'unadj': (C['plum'],  'D', 11, 'Unadjusted (primary)'),
         'adj':   (C['mauve'], 'o', 9,  'Adjusted (sensitivity)')}

for (yp, r, tag) in ypos:
    col, mk, ms, _ = style[tag]
    yy = max_y - yp
    if pd.notna(r[f'irr_{tag}']):
        ax.plot([r[f'lo_{tag}'], r[f'hi_{tag}']], [yy, yy], color=col, lw=3,
                solid_capstyle='round', zorder=3, alpha=0.9)
        ax.plot(r[f'irr_{tag}'], yy, mk, color=col, markersize=ms,
                markeredgecolor='white', markeredgewidth=1.4, zorder=5)
        p = r[f'p_{tag}']
        p_str = 'p < 0.001' if p < 0.001 else (f'p = {p:.3f}' if p < 0.05 else f'p = {p:.2f}')
        ax.text(1.02, yy, f"IRR {r[f'irr_{tag}']:.2f}  "
                          f"[{r[f'lo_{tag}']:.2f}–{r[f'hi_{tag}']:.2f}]   {p_str}",
                transform=ax.get_yaxis_transform(), va='center',
                fontsize=8.5, family='monospace', color='#333')

for i, (_, r) in enumerate(adjusted_df.iterrows()):
    y_center = max_y - (i * (2 * row_h + group_gap) + row_h / 2)
    ax.text(-0.02, y_center, f"{r['window']}\n(n={int(r['n'])})",
            transform=ax.get_yaxis_transform(), va='center', ha='right',
            fontsize=10, fontweight='bold', color=C['navy'])

ax.axvline(1.0, color=C['navy'], ls='--', lw=1.3, alpha=0.45, zorder=0)
ax.set_yticks([])
ax.set_xlabel('Incidence Rate Ratio  (on-acitretin / pre-treatment)', fontsize=10)
ax.set_title('Covariate adjustment does not move the within-patient estimate\n'
             'NB GEE · exchangeable · log(person-months) offset · robust SEs\n'
             'Adjusted for age, immunosuppression type, field therapy, nicotinamide',
             fontsize=11, fontweight='bold', pad=12)
for sp in ('top', 'right', 'left'):
    ax.spines[sp].set_visible(False)
ax.spines['bottom'].set_linewidth(2.0)
_lo = min(adjusted_df[['lo_unadj', 'lo_adj']].min()) * 0.85
_hi = max(adjusted_df[['hi_unadj', 'hi_adj']].max()) * 1.08
ax.set_xlim(_lo, _hi)
_yy = [max_y - yp for (yp, _, _) in ypos]
ax.set_ylim(min(_yy) - 1.1, max(_yy) + 0.8)

from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0], [0], marker=style[t][1], color='w',
                          markerfacecolor=style[t][0], markersize=9,
                          markeredgecolor='white', label=style[t][3], lw=0)
                   for t in ('unadj', 'adj')],
          fontsize=9, loc='lower right', framealpha=0.95, edgecolor='#666666')

plt.savefig(os.path.join(FIGURE_DIR, 'fig_forest_adjusted_sensitivity.png'), bbox_inches='tight', dpi=600)
plt.show()
print("  ✓ Saved to figures/fig_forest_adjusted_sensitivity.png")

In [ ]:
# ============================================================================
# CELL 4 (UPDATED) — run_paired_analysis function, NB GEE primary
# ============================================================================

from statsmodels.genmod.families import NegativeBinomial as NBFamily

# Kidney SOTR identification (safe to re-run)
kidney_col = [c for c in df.columns if 'SOTR, what type' in c and 'Kidney' in c]
if kidney_col:
    df['is_kidney'] = (df[kidney_col[0]] == 'Checked')
else:
    df['is_kidney'] = False

strata = {
    'Immunocompetent':       df['immune'] == 'Immunocompetent',
    'Immunocompromised':     df['immune'] == 'Immunosuppressed',
    'SOTR':                  (df['immune'] == 'Immunosuppressed') & (df['is_sotr']),
    'Non-SOTR immunocomp.':  (df['immune'] == 'Immunosuppressed') & (~df['is_sotr']),
    'Kidney SOTR':           (df['immune'] == 'Immunosuppressed') & (df['is_sotr']) & (df['is_kidney']),
    'Non-kidney SOTR':       (df['immune'] == 'Immunosuppressed') & (df['is_sotr']) & (~df['is_kidney']),
}


def run_paired_analysis(stratum_mask, stratum_label):
    """Run all 4 pre-window comparisons for a given subgroup.
    Primary: NB GEE. Sensitivity: Poisson GEE + Wilcoxon."""
    sub_results = []

    for label, info in pre_comp.items():
        mask = info['mask'] & stratum_mask
        n_pairs = mask.sum()

        rec = {
            'stratum':      stratum_label,
            'window':       label,
            'n':            n_pairs,
            'irr':          np.nan,
            'ci_lo':        np.nan,
            'ci_hi':        np.nan,
            'p_gee':        np.nan,
            'scale':        np.nan,
            'irr_pois':     np.nan,
            'ci_lo_pois':   np.nan,
            'ci_hi_pois':   np.nan,
            'p_pois':       np.nan,
            'alpha_nb':     np.nan,
            'p_lrt':        np.nan,
            'p_wilcoxon':   np.nan,
            'median_pre':   np.nan,
            'median_on':    np.nan,
            'n_decreased':  0,
            'n_increased':  0,
        }

        if n_pairs < 5:
            print(f"    {label:12s}  n={n_pairs} — skipped (too few)")
            sub_results.append(rec)
            continue

        # Stack pre + on
        pre_scc_vals = info['scc'].loc[mask[mask].index].values
        on_scc_vals  = df.loc[mask, 'on_scc'].values
        on_time_vals = df.loc[mask, 'on_time_mo'].values

        pre_df = pd.DataFrame({
            'scc':      pre_scc_vals,
            'on_drug':  0,
            'log_time': np.log(info['time_mo']),
            'patient':  np.arange(n_pairs),
        })
        on_df = pd.DataFrame({
            'scc':      on_scc_vals,
            'on_drug':  1,
            'log_time': np.log(on_time_vals),
            'patient':  np.arange(n_pairs),
        })
        stacked = pd.concat([pre_df, on_df], ignore_index=True)
        stacked = stacked.sort_values('patient').reset_index(drop=True)
        X = sm.add_constant(stacked['on_drug'])

        # ── NB GEE — PRIMARY (iterative alpha estimation) ───────────────
        try:
            # Estimate alpha from NB2 GLM, then plug into GEE
            _nb_glm = NB2(stacked['scc'].values, X.values,
                           offset=stacked['log_time'].values)
            _nb_glm_res = _nb_glm.fit(disp=0, maxiter=300)
            try:
                alpha_hat = np.exp(_nb_glm_res.lnalpha)
            except AttributeError:
                alpha_hat = 1.0

            nb_gee_mod = GEE(
                stacked['scc'], X,
                groups=stacked['patient'],
                family=NBFamily(alpha=alpha_hat),
                cov_struct=Exchangeable(),
                offset=stacked['log_time'],
            )
            nb_gee_res = nb_gee_mod.fit(maxiter=200)

            rec['irr']   = np.exp(nb_gee_res.params.iloc[1])
            ci_arr       = nb_gee_res.conf_int().iloc[1]
            rec['ci_lo'] = np.exp(ci_arr.iloc[0])
            rec['ci_hi'] = np.exp(ci_arr.iloc[1])
            rec['p_gee'] = nb_gee_res.pvalues.iloc[1]
            rec['scale'] = nb_gee_res.scale

            # Overflow guard: baseline rate ≈ 0 → IRR explodes
            if rec['irr'] > 1e6 or np.isinf(rec['irr']):
                rec['irr'] = rec['ci_lo'] = rec['ci_hi'] = rec['p_gee'] = np.nan
        except Exception as e:
            print(f"    {label:12s}  NB GEE FAILED: {e}")

        # ── Poisson GEE — Sensitivity 1 ─────────────────────────────────
        try:
            pois_gee_mod = GEE(
                stacked['scc'], X,
                groups=stacked['patient'],
                family=PoissonFamily(),
                cov_struct=Exchangeable(),
                offset=stacked['log_time'],
            )
            pois_gee_res = pois_gee_mod.fit()

            rec['irr_pois']   = np.exp(pois_gee_res.params.iloc[1])
            ci_arr_p          = pois_gee_res.conf_int().iloc[1]
            rec['ci_lo_pois'] = np.exp(ci_arr_p.iloc[0])
            rec['ci_hi_pois'] = np.exp(ci_arr_p.iloc[1])
            rec['p_pois']     = pois_gee_res.pvalues.iloc[1]

            if rec['irr_pois'] > 1e6 or np.isinf(rec['irr_pois']):
                rec['irr_pois'] = rec['ci_lo_pois'] = rec['ci_hi_pois'] = rec['p_pois'] = np.nan
        except Exception:
            pass

        # ── NB vs Poisson GLM — Dispersion diagnostic ───────────────────
        try:
            mod_nb = NB2(stacked['scc'].values, X.values,
                         offset=stacked['log_time'].values)
            res_nb = mod_nb.fit(disp=0, maxiter=300)
            ll_nb = res_nb.llf
            try:
                rec['alpha_nb'] = np.exp(res_nb.lnalpha)
            except AttributeError:
                pass

            mod_pois = PoissonDisc(stacked['scc'].values, X.values,
                                   offset=stacked['log_time'].values)
            res_pois = mod_pois.fit(disp=0, maxiter=300)
            ll_pois = res_pois.llf

            lrt_stat = -2 * (ll_pois - ll_nb)
            # One-sided test: α is on the boundary (α ≥ 0), so the asymptotic
            # null distribution is a 50:50 mixture of χ²(0) and χ²(1).
            # Ref: Self & Liang (1987), JASA 82:605–610.
            rec['p_lrt'] = chi2.sf(lrt_stat, 1) / 2
        except Exception:
            pass

        # ── Wilcoxon — Sensitivity 2 ────────────────────────────────────
        pre_rate_ann = pre_scc_vals / info['time_mo'] * 12
        on_rate_ann  = on_scc_vals / on_time_vals * 12
        diff = on_rate_ann - pre_rate_ann

        rec['n_decreased'] = int((diff < 0).sum())
        rec['n_increased'] = int((diff > 0).sum())
        rec['median_pre']  = np.median(pre_rate_ann)
        rec['median_on']   = np.median(on_rate_ann)

        nonzero = diff != 0
        if nonzero.sum() > 0:
            stat_w, p_w = wilcoxon(diff[nonzero])
            rec['p_wilcoxon'] = p_w

        # ── Print ────────────────────────────────────────────────────────
        irr_str = f"{rec['irr']:.3f}" if pd.notna(rec['irr']) else "—"
        ci_str  = f"[{rec['ci_lo']:.3f}, {rec['ci_hi']:.3f}]" if pd.notna(rec['ci_lo']) else ""
        p_str   = f"p={rec['p_gee']:.2e}" if pd.notna(rec['p_gee']) else ""
        pw_str  = f"p={rec['p_wilcoxon']:.2e}" if pd.notna(rec['p_wilcoxon']) else "—"
        pois_str = f"Pois IRR={rec['irr_pois']:.3f}" if pd.notna(rec['irr_pois']) else ""

        print(f"    {label:12s}  n={n_pairs:3d}  "
              f"IRR={irr_str:>6s} {ci_str:>22s}  {p_str:>12s}  "
              f"| {pois_str}  "
              f"| W {pw_str:>10s}  "
              f"↓{rec['n_decreased']}  ↑{rec['n_increased']}")

        sub_results.append(rec)

    return sub_results


# ── Run all strata ───────────────────────────────────────────────────────
all_strat_results = []

print("=" * 70)
print("  STRATIFIED PAIRED ANALYSIS")
print("=" * 70)

for stratum_name, stratum_mask in strata.items():
    n_stratum = stratum_mask.sum()
    print(f"\n{'═' * 70}")
    print(f"  {stratum_name.upper()}  (n = {n_stratum})")
    print(f"{'═' * 70}")
    sub = run_paired_analysis(stratum_mask, stratum_name)
    all_strat_results.extend(sub)

strat_df = pd.DataFrame(all_strat_results)

print("\n\n" + "=" * 70)
print("  FULL STRATIFIED RESULTS TABLE")
print("=" * 70)
with pd.option_context('display.max_columns', None, 'display.width', 160, 'display.max_rows', None):
    cols_show = ['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee',
                 'irr_pois', 'p_pois', 'p_wilcoxon', 'n_decreased', 'n_increased']
    print(strat_df[cols_show].to_string(index=False, float_format='%.4f'))

In [ ]:
# ============================================================================
# CELL 5 — Combined forest plot (all strata, side by side)
# ============================================================================

# ── Merge unstratified + stratified ──────────────────────────────────────
# Add "All patients" from the earlier results_df
all_df = results_df.copy()
all_df['stratum'] = 'All patients'
all_df['window']  = all_df['label']

combined = pd.concat([
    all_df[['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee']],
    strat_df[['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee']],
], ignore_index=True)

# Drop rows with no estimate
combined = combined.dropna(subset=['irr']).reset_index(drop=True)

# ── Layout: grouped by stratum ───────────────────────────────────────────
stratum_order = [
    'All patients',
    'Immunocompetent',
    'Immunocompromised',
    'SOTR',
    'Non-SOTR immunocomp.',
    'Kidney SOTR',
    'Non-kidney SOTR',
]

window_order = [w for w, _ in pre_windows]

# Colors per window
win_colors = {
    'Pre 1':    C['plum'],
    'Pre 1–2':  C['mauve'],
    'Pre 1–5':  C['pink'],
    'Pre 1–10': C['slate'],
}
win_markers = {
    'Pre 1':    'D',
    'Pre 1–2':  'D',
    'Pre 1–5':  'D',
    'Pre 1–10': 'D',
}

# Build y positions: strata are groups separated by gaps
y_positions = []   # (y, stratum_label, window_label, row_data)
y = 0
group_label_positions = []   # (y_center, stratum_name)

for s in stratum_order:
    sub = combined[combined['stratum'] == s].copy()
    if len(sub) == 0:
        continue

    # Stratum header gap
    y += 1.2

    start_y = y
    for w in window_order:
        row = sub[sub['window'] == w]
        if len(row) == 0:
            continue
        row = row.iloc[0]
        y_positions.append((y, s, w, row))
        y += 1.0
    end_y = y - 1.0

    group_label_positions.append(((start_y + end_y) / 2, s))

# ── Draw ─────────────────────────────────────────────────────────────────
n_rows = len(y_positions)
fig_h = max(n_rows * 0.52 + 3, 8)
fig, ax = plt.subplots(figsize=(14, fig_h))

# Invert so first stratum is at top
max_y = y + 0.5
yticks = []
ytick_labels = []

for (ypos, stratum, window, row) in y_positions:
    yp = max_y - ypos   # flip

    color = win_colors.get(window, C['plum'])

    # CI bar
    ax.plot([row['ci_lo'], row['ci_hi']], [yp, yp],
            color=color, lw=3.5, solid_capstyle='round', zorder=3, alpha=0.85)

    # Diamond
    ax.plot(row['irr'], yp, 'D', color=color, markersize=10,
            markeredgecolor='white', markeredgewidth=1.5, zorder=5)

    # Right-side annotation
    p = row['p_gee']
    if   p < 0.001: p_str = "p < 0.001"
    elif p < 0.01:  p_str = f"p = {p:.3f}"
    elif p < 0.05:  p_str = f"p = {p:.3f}"
    else:           p_str = f"p = {p:.2f}"

    txt = f"{window}  n={int(row['n']):>3d}   IRR {row['irr']:.2f}  [{row['ci_lo']:.2f}–{row['ci_hi']:.2f}]  {p_str}"

    # Decide text x position based on CI
    ax.text(0.01, yp, txt, va='center', fontsize=7.5, color='#333',
            family='monospace', transform=ax.get_yaxis_transform(), zorder=6)

# ── Stratum labels (left side) ───────────────────────────────────────────
for (y_center, s_name) in group_label_positions:
    yp = max_y - y_center
    ax.text(-0.02, yp, s_name, va='center', ha='right', fontsize=10,
            fontweight='bold', color=C['navy'],
            transform=ax.get_yaxis_transform())

# ── Dividers between strata ──────────────────────────────────────────────
prev_stratum = None
for (ypos, stratum, window, row) in y_positions:
    yp = max_y - ypos
    if prev_stratum is not None and stratum != prev_stratum:
        ax.axhline(yp + 0.55, color='#CCCCCC', lw=0.8, ls='-', zorder=0)
    prev_stratum = stratum

# ── Reference line ───────────────────────────────────────────────────────
ax.axvline(1.0, color=C['navy'], ls='--', lw=2.0, alpha=0.4, zorder=0)

# ── Direction arrows at bottom ───────────────────────────────────────────
y_bottom = max_y - y - 0.5
ax.text(0.5, y_bottom - 0.6, '← Favours acitretin',
        ha='center', fontsize=9, color=C['plum'], style='italic')
ax.text(2.0, y_bottom - 0.6, 'Favours pre-treatment →',
        ha='center', fontsize=9, color=C['coral'], style='italic')

# ── Axis formatting ──────────────────────────────────────────────────────
ax.set_xlabel('Incidence Rate Ratio  (on-acitretin / pre-treatment)', fontsize=11)
ax.set_title('Stratified within-patient SCC rate comparison\n'
             'Negative binomial GEE · exchangeable correlation · log(person-months) offset · robust SEs',
             fontsize=14, fontweight='bold', pad=14)

ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_linewidth(2.0)

# x limits: accommodate CIs + text
x_lo = min(combined['ci_lo'].min() * 0.6, 0.05)
x_hi = max(combined['ci_hi'].max() * 1.15, 4.0)
ax.set_xlim(x_lo, x_hi)
ax.set_xscale('log')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.2f}' if v < 1 else f'{v:.1f}'))
ax.set_xticks([0.1, 0.25, 0.5, 1.0, 2.0, 4.0])

y_top_pos = max_y - min(yp for yp, _, _, _ in y_positions) + 1.0
ax.set_ylim(y_bottom - 1.2, y_top_pos + 0.5)

# ── Legend for window colors ─────────────────────────────────────────────
from matplotlib.lines import Line2D
legend_els = [
    Line2D([0], [0], marker='D', color='w', markerfacecolor=win_colors[w],
           markersize=8, markeredgecolor='white', markeredgewidth=1,
           label=w, lw=0)
    for w in window_order
]
ax.legend(handles=legend_els, fontsize=8.5, loc='upper right',
          title='Pre-treatment window', title_fontsize=9,
          framealpha=0.9, edgecolor='#666666')

plt.savefig(os.path.join(FIGURE_DIR, 'fig_forest_stratified.png'), bbox_inches='tight', dpi=600)
plt.show()
print("  ✓ Saved to figures/fig_forest_stratified.png")

# ── Also print a compact table for the manuscript ────────────────────────
print("\n" + "=" * 70)
print("  COMPACT TABLE — STRATIFIED IRRs")
print("=" * 70)
pivot_data = []
for _, row in combined.iterrows():
    p = row['p_gee']
    if   p < 0.001: p_str = "< 0.001"
    elif p < 0.01:  p_str = f"{p:.3f}"
    else:           p_str = f"{p:.2f}"

    pivot_data.append({
        'Stratum':  row['stratum'],
        'Window':   row['window'],
        'n':        int(row['n']),
        'IRR':      f"{row['irr']:.2f}",
        'CI':       f"{row['ci_lo']:.2f}–{row['ci_hi']:.2f}",
        'p':        p_str,
    })

pivot_df = pd.DataFrame(pivot_data)
with pd.option_context('display.max_rows', None, 'display.width', 140):
    print(pivot_df.to_string(index=False))

In [ ]:
# ============================================================================
# CELL 6 — Stratified paired analysis: Nicotinamide & Field Therapy
# ============================================================================
# Paste after Cells 1–5.
# Requires: df, scc_month_cens, pre_scc, pre_comp, C, np, pd,
#           sm, GEE, PoissonFamily, Exchangeable, NB2, PoissonDisc,
#           wilcoxon, chi2, results_df, run_paired_analysis, mticker

strata_adj = {
    'Nicotinamide':        df['nicotinamide'] == 1,
    'No nicotinamide':     df['nicotinamide'] == 0,
    'Field therapy':       df['field_therapy'] == 1,
    'No field therapy':    df['field_therapy'] == 0,
}

all_adj_results = []

print("=" * 70)
print("  STRATIFIED PAIRED ANALYSIS — ADJUNCTIVE THERAPIES")
print("=" * 70)

for stratum_name, stratum_mask in strata_adj.items():
    n_stratum = stratum_mask.sum()
    print(f"\n{'═' * 70}")
    print(f"  {stratum_name.upper()}  (n = {n_stratum})")
    print(f"{'═' * 70}")
    sub = run_paired_analysis(stratum_mask, stratum_name)
    all_adj_results.extend(sub)

adj_df = pd.DataFrame(all_adj_results)

print("\n\n" + "=" * 70)
print("  FULL RESULTS TABLE — ADJUNCTIVE THERAPIES")
print("=" * 70)
with pd.option_context('display.max_columns', None, 'display.width', 160, 'display.max_rows', None):
    cols_show = ['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee',
                 'alpha_nb', 'p_lrt', 'p_wilcoxon', 'n_decreased', 'n_increased']
    print(adj_df[cols_show].to_string(index=False, float_format='%.4f'))

In [ ]:
# ============================================================================
# CELL 7 — Combined forest plot: All + Nicotinamide + Field Therapy
# ============================================================================

# Merge unstratified + adjunctive
all_df2 = results_df.copy()
all_df2['stratum'] = 'All patients'
all_df2['window']  = all_df2['label']

combined_adj = pd.concat([
    all_df2[['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee']],
    adj_df[['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee']],
], ignore_index=True)

combined_adj = combined_adj.dropna(subset=['irr']).reset_index(drop=True)

# ── Layout ───────────────────────────────────────────────────────────────
stratum_order_adj = [
    'All patients',
    'Nicotinamide',
    'No nicotinamide',
    'Field therapy',
    'No field therapy',
]

window_order = [w for w, _ in pre_windows]

win_colors = {
    'Pre 1':    C['plum'],
    'Pre 1–2':  C['mauve'],
    'Pre 1–5':  C['pink'],
    'Pre 1–10': C['slate'],
}

# Build y positions
y_positions = []
y = 0
group_label_positions = []

for s in stratum_order_adj:
    sub = combined_adj[combined_adj['stratum'] == s].copy()
    if len(sub) == 0:
        continue

    y += 1.2
    start_y = y
    for w in window_order:
        row = sub[sub['window'] == w]
        if len(row) == 0:
            continue
        row = row.iloc[0]
        y_positions.append((y, s, w, row))
        y += 1.0
    end_y = y - 1.0
    group_label_positions.append(((start_y + end_y) / 2, s))

# ── Draw ─────────────────────────────────────────────────────────────────
n_rows = len(y_positions)
fig_h = max(n_rows * 0.52 + 3, 8)
fig, ax = plt.subplots(figsize=(14, fig_h))

max_y = y + 0.5

for (ypos, stratum, window, row) in y_positions:
    yp = max_y - ypos

    color = win_colors.get(window, C['plum'])

    ax.plot([row['ci_lo'], row['ci_hi']], [yp, yp],
            color=color, lw=3.5, solid_capstyle='round', zorder=3, alpha=0.85)

    ax.plot(row['irr'], yp, 'D', color=color, markersize=10,
            markeredgecolor='white', markeredgewidth=1.5, zorder=5)

    p = row['p_gee']
    if   p < 0.001: p_str = "p < 0.001"
    elif p < 0.01:  p_str = f"p = {p:.3f}"
    elif p < 0.05:  p_str = f"p = {p:.3f}"
    else:           p_str = f"p = {p:.2f}"

    txt = f"{window}  n={int(row['n']):>3d}   IRR {row['irr']:.2f}  [{row['ci_lo']:.2f}–{row['ci_hi']:.2f}]  {p_str}"
    ax.text(0.01, yp, txt, va='center', fontsize=7.5, color='#333',
            family='monospace', transform=ax.get_yaxis_transform(), zorder=6)

# Stratum labels
for (y_center, s_name) in group_label_positions:
    yp = max_y - y_center
    ax.text(-0.02, yp, s_name, va='center', ha='right', fontsize=10,
            fontweight='bold', color=C['navy'],
            transform=ax.get_yaxis_transform())

# Dividers
prev_stratum = None
for (ypos, stratum, window, row) in y_positions:
    yp = max_y - ypos
    if prev_stratum is not None and stratum != prev_stratum:
        ax.axhline(yp + 0.55, color='#CCCCCC', lw=0.8, ls='-', zorder=0)
    prev_stratum = stratum

# Reference line
ax.axvline(1.0, color=C['navy'], ls='--', lw=2.0, alpha=0.4, zorder=0)

# Direction arrows
y_bottom = max_y - y - 0.5
ax.text(0.5, y_bottom - 0.6, '← Favours acitretin',
        ha='center', fontsize=9, color=C['plum'], style='italic')
ax.text(2.0, y_bottom - 0.6, 'Favours pre-treatment →',
        ha='center', fontsize=9, color=C['coral'], style='italic')

# Axis formatting
ax.set_xlabel('Incidence Rate Ratio  (on-acitretin / pre-treatment)', fontsize=11)
ax.set_title('Stratified within-patient SCC rate comparison — adjunctive therapies\n'
             'Negative binomial GEE · exchangeable correlation · log(person-months) offset · robust SEs',
             fontsize=14, fontweight='bold', pad=14)

ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_linewidth(2.0)

x_lo = min(combined_adj['ci_lo'].min() * 0.6, 0.05)
x_hi = max(combined_adj['ci_hi'].max() * 1.15, 4.0)
ax.set_xlim(x_lo, x_hi)
ax.set_xscale('log')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.2f}' if v < 1 else f'{v:.1f}'))
ax.set_xticks([0.1, 0.25, 0.5, 1.0, 2.0, 4.0])

y_top_pos = max_y - min(yp for yp, _, _, _ in y_positions) + 1.0
ax.set_ylim(y_bottom - 1.2, y_top_pos + 0.5)

# Legend
from matplotlib.lines import Line2D
legend_els = [
    Line2D([0], [0], marker='D', color='w', markerfacecolor=win_colors[w],
           markersize=8, markeredgecolor='white', markeredgewidth=1,
           label=w, lw=0)
    for w in window_order
]
ax.legend(handles=legend_els, fontsize=8.5, loc='upper right',
          title='Pre-treatment window', title_fontsize=9,
          framealpha=0.9, edgecolor='#666666')

plt.savefig(os.path.join(FIGURE_DIR, 'fig_forest_adjunctive.png'), bbox_inches='tight', dpi=600)
plt.show()
print("  ✓ Saved to figures/fig_forest_adjunctive.png")

# ── Compact table ────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  COMPACT TABLE — ADJUNCTIVE THERAPY IRRs")
print("=" * 70)
pivot_data = []
for _, row in combined_adj.iterrows():
    p = row['p_gee']
    if   p < 0.001: p_str = "< 0.001"
    elif p < 0.01:  p_str = f"{p:.3f}"
    else:           p_str = f"{p:.2f}"

    pivot_data.append({
        'Stratum':  row['stratum'],
        'Window':   row['window'],
        'n':        int(row['n']),
        'IRR':      f"{row['irr']:.2f}",
        'CI':       f"{row['ci_lo']:.2f}–{row['ci_hi']:.2f}",
        'p':        p_str,
    })

pivot_df = pd.DataFrame(pivot_data)
with pd.option_context('display.max_rows', None, 'display.width', 140):
    print(pivot_df.to_string(index=False))

In [ ]:
# ============================================================================
# CELL 8 — Stratified paired analysis: Minimum treatment duration
# ============================================================================
# Paste after Cells 1–5 (needs run_paired_analysis from Cell 4).
# Requires: df, scc_month_cens, pre_scc, pre_comp, C, np, pd,
#           sm, GEE, PoissonFamily, Exchangeable, NB2, PoissonDisc,
#           wilcoxon, chi2, results_df, run_paired_analysis, mticker

strata_dur = {
    'All patients (≥3 mo)':  pd.Series(True, index=df.index),
    '≥6 months on drug':     df['months_on_drug'] >= 6,
    '≥12 months on drug':    df['months_on_drug'] >= 12,
    'Full 24 months':        df['months_on_drug'] >= 24,
}

all_dur_results = []

print("=" * 70)
print("  STRATIFIED PAIRED ANALYSIS — MINIMUM TREATMENT DURATION")
print("=" * 70)

for stratum_name, stratum_mask in strata_dur.items():
    n_stratum = stratum_mask.sum()
    print(f"\n{'═' * 70}")
    print(f"  {stratum_name.upper()}  (n = {n_stratum})")
    print(f"{'═' * 70}")
    sub = run_paired_analysis(stratum_mask, stratum_name)
    all_dur_results.extend(sub)

dur_df = pd.DataFrame(all_dur_results)

print("\n\n" + "=" * 70)
print("  FULL RESULTS TABLE — TREATMENT DURATION STRATA")
print("=" * 70)
with pd.option_context('display.max_columns', None, 'display.width', 160, 'display.max_rows', None):
    cols_show = ['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee',
                 'alpha_nb', 'p_lrt', 'p_wilcoxon', 'n_decreased', 'n_increased']
    print(dur_df[cols_show].to_string(index=False, float_format='%.4f'))

In [ ]:
# ============================================================================
# CELL 9 — Combined forest plot: Treatment duration strata
# ============================================================================

# "All patients (≥3 mo)" is already in dur_df, so no need to merge results_df
combined_dur = dur_df[['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee']].copy()
combined_dur = combined_dur.dropna(subset=['irr']).reset_index(drop=True)

# ── Layout ───────────────────────────────────────────────────────────────
stratum_order_dur = [
    'All patients (≥3 mo)',
    '≥6 months on drug',
    '≥12 months on drug',
    'Full 24 months',
]

window_order = [w for w, _ in pre_windows]

win_colors = {
    'Pre 1':    C['plum'],
    'Pre 1–2':  C['mauve'],
    'Pre 1–5':  C['pink'],
    'Pre 1–10': C['slate'],
}

# Build y positions
y_positions = []
y = 0
group_label_positions = []

for s in stratum_order_dur:
    sub = combined_dur[combined_dur['stratum'] == s].copy()
    if len(sub) == 0:
        continue

    y += 1.2
    start_y = y
    for w in window_order:
        row = sub[sub['window'] == w]
        if len(row) == 0:
            continue
        row = row.iloc[0]
        y_positions.append((y, s, w, row))
        y += 1.0
    end_y = y - 1.0
    group_label_positions.append(((start_y + end_y) / 2, s))

# ── Draw ─────────────────────────────────────────────────────────────────
n_rows = len(y_positions)
fig_h = max(n_rows * 0.52 + 3, 8)
fig, ax = plt.subplots(figsize=(14, fig_h))

max_y = y + 0.5

for (ypos, stratum, window, row) in y_positions:
    yp = max_y - ypos

    color = win_colors.get(window, C['plum'])

    ax.plot([row['ci_lo'], row['ci_hi']], [yp, yp],
            color=color, lw=3.5, solid_capstyle='round', zorder=3, alpha=0.85)

    ax.plot(row['irr'], yp, 'D', color=color, markersize=10,
            markeredgecolor='white', markeredgewidth=1.5, zorder=5)

    p = row['p_gee']
    if   p < 0.001: p_str = "p < 0.001"
    elif p < 0.01:  p_str = f"p = {p:.3f}"
    elif p < 0.05:  p_str = f"p = {p:.3f}"
    else:           p_str = f"p = {p:.2f}"

    txt = f"{window}  n={int(row['n']):>3d}   IRR {row['irr']:.2f}  [{row['ci_lo']:.2f}–{row['ci_hi']:.2f}]  {p_str}"
    ax.text(0.01, yp, txt, va='center', fontsize=7.5, color='#333',
            family='monospace', transform=ax.get_yaxis_transform(), zorder=6)

# Stratum labels
for (y_center, s_name) in group_label_positions:
    yp = max_y - y_center
    ax.text(-0.02, yp, s_name, va='center', ha='right', fontsize=10,
            fontweight='bold', color=C['navy'],
            transform=ax.get_yaxis_transform())

# Dividers
prev_stratum = None
for (ypos, stratum, window, row) in y_positions:
    yp = max_y - ypos
    if prev_stratum is not None and stratum != prev_stratum:
        ax.axhline(yp + 0.55, color='#CCCCCC', lw=0.8, ls='-', zorder=0)
    prev_stratum = stratum

# Reference line
ax.axvline(1.0, color=C['navy'], ls='--', lw=2.0, alpha=0.4, zorder=0)

# Direction arrows
y_bottom = max_y - y - 0.5
ax.text(0.5, y_bottom - 0.6, '← Favours acitretin',
        ha='center', fontsize=9, color=C['plum'], style='italic')
ax.text(2.0, y_bottom - 0.6, 'Favours pre-treatment →',
        ha='center', fontsize=9, color=C['coral'], style='italic')

# Axis formatting
ax.set_xlabel('Incidence Rate Ratio  (on-acitretin / pre-treatment)', fontsize=11)
ax.set_title('Stratified within-patient SCC rate comparison — treatment duration\n'
             'Negative binomial GEE · exchangeable correlation · log(person-months) offset · robust SEs',
             fontsize=14, fontweight='bold', pad=14)

ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_linewidth(2.0)

x_lo = min(combined_dur['ci_lo'].min() * 0.6, 0.05)
x_hi = max(combined_dur['ci_hi'].max() * 1.15, 4.0)
ax.set_xlim(x_lo, x_hi)
ax.set_xscale('log')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.2f}' if v < 1 else f'{v:.1f}'))
ax.set_xticks([0.1, 0.25, 0.5, 1.0, 2.0, 4.0])

y_top_pos = max_y - min(yp for yp, _, _, _ in y_positions) + 1.0
ax.set_ylim(y_bottom - 1.2, y_top_pos + 0.5)

# Legend
from matplotlib.lines import Line2D
legend_els = [
    Line2D([0], [0], marker='D', color='w', markerfacecolor=win_colors[w],
           markersize=8, markeredgecolor='white', markeredgewidth=1,
           label=w, lw=0)
    for w in window_order
]
ax.legend(handles=legend_els, fontsize=8.5, loc='upper right',
          title='Pre-treatment window', title_fontsize=9,
          framealpha=0.9, edgecolor='#666666')

plt.savefig(os.path.join(FIGURE_DIR, 'fig_forest_duration.png'), bbox_inches='tight', dpi=600)
plt.show()
print("  ✓ Saved to figures/fig_forest_duration.png")

# ── Compact table ────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  COMPACT TABLE — TREATMENT DURATION IRRs")
print("=" * 70)
pivot_data = []
for _, row in combined_dur.iterrows():
    p = row['p_gee']
    if   p < 0.001: p_str = "< 0.001"
    elif p < 0.01:  p_str = f"{p:.3f}"
    else:           p_str = f"{p:.2f}"

    pivot_data.append({
        'Stratum':  row['stratum'],
        'Window':   row['window'],
        'n':        int(row['n']),
        'IRR':      f"{row['irr']:.2f}",
        'CI':       f"{row['ci_lo']:.2f}–{row['ci_hi']:.2f}",
        'p':        p_str,
    })

pivot_df = pd.DataFrame(pivot_data)
with pd.option_context('display.max_rows', None, 'display.width', 140):
    print(pivot_df.to_string(index=False))

In [ ]:
# ============================================================================
# CELL — Pre-treatment SCC burden distribution (for choosing strata)
# ============================================================================
# Paste after cells 1–6.

yr1 = pre_scc['yr1'].copy()
yr1_valid = yr1.dropna()

print("=" * 70)
print("  PRE-TREATMENT YEAR-1 SCC DISTRIBUTION")
print("=" * 70)
print(f"  n = {len(yr1_valid)}  (missing: {yr1.isna().sum()})")
print(f"  Mean:   {yr1_valid.mean():.2f}")
print(f"  Median: {yr1_valid.median():.0f}")
print(f"  IQR:    {yr1_valid.quantile(.25):.0f} – {yr1_valid.quantile(.75):.0f}")
print(f"  Range:  {yr1_valid.min():.0f} – {yr1_valid.max():.0f}")

print(f"\n  Value counts:")
vc = yr1_valid.value_counts().sort_index()
cum = 0
for val, cnt in vc.items():
    cum += cnt
    pct = 100 * cnt / len(yr1_valid)
    cum_pct = 100 * cum / len(yr1_valid)
    print(f"    {val:5.0f} SCCs:  n={cnt:3d}  ({pct:5.1f}%)   cumulative: {cum:3d}  ({cum_pct:5.1f}%)")

print(f"\n  Percentiles:")
for p in [10, 20, 25, 30, 33, 40, 50, 60, 67, 70, 75, 80, 90, 95]:
    print(f"    P{p:2d} = {yr1_valid.quantile(p/100):.1f}")

print(f"\n  Candidate binary splits:")
for cut in [1, 2, 3, 4, 5, 6, 8, 10]:
    n_lo = (yr1_valid <= cut).sum()
    n_hi = (yr1_valid > cut).sum()
    print(f"    ≤{cut:2d} vs >{cut:2d}:   {n_lo:3d} vs {n_hi:3d}   ({100*n_lo/len(yr1_valid):.0f}% / {100*n_hi/len(yr1_valid):.0f}%)")

print(f"\n  Candidate tertile splits:")
for lo, hi in [(1,3), (1,4), (1,5), (2,5), (2,6), (3,6), (2,4)]:
    n_low  = (yr1_valid <= lo).sum()
    n_mid  = ((yr1_valid > lo) & (yr1_valid <= hi)).sum()
    n_high = (yr1_valid > hi).sum()
    print(f"    ≤{lo} / {lo+1}–{hi} / >{hi}:   {n_low:3d} / {n_mid:3d} / {n_high:3d}   "
          f"({100*n_low/len(yr1_valid):.0f}% / {100*n_mid/len(yr1_valid):.0f}% / {100*n_high/len(yr1_valid):.0f}%)")

In [ ]:
# ============================================================================
# CELL 10 — Stratified paired analysis: Pre-treatment SCC burden
# ============================================================================
# Paste after Cells 1–5 (needs run_paired_analysis from Cell 4).
# Requires: df, scc_month_cens, pre_scc, pre_comp, C, np, pd,
#           sm, GEE, PoissonFamily, Exchangeable, NB2, PoissonDisc,
#           wilcoxon, chi2, results_df, run_paired_analysis, mticker

# ── Assign burden groups based on Year-1 pre-treatment SCC ───────────────
df['pre_yr1_scc'] = pre_scc['yr1'].values

df['burden_group'] = pd.cut(
    df['pre_yr1_scc'],
    bins=[-0.1, 0, 1, 3, float('inf')],
    labels=['0 SCCs', '1 SCC', '2–3 SCCs', '4+ SCCs'],
)

print("=" * 70)
print("  PRE-TREATMENT BURDEN GROUPS (Year −1)")
print("=" * 70)
for grp in ['0 SCCs', '1 SCC', '2–3 SCCs', '4+ SCCs']:
    mask = df['burden_group'] == grp
    n = mask.sum()
    on_rate = df.loc[mask, 'on_scc'].sum() / df.loc[mask, 'on_time_mo'].sum() * 12
    pre_rate = df.loc[mask, 'pre_yr1_scc'].mean()
    print(f"  {grp:10s}  n={n:3d}  "
          f"pre mean={pre_rate:.2f}/yr  on ann.={on_rate:.2f}/yr")

strata_burden = {
    '0 SCCs (Year −1)':   df['burden_group'] == '0 SCCs',
    '1 SCC (Year −1)':    df['burden_group'] == '1 SCC',
    '2–3 SCCs (Year −1)': df['burden_group'] == '2–3 SCCs',
    '4+ SCCs (Year −1)':  df['burden_group'] == '4+ SCCs',
}

all_burden_results = []

print("\n" + "=" * 70)
print("  STRATIFIED PAIRED ANALYSIS — PRE-TREATMENT SCC BURDEN")
print("=" * 70)

for stratum_name, stratum_mask in strata_burden.items():
    n_stratum = stratum_mask.sum()
    print(f"\n{'═' * 70}")
    print(f"  {stratum_name.upper()}  (n = {n_stratum})")
    print(f"{'═' * 70}")
    sub = run_paired_analysis(stratum_mask, stratum_name)
    all_burden_results.extend(sub)

burden_df = pd.DataFrame(all_burden_results)

print("\n\n" + "=" * 70)
print("  FULL RESULTS TABLE — PRE-TREATMENT BURDEN STRATA")
print("=" * 70)
with pd.option_context('display.max_columns', None, 'display.width', 160, 'display.max_rows', None):
    cols_show = ['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee',
                 'alpha_nb', 'p_lrt', 'p_wilcoxon', 'n_decreased', 'n_increased']
    print(burden_df[cols_show].to_string(index=False, float_format='%.4f'))

In [ ]:
# ============================================================================
# CELL 11 — Combined forest plot: Pre-treatment burden strata
# ============================================================================

# Include "All patients" from results_df
all_df3 = results_df.copy()
all_df3['stratum'] = 'All patients'
all_df3['window']  = all_df3['label']

combined_burden = pd.concat([
    all_df3[['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee']],
    burden_df[['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee']],
], ignore_index=True)

combined_burden = combined_burden.dropna(subset=['irr']).reset_index(drop=True)

# ── Layout ───────────────────────────────────────────────────────────────
stratum_order_burden = [
    'All patients',
    '0 SCCs (Year −1)',
    '1 SCC (Year −1)',
    '2–3 SCCs (Year −1)',
    '4+ SCCs (Year −1)',
]

window_order = [w for w, _ in pre_windows]

win_colors = {
    'Pre 1':    C['plum'],
    'Pre 1–2':  C['mauve'],
    'Pre 1–5':  C['pink'],
    'Pre 1–10': C['slate'],
}

# Build y positions
y_positions = []
y = 0
group_label_positions = []

for s in stratum_order_burden:
    sub = combined_burden[combined_burden['stratum'] == s].copy()
    if len(sub) == 0:
        continue

    y += 1.2
    start_y = y
    for w in window_order:
        row = sub[sub['window'] == w]
        if len(row) == 0:
            continue
        row = row.iloc[0]
        y_positions.append((y, s, w, row))
        y += 1.0
    end_y = y - 1.0
    group_label_positions.append(((start_y + end_y) / 2, s))

# ── Draw ─────────────────────────────────────────────────────────────────
n_rows = len(y_positions)
fig_h = max(n_rows * 0.52 + 3, 6)
fig, ax = plt.subplots(figsize=(14, fig_h))

max_y = y + 0.5

for (ypos, stratum, window, row) in y_positions:
    yp = max_y - ypos

    color = win_colors.get(window, C['plum'])

    # Check for undefined IRR (baseline rate = 0 → IRR undefined)
    if pd.isna(row['ci_lo']) or pd.isna(row['ci_hi']) or row['irr'] > 1e6:
        ax.plot(1.0, yp, 'x', color='#999999', markersize=8, markeredgewidth=2, zorder=5)
        txt = f"{window}  n={int(row['n']):>3d}   IRR undefined — all patients had 0 SCCs in baseline window *"
        ax.text(0.01, yp, txt, va='center', fontsize=7.5, color='#999999',
                family='monospace', transform=ax.get_yaxis_transform(), zorder=6)
        continue

    ax.plot([row['ci_lo'], row['ci_hi']], [yp, yp],
            color=color, lw=3.5, solid_capstyle='round', zorder=3, alpha=0.85)

    ax.plot(row['irr'], yp, 'D', color=color, markersize=10,
            markeredgecolor='white', markeredgewidth=1.5, zorder=5)

    p = row['p_gee']
    if   p < 0.001: p_str = "p < 0.001"
    elif p < 0.01:  p_str = f"p = {p:.3f}"
    elif p < 0.05:  p_str = f"p = {p:.3f}"
    else:           p_str = f"p = {p:.2f}"

    txt = f"{window}  n={int(row['n']):>3d}   IRR {row['irr']:.2f}  [{row['ci_lo']:.2f}–{row['ci_hi']:.2f}]  {p_str}"
    ax.text(0.01, yp, txt, va='center', fontsize=7.5, color='#333',
            family='monospace', transform=ax.get_yaxis_transform(), zorder=6)

# Stratum labels
for (y_center, s_name) in group_label_positions:
    yp = max_y - y_center
    ax.text(-0.02, yp, s_name, va='center', ha='right', fontsize=10,
            fontweight='bold', color=C['navy'],
            transform=ax.get_yaxis_transform())

# Dividers
prev_stratum = None
for (ypos, stratum, window, row) in y_positions:
    yp = max_y - ypos
    if prev_stratum is not None and stratum != prev_stratum:
        ax.axhline(yp + 0.55, color='#CCCCCC', lw=0.8, ls='-', zorder=0)
    prev_stratum = stratum

# Reference line
ax.axvline(1.0, color=C['navy'], ls='--', lw=2.0, alpha=0.4, zorder=0)

# Direction arrows
y_bottom = max_y - y - 0.5
ax.text(0.5, y_bottom - 0.6, '← Favours acitretin',
        ha='center', fontsize=9, color=C['plum'], style='italic')
ax.text(2.0, y_bottom - 0.6, 'Favours pre-treatment →',
        ha='center', fontsize=9, color=C['coral'], style='italic')

# Axis formatting
ax.set_xlabel('Incidence Rate Ratio  (on-acitretin / pre-treatment)', fontsize=11)
ax.set_title('Stratified within-patient SCC rate comparison — pre-treatment SCC burden\n'
             'Negative binomial GEE · exchangeable correlation · log(person-months) offset · robust SEs',
             fontsize=14, fontweight='bold', pad=14)

ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_linewidth(2.0)

x_lo = min(combined_burden['ci_lo'].min() * 0.6, 0.05)
x_hi = max(combined_burden['ci_hi'].max() * 1.15, 4.0)
ax.set_xlim(x_lo, x_hi)
ax.set_xscale('log')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.2f}' if v < 1 else f'{v:.1f}'))
ax.set_xticks([0.1, 0.25, 0.5, 1.0, 2.0, 4.0])

y_top_pos = max_y - min(yp for yp, _, _, _ in y_positions) + 1.0
ax.set_ylim(y_bottom - 1.8, y_top_pos + 0.5)

# Legend
from matplotlib.lines import Line2D
legend_els = [
    Line2D([0], [0], marker='D', color='w', markerfacecolor=win_colors[w],
           markersize=8, markeredgecolor='white', markeredgewidth=1,
           label=w, lw=0)
    for w in window_order
]
ax.legend(handles=legend_els, fontsize=8.5, loc='upper right',
          title='Pre-treatment window', title_fontsize=9,
          framealpha=0.9, edgecolor='#666666')

# Footnote
ax.text(0.01, -0.13,
        '* IRR is undefined when the pre-treatment baseline rate is exactly 0 (no SCCs in window).\n'
        '  A rate ratio requires a non-zero denominator. Longer windows (Pre 1–2, etc.) may recover\n'
        '  estimability if patients had SCCs in earlier years.',
        transform=ax.transAxes, fontsize=7.5, color='#777777', style='italic',
        va='top', family='sans-serif')

plt.savefig(os.path.join(FIGURE_DIR, 'fig_forest_burden_4group.png'), bbox_inches='tight', dpi=600)
plt.show()
print("  ✓ Saved to figures/fig_forest_burden_4group.png")

# ── Compact table ────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  COMPACT TABLE — PRE-TREATMENT BURDEN IRRs")
print("=" * 70)
pivot_data = []
for _, row in combined_burden.iterrows():
    p = row['p_gee']
    if   p < 0.001: p_str = "< 0.001"
    elif p < 0.01:  p_str = f"{p:.3f}"
    else:           p_str = f"{p:.2f}"

    pivot_data.append({
        'Stratum':  row['stratum'],
        'Window':   row['window'],
        'n':        int(row['n']),
        'IRR':      f"{row['irr']:.2f}",
        'CI':       f"{row['ci_lo']:.2f}–{row['ci_hi']:.2f}",
        'p':        p_str,
    })

pivot_df = pd.DataFrame(pivot_data)
with pd.option_context('display.max_rows', None, 'display.width', 140):
    print(pivot_df.to_string(index=False))

In [ ]:
# ============================================================================
# CELL 12 — Binary burden split: 0–1 vs 2+ SCCs (Year −1)
# ============================================================================
# Paste after Cell 10–11.
# Requires: run_paired_analysis, df, pre_scc, pre_comp, results_df, C, etc.

strata_binary = {
    '0–1 SCCs (Year −1)': df['pre_yr1_scc'] <= 1,
    '2+ SCCs (Year −1)':  df['pre_yr1_scc'] >= 2,
}

all_binary_results = []

print("=" * 70)
print("  STRATIFIED PAIRED ANALYSIS — BINARY BURDEN SPLIT (0–1 vs 2+)")
print("=" * 70)

for stratum_name, stratum_mask in strata_binary.items():
    n_stratum = stratum_mask.sum()
    print(f"\n{'═' * 70}")
    print(f"  {stratum_name.upper()}  (n = {n_stratum})")
    print(f"{'═' * 70}")
    sub = run_paired_analysis(stratum_mask, stratum_name)
    all_binary_results.extend(sub)

binary_df = pd.DataFrame(all_binary_results)

print("\n\n" + "=" * 70)
print("  FULL RESULTS TABLE — BINARY BURDEN SPLIT")
print("=" * 70)
with pd.option_context('display.max_columns', None, 'display.width', 160, 'display.max_rows', None):
    cols_show = ['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee',
                 'alpha_nb', 'p_lrt', 'p_wilcoxon', 'n_decreased', 'n_increased']
    print(binary_df[cols_show].to_string(index=False, float_format='%.4f'))


# ============================================================================
# CELL 13 — Forest plot: All + 4-group burden + binary burden
# ============================================================================

# Combine everything into one figure
all_df4 = results_df.copy()
all_df4['stratum'] = 'All patients'
all_df4['window']  = all_df4['label']

combined_all_burden = pd.concat([
    all_df4[['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee']],
    burden_df[['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee']],
    binary_df[['stratum', 'window', 'n', 'irr', 'ci_lo', 'ci_hi', 'p_gee']],
], ignore_index=True)

combined_all_burden = combined_all_burden.dropna(subset=['irr']).reset_index(drop=True)

# ── Layout ───────────────────────────────────────────────────────────────
stratum_order_all_burden = [
    'All patients',
    '0–1 SCCs (Year −1)',
    '2+ SCCs (Year −1)',
]

window_order = [w for w, _ in pre_windows]

win_colors = {
    'Pre 1':    C['plum'],
    'Pre 1–2':  C['mauve'],
    'Pre 1–5':  C['pink'],
    'Pre 1–10': C['slate'],
}

# Build y positions
y_positions = []
y = 0
group_label_positions = []

for s in stratum_order_all_burden:
    sub = combined_all_burden[combined_all_burden['stratum'] == s].copy()
    if len(sub) == 0:
        continue

    y += 1.2
    start_y = y
    for w in window_order:
        row = sub[sub['window'] == w]
        if len(row) == 0:
            continue
        row = row.iloc[0]
        y_positions.append((y, s, w, row))
        y += 1.0
    end_y = y - 1.0
    group_label_positions.append(((start_y + end_y) / 2, s))

# ── Draw ─────────────────────────────────────────────────────────────────
n_rows = len(y_positions)
fig_h = max(n_rows * 0.48 + 4, 6)
fig, ax = plt.subplots(figsize=(14, fig_h))

max_y = y + 0.5

for (ypos, stratum, window, row) in y_positions:
    yp = max_y - ypos

    color = win_colors.get(window, C['plum'])

    # Handle undefined IRR
    if pd.isna(row['ci_lo']) or pd.isna(row['ci_hi']) or row['irr'] > 1e6:
        ax.plot(1.0, yp, 'x', color='#999999', markersize=8, markeredgewidth=2, zorder=5)
        txt = f"{window}  n={int(row['n']):>3d}   IRR undefined — all patients had 0 SCCs in baseline window *"
        ax.text(0.01, yp, txt, va='center', fontsize=7.5, color='#999999',
                family='monospace', transform=ax.get_yaxis_transform(), zorder=6)
        continue

    ax.plot([row['ci_lo'], row['ci_hi']], [yp, yp],
            color=color, lw=3.5, solid_capstyle='round', zorder=3, alpha=0.85)

    ax.plot(row['irr'], yp, 'D', color=color, markersize=10,
            markeredgecolor='white', markeredgewidth=1.5, zorder=5)

    p = row['p_gee']
    if   p < 0.001: p_str = "p < 0.001"
    elif p < 0.01:  p_str = f"p = {p:.3f}"
    elif p < 0.05:  p_str = f"p = {p:.3f}"
    else:           p_str = f"p = {p:.2f}"

    txt = f"{window}  n={int(row['n']):>3d}   IRR {row['irr']:.2f}  [{row['ci_lo']:.2f}–{row['ci_hi']:.2f}]  {p_str}"
    ax.text(0.01, yp, txt, va='center', fontsize=7.5, color='#333',
            family='monospace', transform=ax.get_yaxis_transform(), zorder=6)

# Stratum labels
for (y_center, s_name) in group_label_positions:
    yp = max_y - y_center
    ax.text(-0.02, yp, s_name, va='center', ha='right', fontsize=10,
            fontweight='bold', color=C['navy'],
            transform=ax.get_yaxis_transform())

# Dividers between strata
prev_stratum = None
for (ypos, stratum, window, row) in y_positions:
    yp = max_y - ypos
    if prev_stratum is not None and stratum != prev_stratum:
        ax.axhline(yp + 0.55, color='#CCCCCC', lw=0.8, ls='-', zorder=0)
    prev_stratum = stratum

# Reference line
ax.axvline(1.0, color=C['navy'], ls='--', lw=2.0, alpha=0.4, zorder=0)

# Direction arrows
y_bottom = max_y - y - 0.5
ax.text(0.5, y_bottom - 0.6, '← Favours acitretin',
        ha='center', fontsize=9, color=C['plum'], style='italic')
ax.text(2.0, y_bottom - 0.6, 'Favours pre-treatment →',
        ha='center', fontsize=9, color=C['coral'], style='italic')

# Axis formatting
ax.set_xlabel('Incidence Rate Ratio  (on-acitretin / pre-treatment)', fontsize=11)
ax.set_title('Stratified within-patient SCC rate comparison — pre-treatment SCC burden\n'
             'Negative binomial GEE · exchangeable correlation · log(person-months) offset · robust SEs',
             fontsize=14, fontweight='bold', pad=14)

ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_linewidth(2.0)

x_lo = min(combined_all_burden.loc[combined_all_burden['irr'] < 1e6, 'ci_lo'].min() * 0.6, 0.05)
x_hi = max(combined_all_burden.loc[combined_all_burden['irr'] < 1e6, 'ci_hi'].max() * 1.15, 4.0)
ax.set_xlim(x_lo, x_hi)
ax.set_xscale('log')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.2f}' if v < 1 else f'{v:.1f}'))
ax.set_xticks([0.1, 0.25, 0.5, 1.0, 2.0, 4.0])

y_top_pos = max_y - min(yp for yp, _, _, _ in y_positions) + 1.0
ax.set_ylim(y_bottom - 2.0, y_top_pos + 0.5)

# Legend
from matplotlib.lines import Line2D
legend_els = [
    Line2D([0], [0], marker='D', color='w', markerfacecolor=win_colors[w],
           markersize=8, markeredgecolor='white', markeredgewidth=1,
           label=w, lw=0)
    for w in window_order
]
if combined_all_burden['ci_lo'].isna().any() or (combined_all_burden['irr'] > 1e6).any():
    legend_els.append(
        Line2D([0], [0], marker='x', color='#999999', markersize=8,
               markeredgewidth=2, lw=0, label='IRR undefined *')
    )
ax.legend(handles=legend_els, fontsize=8.5, loc='upper right',
          title='Pre-treatment window', title_fontsize=9,
          framealpha=0.9, edgecolor='#666666')

plt.savefig(os.path.join(FIGURE_DIR, 'fig_forest_burden.png'), bbox_inches='tight', dpi=600)
plt.show()
print("  ✓ Saved to figures/fig_forest_burden.png")

# ── Compact table ────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  COMPACT TABLE — ALL BURDEN STRATA IRRs")
print("=" * 70)
pivot_data = []
for _, row in combined_all_burden.iterrows():
    if row['irr'] > 1e6 or pd.isna(row['ci_lo']):
        irr_str = 'undef.'
        ci_str  = '—'
        p_str   = '—'
    else:
        p = row['p_gee']
        if   p < 0.001: p_str = "< 0.001"
        elif p < 0.01:  p_str = f"{p:.3f}"
        else:           p_str = f"{p:.2f}"
        irr_str = f"{row['irr']:.2f}"
        ci_str  = f"{row['ci_lo']:.2f}–{row['ci_hi']:.2f}"

    pivot_data.append({
        'Stratum':  row['stratum'],
        'Window':   row['window'],
        'n':        int(row['n']),
        'IRR':      irr_str,
        'CI':       ci_str,
        'p':        p_str,
    })

pivot_df = pd.DataFrame(pivot_data)
with pd.option_context('display.max_rows', None, 'display.width', 140):
    print(pivot_df.to_string(index=False))


In [ ]:
# ============================================================================
# TABLE 1 — Cohort characteristics
# ============================================================================
# Continuous variables: mean ± SD, median [IQR], and Shapiro-Wilk normality p.
# Categorical variables: n (%).
# All figures computed on the analysis cohort (df) post-exclusions.
# ============================================================================

from scipy.stats import shapiro

def _fmt_p(p):
    if pd.isna(p):     return '—'
    if p < 0.001:      return '<0.001'
    if p < 0.01:       return f'{p:.3f}'
    return f'{p:.2f}'

def summarize_continuous(series, label, unit=''):
    """Return a dict with both parametric and non-parametric summaries + SW test."""
    s = pd.to_numeric(series, errors='coerce').dropna()
    n = len(s)

    mean_sd   = f'{s.mean():.1f} ± {s.std():.1f}'
    med_iqr   = f'{s.median():.1f} [{s.quantile(.25):.1f}–{s.quantile(.75):.1f}]'
    rng       = f'{s.min():.0f}–{s.max():.0f}'

    # Shapiro-Wilk — only valid for 3 ≤ n ≤ 5000
    if 3 <= n <= 5000:
        sw_stat, sw_p = shapiro(s)
        sw_str = f'W={sw_stat:.3f}, p={_fmt_p(sw_p)}'
    else:
        sw_str = '—'

    suffix = f' ({unit})' if unit else ''
    return {
        'Variable':        label + suffix,
        'n':               n,
        'Mean ± SD':       mean_sd,
        'Median [IQR]':    med_iqr,
        'Range':           rng,
        'Shapiro-Wilk':    sw_str,
    }

def summarize_categorical(series, label, order=None):
    """Return a list of dicts: one header row, then one row per level."""
    s = series.dropna()
    n_total = len(s)
    rows = [{'Variable': label, 'n': n_total,
             'Mean ± SD': '', 'Median [IQR]': '', 'Range': '', 'Shapiro-Wilk': ''}]

    vc = s.value_counts()
    levels = order if order else vc.index.tolist()
    for lvl in levels:
        n = int(vc.get(lvl, 0))
        pct = 100 * n / n_total if n_total else 0
        rows.append({
            'Variable':     f'    {lvl}',
            'n':            f'{n} ({pct:.1f}%)',
            'Mean ± SD':    '',
            'Median [IQR]': '',
            'Range':        '',
            'Shapiro-Wilk': '',
        })
    return rows

# ── Build SOTR organ variable from the raw "Checked" columns ──────────────
organ_map = {
    'Kidney':              [c for c in df.columns if 'SOTR, what type' in c and 'Kidney'      in c],
    'Liver':               [c for c in df.columns if 'SOTR, what type' in c and 'Liver'       in c],
    'Heart':               [c for c in df.columns if 'SOTR, what type' in c and 'Heart'       in c],
    'Lung':                [c for c in df.columns if 'SOTR, what type' in c and 'Lung'        in c],
    'Small bowel/pancreas':[c for c in df.columns if 'SOTR, what type' in c and ('Small' in c or 'pancreas' in c.lower())],
}

def _organ(row):
    if not row['is_sotr']:
        return np.nan
    for organ, cols in organ_map.items():
        if cols and any(row[c] == 'Checked' for c in cols if c in row.index):
            return organ
    return 'Unspecified'

df['sotr_organ'] = df.apply(_organ, axis=1)

# ── Assemble table ────────────────────────────────────────────────────────
rows = []

def _section_header(title):
    """Visual separator row — a section break inside the table."""
    return {'Variable': f'── {title} ' + '─' * max(0, 60 - len(title)),
            'n': '', 'Mean ± SD': '', 'Median [IQR]': '',
            'Range': '', 'Shapiro-Wilk': ''}

# ═══════════════════════════════════════════════════════════════════════════
# SECTION 1 — ORIGINAL TABLE 1 (demographics, immune, adjunctive)
# ═══════════════════════════════════════════════════════════════════════════
rows.append(_section_header('Demographics'))
rows.append(summarize_continuous(df['age'], 'Age', 'years'))
rows.extend(summarize_categorical(df['sex'], 'Sex', order=['M', 'F']))

rows.append(_section_header('Immune status'))
rows.extend(summarize_categorical(df['immune'], 'Immune status',
                                   order=['Immunocompetent', 'Immunosuppressed']))
imm_sub = df.loc[df['immune'] == 'Immunosuppressed'].copy()
imm_sub['imm_type'] = np.where(imm_sub['is_sotr'], 'SOTR',
                       np.where(imm_sub['is_cll'], 'CLL', 'Other'))
rows.extend(summarize_categorical(imm_sub['imm_type'],
                                   f'Immunosuppression type (of n={len(imm_sub)} suppressed)',
                                   order=['SOTR', 'CLL', 'Other']))
sotr_sub = df.loc[df['is_sotr']]
rows.extend(summarize_categorical(sotr_sub['sotr_organ'],
                                   f'SOTR organ (of n={len(sotr_sub)} SOTR)',
                                   order=['Kidney', 'Liver', 'Heart', 'Lung', 'Small bowel/pancreas']))

rows.append(_section_header('Adjunctive therapies'))
rows.append(summarize_continuous(df['months_on_drug'], 'Treatment duration', 'months'))
rows.append(summarize_continuous(pre_scc['yr1'], 'Pre-treatment SCC burden (Year −1)', 'SCCs'))

n_both    = int(((df['nicotinamide']==1) & (df['field_therapy']==1)).sum())
n_nic     = int(((df['nicotinamide']==1) & (df['field_therapy']==0)).sum())
n_ft      = int(((df['nicotinamide']==0) & (df['field_therapy']==1)).sum())
n_neither = int(((df['nicotinamide']==0) & (df['field_therapy']==0)).sum())
N = len(df)
rows.append({'Variable': 'Adjunctive therapies', 'n': N,
             'Mean ± SD': '', 'Median [IQR]': '', 'Range': '', 'Shapiro-Wilk': ''})
for lvl, n in [('Nicotinamide + field therapy', n_both),
               ('Nicotinamide only',             n_nic),
               ('Field therapy only',            n_ft),
               ('Neither',                       n_neither)]:
    rows.append({'Variable': f'    {lvl}', 'n': f'{n} ({100*n/N:.1f}%)',
                 'Mean ± SD': '', 'Median [IQR]': '', 'Range': '', 'Shapiro-Wilk': ''})

# ═══════════════════════════════════════════════════════════════════════════
# ─── ADDED AFTER INITIAL DRAFT ─────────────────────────────────────────────
# The sections below were added in a second pass. They are cohort-level
# descriptors (site structure, follow-up completion, pre-treatment data
# depth, on-drug exposure, on-drug outcomes) that aren't traditional
# Table 1 entries but were added at co-author request.
# ═══════════════════════════════════════════════════════════════════════════
rows.append({'Variable': '', 'n': '', 'Mean ± SD': '', 'Median [IQR]': '',
             'Range': '', 'Shapiro-Wilk': ''})
rows.append({'Variable': '══ Added after initial draft ' + '═' * 40,
             'n': '', 'Mean ± SD': '', 'Median [IQR]': '',
             'Range': '', 'Shapiro-Wilk': ''})

# ── Sites ────────────────────────────────────────────────────────────────
rows.append(_section_header('Participating sites'))
site_counts = df['site'].value_counts()
rows.append({'Variable':     'Number of sites',
             'n':            int(site_counts.shape[0]),
             'Mean ± SD':    '', 'Median [IQR]':
                             f'{site_counts.median():.0f} [{site_counts.quantile(.25):.0f}–{site_counts.quantile(.75):.0f}]',
             'Range':        f'{site_counts.min()}–{site_counts.max()}',
             'Shapiro-Wilk': ''})
rows.append({'Variable':     '    Patients per site',
             'n':            '', 'Mean ± SD': '',
             'Median [IQR]': '(see row above)',
             'Range': '', 'Shapiro-Wilk': ''})

# ── Treatment completion ────────────────────────────────────────────────
rows.append(_section_header('Treatment completion'))
for cutoff, lbl in [(6, '≥6 months on drug'),
                    (12, '≥12 months on drug'),
                    (24, 'Full 24 months (completers)')]:
    n = int((df['months_on_drug'] >= cutoff).sum())
    rows.append({'Variable':     f'    {lbl}',
                 'n':            f'{n} ({100*n/N:.1f}%)',
                 'Mean ± SD':    '', 'Median [IQR]': '',
                 'Range':        '', 'Shapiro-Wilk': ''})
n_disc = int((df['months_on_drug'] < 24).sum())
rows.append({'Variable':     '    Discontinued before 24 months',
             'n':            f'{n_disc} ({100*n_disc/N:.1f}%)',
             'Mean ± SD':    '', 'Median [IQR]': '',
             'Range':        '', 'Shapiro-Wilk': ''})

# ── Pre-treatment data depth ────────────────────────────────────────────
rows.append(_section_header('Pre-treatment SCC history available'))
for label, years in [('Year −1 only',           [1]),
                     ('Years −1 to −2 (all)',   [1, 2]),
                     ('Years −1 to −5 (all)',   list(range(1, 6))),
                     ('Years −1 to −10 (all)',  list(range(1, 11)))]:
    cols = [f'yr{y}' for y in years]
    n = int(pre_scc[cols].notna().all(axis=1).sum())
    rows.append({'Variable':     f'    {label}',
                 'n':            f'{n} ({100*n/N:.1f}%)',
                 'Mean ± SD':    '', 'Median [IQR]': '',
                 'Range':        '', 'Shapiro-Wilk': ''})

# ── On-drug exposure (dose) ─────────────────────────────────────────────
rows.append(_section_header('On-drug dose exposure'))

# Mean daily dose per patient, averaged over observed on-drug months
per_patient_mean_dose = []
for idx in df.index:
    last_on = int(df.loc[idx, 'months_on_drug'])
    doses = [dose_mg_cens.loc[idx, f'm{m}'] for m in range(1, last_on + 1)]
    doses = [d for d in doses if pd.notna(d) and d > 0]
    per_patient_mean_dose.append(np.mean(doses) if doses else np.nan)

mean_dose_series = pd.Series(per_patient_mean_dose, index=df.index)
rows.append(summarize_continuous(mean_dose_series,
                                  'Mean daily dose (per patient, averaged over on-drug months)',
                                  'mg'))

# ── On-drug outcomes ────────────────────────────────────────────────────
rows.append(_section_header('On-drug outcomes'))

total_on_scc   = int(df['on_scc'].sum()) if 'on_scc' in df.columns else int(scc_month_cens.sum().sum())
total_on_ptmo  = int(df['months_on_drug'].sum())
rate_per_pyr   = total_on_scc / (total_on_ptmo / 12)

rows.append({'Variable':     'Total on-drug SCCs',
             'n':            total_on_scc,
             'Mean ± SD':    '', 'Median [IQR]': '',
             'Range':        '', 'Shapiro-Wilk': ''})
rows.append({'Variable':     'Total on-drug patient-months',
             'n':            total_on_ptmo,
             'Mean ± SD':    '', 'Median [IQR]': '',
             'Range':        '', 'Shapiro-Wilk': ''})
rows.append({'Variable':     'On-drug SCC rate (per patient-year)',
             'n':            f'{rate_per_pyr:.2f}',
             'Mean ± SD':    '', 'Median [IQR]': '',
             'Range':        '', 'Shapiro-Wilk': ''})

n_zero = int((df.get('is_zero_scc', (scc_month_cens.sum(axis=1) == 0).astype(int)) == 1).sum())
rows.append({'Variable':     'Patients with zero on-drug SCCs',
             'n':            f'{n_zero} ({100*n_zero/N:.1f}%)',
             'Mean ± SD':    '', 'Median [IQR]': '',
             'Range':        '', 'Shapiro-Wilk': ''})

# Per-patient on-drug SCC count (distribution)
per_pt_on_scc = scc_month_cens.sum(axis=1, min_count=1).fillna(0)
rows.append(summarize_continuous(per_pt_on_scc, 'On-drug SCC count (per patient)', 'SCCs'))

# ── DataFrame + display + export ─────────────────────────────────────────
table1 = pd.DataFrame(rows)

print("=" * 100)
print(f"  TABLE 1 — Cohort characteristics  (N = {len(df)})")
print("=" * 100)
with pd.option_context('display.max_columns', None, 'display.width', 140,
                        'display.max_rows', None, 'display.colheader_justify', 'left'):
    print(table1.to_string(index=False))

table1.to_csv('./table1_cohort_characteristics.csv', index=False)
print(f"\n  ✓ Saved to ./table1_cohort_characteristics.csv")

In [ ]:
table1

In [ ]:
# %% NEW CELL — Table 1a & 1b (stratified, with between-group tests)
# Requires upstream: df, pre_scc, np, pd, df['is_zero_scc'], df['mean_dose_mg'], df['on_scc']
from scipy.stats import mannwhitneyu, kruskal, chi2_contingency, fisher_exact

def _fmtp(p):
    if pd.isna(p):  return '—'
    if p < 0.001:   return '<0.001'
    if p < 0.01:    return f'{p:.3f}'
    return f'{p:.2f}'

def _cont_cell(s):
    s = pd.to_numeric(s, errors='coerce').dropna()
    if len(s) == 0: return '—'
    return f'{s.median():.1f} [{s.quantile(.25):.1f}–{s.quantile(.75):.1f}]  (n={len(s)})'

def _cont_test(groups):
    gs = [pd.to_numeric(pd.Series(g), errors='coerce').dropna().values for g in groups]
    gs = [g for g in gs if len(g) > 0]
    if len(gs) < 2: return np.nan, '—'
    if len(gs) == 2:
        try:    _, p = mannwhitneyu(gs[0], gs[1], alternative='two-sided')
        except ValueError: p = np.nan
        return p, 'Mann–Whitney U'
    try:    _, p = kruskal(*gs)
    except ValueError: p = np.nan
    return p, 'Kruskal–Wallis'

def _cat_test(series, group_masks, levels):
    table = np.array([[int(((series == lvl) & m).sum()) for m in group_masks]
                      for lvl in levels], dtype=float)
    table = table[table.sum(axis=1) > 0]                      # drop empty levels
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan, '—'
    if table.shape == (2, 2):
        exp = chi2_contingency(table)[3]
        if (exp < 5).any():
            _, p = fisher_exact(table)
            return p, 'Fisher exact'
    try:    _, p, _, _ = chi2_contingency(table)
    except ValueError: p = np.nan
    return p, 'Chi-square'

def build_strat_table(specs, group_masks, group_order):
    gcol = {g: f'{g} (n={int(group_masks[g].sum())})' for g in group_order}
    out = []
    for spec in specs:
        kind = spec[0]
        if kind == 'section':
            row = {'Variable': f'── {spec[1]} ' + '─' * max(0, 46 - len(spec[1]))}
            for g in group_order: row[gcol[g]] = ''
            row['p-value'] = ''; row['Test'] = ''
            out.append(row); continue
        if kind == 'cont':
            _, label, series = spec
            groups = [series[group_masks[g]] for g in group_order]
            p, test = _cont_test(groups)
            row = {'Variable': label}
            for g, grp in zip(group_order, groups):
                row[gcol[g]] = _cont_cell(grp)
            row['p-value'] = _fmtp(p); row['Test'] = test
            out.append(row)
        elif kind == 'cat':
            _, label, series, levels = spec
            p, test = _cat_test(series, [group_masks[g] for g in group_order], levels)
            hdr = {'Variable': label}
            for g in group_order: hdr[gcol[g]] = ''
            hdr['p-value'] = _fmtp(p); hdr['Test'] = test
            out.append(hdr)
            for lvl in levels:
                row = {'Variable': f'    {lvl}'}
                for g in group_order:
                    sub = series[group_masks[g]].dropna()
                    n, tot = int((sub == lvl).sum()), len(sub)
                    row[gcol[g]] = f'{n} ({100*n/tot:.1f}%)' if tot else '0'
                row['p-value'] = ''; row['Test'] = ''
                out.append(row)
    cols = ['Variable'] + [gcol[g] for g in group_order] + ['p-value', 'Test']
    return pd.DataFrame(out)[cols]

# ── Shared variable specification (same rows for both tables) ─────────────
_nic = df['nicotinamide'].map({1: 'Yes', 0: 'No'})
_ft  = df['field_therapy'].map({1: 'Yes', 0: 'No'})

specs = [
    ('section', 'Demographics'),
    ('cont', 'Age (years)', df['age']),
    ('cat',  'Sex', df['sex'], ['M', 'F']),
    ('section', 'Immune status'),
    ('cat',  'Immune status', df['immune'], ['Immunocompetent', 'Immunosuppressed']),
    ('section', 'Treatment exposure'),
    ('cont', 'Treatment duration (months)', df['months_on_drug']),
    ('cont', 'Mean daily dose (mg)', df['mean_dose_mg']),          # NEW metric
    ('cat',  'Nicotinamide', _nic, ['Yes', 'No']),
    ('cat',  'Field therapy', _ft, ['Yes', 'No']),
    ('section', 'Pre-treatment SCC burden'),
    ('cont', 'Year −1 (SCCs)', pre_scc['yr1']),
    ('cont', 'Years 1–2 sum (SCCs)', pre_scc[['yr1', 'yr2']].sum(axis=1)
                                     .where(pre_scc[['yr1', 'yr2']].notna().all(axis=1))),
    ('section', 'On-drug outcome'),
    ('cont', 'On-drug SCC count (SCCs)', df['on_scc']),
]

def _show(tbl, title):
    print("=" * 110)
    print(f"  {title}")
    print("=" * 110)
    with pd.option_context('display.max_columns', None, 'display.width', 160,
                           'display.max_rows', None, 'display.colheader_justify', 'left'):
        print(tbl.to_string(index=False))
    print()

# ── TABLE 1a — stratified by on-drug SCC status (2 groups) ───────────────
masks_a = {'Zero on-drug SCC': df['is_zero_scc'] == 1,
           'Any on-drug SCC':  df['is_zero_scc'] == 0}
order_a = ['Zero on-drug SCC', 'Any on-drug SCC']
table1a = build_strat_table(specs, masks_a, order_a)
_show(table1a, f'TABLE 1a — Cohort characteristics by on-drug SCC status  (N={len(df)})')
table1a.to_csv('./table1a_by_ondrug_scc.csv', index=False)

# ── TABLE 1b — stratified by Year −1 pre-treatment SCC count (3 groups) ──
_y1 = pre_scc['yr1']
masks_b = {'1 SCC':    _y1 == 1,
           '2–3 SCCs': _y1.isin([2, 3]),
           '4+ SCCs':  _y1 >= 4}
order_b = ['1 SCC', '2–3 SCCs', '4+ SCCs']
table1b = build_strat_table(specs, masks_b, order_b)
_show(table1b, f'TABLE 1b — Cohort characteristics by Year −1 SCC count  '
               f'(n={int(sum(m.sum() for m in masks_b.values()))}; excludes Year −1 = 0)')
table1b.to_csv('./table1b_by_pretx_yr1.csv', index=False)

print("✓ Saved ./table1a_by_ondrug_scc.csv and ./table1b_by_pretx_yr1.csv")

In [ ]:
# %% Cell — Zero-SCC patient lists (strict + zero-or-missing, fully accounted)

windows = {
    '2 years prior':  ['yr1', 'yr2'],
    '10 years prior': [f'yr{y}' for y in range(1, 11)],
}

strict_lists, lenient_lists = {}, {}

for label, cols in windows.items():
    n_pos     = (pre_scc[cols] > 0).sum(axis=1)      # years with a recorded SCC
    n_zero    = (pre_scc[cols] == 0).sum(axis=1)     # years recorded as 0
    n_missing = pre_scc[cols].isna().sum(axis=1)     # blank years

    no_positive = n_pos == 0                         # never had a recorded SCC in window
    fully_obs   = n_missing == 0                     # complete record for window

    # ---- LIST 1: strict — always zero (complete record, all zeros) ----
    strict_mask = no_positive & fully_obs
    strict = (df.loc[strict_mask, ['site', 'patient_id']]
                .rename(columns={'site': 'Site ID', 'patient_id': 'Patient ID'})
                .sort_values(['Site ID', 'Patient ID']).reset_index(drop=True))

    # ---- LIST 2: zero-or-missing — no positive year, blanks allowed ----
    lenient = df.loc[no_positive, ['site', 'patient_id']].copy()
    lenient['zero_years']    = n_zero[no_positive].values
    lenient['missing_years'] = n_missing[no_positive].values
    lenient['status'] = np.where(n_missing[no_positive].values == 0,
                                 'all observed = 0',
                                 'zero + missing')
    lenient = (lenient.rename(columns={'site': 'Site ID', 'patient_id': 'Patient ID'})
                      .sort_values(['status', 'Site ID', 'Patient ID'])
                      .reset_index(drop=True))

    strict_lists[label], lenient_lists[label] = strict, lenient

    # ---- accounting (reconciles to full cohort N) ----
    n_had_scc = int((n_pos > 0).sum())
    print("=" * 60)
    print(f"  {label}   (cohort N = {len(df)})")
    print("=" * 60)
    print(f"  Strict 'always zero' (complete record, all 0): {len(strict)}")
    print(f"  Zero-or-missing (no recorded SCC, blanks ok):  {len(lenient)}")
    print(f"     - of those, fully observed zero:            {(lenient['status']=='all observed = 0').sum()}")
    print(f"     - of those, zero + some missing:            {(lenient['status']=='zero + missing').sum()}")
    print(f"  Had >=1 recorded SCC in window:                {n_had_scc}")
    print(f"  Check: {len(lenient)} + {n_had_scc} = {len(lenient)+n_had_scc} (should equal {len(df)})")
    print()
    print("  --- STRICT (always zero) ---")
    print(strict.to_string(index=False))
    print("\n  --- ZERO-OR-MISSING (with breakdown) ---")
    print(lenient.to_string(index=False))
    print()

# ---- save ----
for label, cols in windows.items():
    tag = label.split()[0]  # '2' or '10'
    strict_lists[label].to_csv(f'./zero_scc_{tag}yr_strict.csv', index=False)
    lenient_lists[label].to_csv(f'./zero_scc_{tag}yr_zero_or_missing.csv', index=False)
print("✓ Saved 4 CSVs: strict + zero_or_missing for each window")

In [ ]:
df

In [ ]:
# %% Cell — Regenerate lists keyed on the REAL "Patient ID" column

REPORT_ID_COL = 'Patient ID'   # <-- the column they actually want reported

assert REPORT_ID_COL in df.columns, (
    f"'{REPORT_ID_COL}' not found in df. ID-ish columns available: "
    f"{[c for c in df.columns if 'id' in c.lower()]}"
)
print(f"Reporting from '{REPORT_ID_COL}'. Sample: "
      f"{df[REPORT_ID_COL].astype(str).str.strip().head(5).tolist()}\n")

windows = {'2 years prior': ['yr1', 'yr2'],
           '10 years prior': [f'yr{y}' for y in range(1, 11)]}

def fmt(sub, extra=None):
    out = pd.DataFrame({'Site ID': df.loc[sub.index, 'site'].values,
                        'Patient ID': df.loc[sub.index, REPORT_ID_COL].astype(str).str.strip().values})
    if extra is not None:
        out = pd.concat([out, extra.reset_index(drop=True)], axis=1)
    return out.sort_values(['Site ID', 'Patient ID']).reset_index(drop=True)

lines = ["SCC-FREE PATIENT LISTS (Site ID / Patient ID)\n"]
for label, cols in windows.items():
    n_pos     = (pre_scc[cols] > 0).sum(axis=1)
    n_zero    = (pre_scc[cols] == 0).sum(axis=1)
    n_missing = pre_scc[cols].isna().sum(axis=1)
    no_pos, full = n_pos == 0, n_missing == 0

    strict = fmt(df[no_pos & full])
    lines += ["=" * 60,
              f"0 SCCs IN {label.upper()} — CONFIRMED (n={len(strict)})",
              "Complete record, all years zero.", "=" * 60,
              strict.to_string(index=False), ""]

    amb = df[no_pos & ~full]
    amb_tbl = fmt(amb, pd.DataFrame({'Zero yrs': n_zero[no_pos & ~full].values,
                                     'Missing yrs': n_missing[no_pos & ~full].values}))
    lines += [f"SUPPLEMENT — {label} zero-or-missing, adds {len(amb_tbl)} "
              f"(total {len(strict)+len(amb_tbl)})",
              "No recorded SCC but some years blank; Zero yrs = 0 means NO data.",
              amb_tbl.to_string(index=False), ""]

email_block = "\n".join(lines)
print(email_block)